# Cross-Model Hard Problem Benchmark — AIMO3

**Stress-test your prompts and model configurations against 9,000+ competition-grade problems** sourced from HLE, Olympiad-Math, and AIME datasets.

### How to use
1. Swap in your own **system prompts** and **model endpoints** (Block I / Block K)
2. Pick a dataset slice or run the full suite (Block N)
3. Review per-problem traces, vote distributions, and LLM-judge evaluations in the output directory
4. Benchmark mode only -- do not use this edition for submission

### Symbolic math support
Answers are **not limited to integers**. The benchmark pipeline handles fractions, radicals, tuples, symbolic expressions, and free-form mathematical objects — with automatic LaTeX normalization for vote aggregation and a three-stage evaluation cascade: exact match → normalized string match → LLM judge.

### Output structure
All reasoning traces from every agent and per-problem statistics are saved to `benchmark_output/`, organized by problem ID and attempt index:
- `attempts/{pb_id}__{agent_name}__{attempt}.json` — full conversation trace per agent
- `meta/{pb_id}__meta.json` — MetaAgent reconciliation traces
- `problems/{pb_id}.json` — vote distribution, effort stats, final answer, correctness

> **Note:** Speculative decoding can be enabled for faster inference but remains unstable — not recommended for submission runs.


# Blackboard Multi-Suite Multi-Agent TIR Solver - AIMO3

*Multi-model inference engine · Multi-layer agentic orchestration · Effort-weighted consensus voting*

*Adaptive time budget · Inference acceleration via speculative decoding · Cross-model generic architecture*

*Persistent sandbox pool · Dual-mode SUBMISSION / BENCHMARK · Structured empirical output*



## Quick Start & Reference

---

### TL;DR

This notebook is a ready-to-run multi-agent mathematical reasoning solver for
AIMO3. The architecture is complete. To run a submission, set `MODE = 'SUBMISSION'`
in Block I, configure the three sections below, and execute all cells in order
from Block A through Block N.

#### Session requirements (H100 80 GB)

| Configuration | GPU memory | Agents | Notes |
|---|---|---|---|
| `gpt-oss-120b` solo | ~76-78 GB | 8 | `gpu_memory_utilization = 0.96`. Fits H100 80 GB with correct vLLM config. |
| `gpt-oss-120b` + Eagle3 speculative decoding | ~76-78 GB + draft overhead | 8 | Reduce `batch_size` to 4-8. Draft model adds KV cache pressure. Monitor OOM. |
| `gpt-oss-20b` + `gpt-oss-20b-math` (two-suite) | ~18 GB + ~18 GB | 8 + 8 | `gpu_memory_utilization = 0.45` each. ~36 GB total, leaves ~44 GB for KV cache. |
| `Qwen35-27B-FP8` solo | ~30 GB | 8 | Enable MTP: `spec_config = mtp_spec(n=5)`. |

`gpt-oss-120b` fits an H100 80 GB because vLLM loads weights in the configured
dtype and allocates remaining memory for KV cache - 120B parameters in
FP8/INT8 quantisation sit within the 76-78 GB envelope. Two-suite configurations
must keep the combined `gpu_memory_utilization` sum below around 75 GB to leave
headroom for CUDA kernels (around 5 GB).

#### The three blocks to configure before each run

**Block I** - model and engine (`ACTIVE_ENGINE_CONFIGS`, model paths, `n_agents`,
`gpu_memory_utilization`, `spec_config`)
```python
# Solo run (default)
ACTIVE_ENGINE_CONFIGS = [_ec_gpt_120b]

# Two-suite experiment
ACTIVE_ENGINE_CONFIGS = [_ec_gpt_20b, _ec_gpt_20b_sft_1]

# Enable speculative decoding (one field on the EngineConfig)
_ec_gpt_120b = GptOssEngineConfig(..., spec_config = spec_decode_amazon)
```

**Block J** - solver behaviour (`CFG`: context budget, temperatures, stop
thresholds, timing bounds)
```python
cfg = CFG(
    agent_context_tokens = int(16384 * 4),  # reduce under speculative decoding
    temp_agent_min = 1.0, temp_agent_max = 1.0,  # widen for more path diversity, for example min = 0.8 max = 1.0 for Monte Carlo sampling
    k2_raw_vote_stop = 4,    # Stop-A: raw majority threshold
    k1_min_valid     = 12,   # Stop-B: minimum answer count before effort gate
)
```

**Block K** - prompts (`SYSTEM_PROMPT`, `PREFERENCE_PROMPT`, per-suite variants)

All other blocks are read-only between experiments. Sandbox pool sizing,
executor worker count, JSON output, adaptive deadline, and MetaAgent
spawning are computed automatically from Block I and J values.

#### Mode toggle (Block I)

```python
MODE = 'SUBMISSION'   # wires to Kaggle evaluation harness
MODE = 'BENCHMARK'    # passes expected answers, labels correctness, writes JSON
```

No other code changes are required to switch modes. In BENCHMARK mode, three
individual JSON files are written to subdirectories of `BENCHMARK_OUTPUT_DIR`
after each problem: `attempts/`, `meta/`, and `problems/`. Each file is a
standalone JSON object named by problem and agent ID.

---

### Summary

This notebook implements a multi-agent Tool-Integrated Reasoning solver for the
AI Mathematical Olympiad Progress Prize 3 competition. Multiple reasoning agents
run concurrently, each operating an independent think-code-execute loop backed
by a persistent Jupyter kernel. Agents write results to a shared Blackboard
without communicating directly. A Controller monitors the Blackboard, evaluates
five stop conditions after each completed attempt, and cancels remaining work
when sufficient evidence accumulates.

Answer selection uses effort-weighted aggregation: each candidate answer is
weighted by the total tokens generated in producing it, treating deeper
multi-turn reasoning as stronger evidence relative to classical raw vote counting.
When the agent population splits across competing answers, a MetaAgent is
spawned as an independent secondary solver - it is given the competing answers
as context but instructed to solve from scratch, a protocol designed to be
informed about the solution space without being anchored to any particular answer.

Time is allocated adaptively per problem using a rolling average of recent solve
durations, self-correcting across the 50-problem run. Multiple model families
are supported under a unified abstract interface with no structural changes
required between experiments. A BENCHMARK mode produces structured JSON output
for empirical analysis of agent behaviour, stop condition distributions, and
effort-accuracy correlation across configurations.

The notebook demonstrates inference acceleration for both stable model families.
For `gpt-oss-120b`, five community-trained Eagle3 draft models are pre-catalogued
and ready to enable with a single `spec_config` field change - each offering a
different throughput/stability trade-off that competitors can benchmark against
their problem set. For `Qwen3.5-27B-FP8` and `Qwen3.5-35B-A3B-FP8`, the
built-in Multi-Token Prediction head delivers approximately 150-200% throughput
improvement with no external draft model required, activated by
`spec_config = mtp_spec(n=5)`. Higher throughput is not only a speed gain: it
directly expands the token budget available per session, enabling more agents,
deeper TIR loops, and the more complex orchestration patterns that are currently
excluded by compute cost.

The architecture is intentionally constrained in scope. Inter-agent
communication, Tree-of-Thought branching, Graph-of-Thought, and multi-stage
verification pipelines are not implemented. These are not oversights. They add
structural complexity that is not yet justified by empirical evidence of benefit
under AIMO3 conditions, and their absence keeps the codebase auditable and
modifiable without deep familiarity with the full system. The primary design
goal is to remove the technical barrier between a competitor's idea and their
ability to test it - not to demonstrate the most sophisticated orchestration
pattern available. Competitors who want to implement these patterns have a clean,
instrumented base to build from. That is the intended outcome.

---

### Disclaimers

**No performance guarantees.** This architecture does not guarantee a competitive
leaderboard score. Score depends on model quality, prompt calibration, stop
condition tuning, and inference throughput - variables that interact in ways
that depend on the specific hardware allocation and problem distribution.
This notebook provides infrastructure for empirical tuning, not tuned values.

**Speculative decoding is experimental for `gpt-oss-120b`.** The Eagle3 variants
catalogued in Block I have been reported to provide significant throughput gains,
but stability under high concurrency and long context is not guaranteed. OOM
under Eagle3 + `batch_size > 8` is a known failure mode. Test on a sample
before a full submission run.

**BENCHMARK results are not required to share**, but are welcomed. If you run a
sweep across model configurations, stop condition settings, or prompt variants,
sharing the `problems/` summaries (full traces are optional) helps the
community understand what configurations are effective under competition
conditions. There is no obligation to do so.

**This is not the only valid architecture.** The Blackboard multi-agent pattern
is one point in a large space of possible approaches. Single-model
chain-of-thought with a higher inference budget, Tree-of-Thought, and other
orchestration patterns are all plausible alternatives. This architecture is
designed to generate empirical data about what works on competition problems,
not to assert a conclusion from it.

---

### References

**Blackboard pattern**
- Hayes-Roth, B. (1985). A Blackboard Architecture for Control. *Artificial Intelligence*, 26(3), 251-321.
- Erman et al. (1980). The Hearsay-II Speech-Understanding System. *ACM Computing Surveys*, 12(2), 213-253.

**Tool-Integrated Reasoning**
- Gou et al. (2023). ToRA: A Tool-Integrated Reasoning Agent for Mathematical Problem Solving. *arXiv:2309.17452*.
- Chen et al. (2022). Program of Thoughts Prompting. *arXiv:2211.12588*.

**Speculative decoding**
- Leviathan et al. (2023). Fast Inference from Transformers via Speculative Decoding. *ICML 2023*. arXiv:2211.17192.
- Li et al. (2024). EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty. *arXiv:2401.15077*.
- Eagle3-AIMO3 community variants: https://github.com/juemifuji/eagle3-aimo3

**Multi-agent and self-consistency**
- Wang et al. (2022). Self-Consistency Improves Chain of Thought Reasoning. *arXiv:2203.11171*.
- Yao et al. (2023). Tree of Thoughts. *arXiv:2305.10601*.
- Du et al. (2023). Improving Factuality and Reasoning through Multiagent Debate. *arXiv:2305.14325*.

**Models and infrastructure**
- OpenAI gpt-oss-120b: https://www.kaggle.com/models/danielhanchen/gpt-oss-120b/
- Qwen3.5 (shelterw uploaded): https://www.kaggle.com/models/shelterw/qwen3.5
- DeepSeek-R1: https://www.kaggle.com/models/deepseek-ai/deepseek-r1-0528
- vLLM: https://docs.vllm.ai/en/latest/
- AIMO3 competition: https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3

---

### Related Community Work

Notebooks from the AIMO3 community that informed or complement this work:

- Qwen3.5 with Python TIR (shelterw): https://www.kaggle.com/code/shelterw/qwen3-5-w-python
- Qwen3.5 Lab (boristown): https://www.kaggle.com/code/boristown/qwen3-5-lab
- AIMO3 Carto - fixed params reference (khoinguyennguyen): https://www.kaggle.com/code/khoinguyennguyen/aimo3-carto-notebook-a-fixed-params
- 44/50 score reference notebook (nihilisticneuralnet): https://www.kaggle.com/code/nihilisticneuralnet/44-50-let-me-over-cook
- Full community notebook listing: https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3/code

If you fork this notebook and make structural changes, the most useful
contribution is to paste the modified cell into the comments with a brief note
on what changed and why. This makes the change auditable without requiring a
full notebook diff.

---

# Blackboard Multi-Suite Multi-Agent TIR Solver - AIMO3

## Conceptual Architecture

---

### Context

The AI Mathematical Olympiad Progress Prize 3 (AIMO3) is a Kaggle competition
in which a submitted notebook must solve 50 mathematical olympiad problems
within a single inference session (50 problems in the public leaderboard,
50 problems in the private leaderboard). Each answer is a non-negative integer
in $[0, 99999]$. The session runs on Kaggle infrastructure (up to 1x H100)
under a wall-clock budget of 5 hours (approximately 4 hours and 50 minutes in
practice, after environment setup). The session has no outbound network access
during inference.

These constraints are not incidental. They define the problem space precisely:

- **Unknown difficulty distribution.** Problems range from routine computation
  to multi-step reasoning that single-pass generation cannot handle. The system
  cannot know in advance how many problems will be hard or how hard they will
  be. A fixed per-problem time budget is therefore suboptimal - easy problems
  waste budget, hard problems are starved.

- **Constrained compute.** Constrained resources (up to 1x H100), open models,
  one session. Decisions about how to spend the token generation budget must be
  made from internal evidence, not from external feedback. There is no
  mechanism to request additional compute mid-run, and no oracle to consult
  when the agent population disagrees.

- **No answer verification.** The system cannot check whether its answers are
  correct in SUBMISSION mode. All decisions about when to stop generating and
  which answer to commit must be made from the distribution of agent outputs
  alone. The only signal available is internal: how many agents agreed, how
  much reasoning depth each invested, and whether an independent secondary
  solver corroborates a contested answer.

These three constraints motivate every major architectural decision in this
notebook.

---

### The Technical Barrier Problem

A competitor who wants to experiment with a new model family faces a specific
class of obstacle before they can run a single experiment. Different model
families use different tokenization schemes, different chat template structures,
different stop token semantics, different reasoning block delimiters, and in
some cases entirely different API endpoint conventions - token-ID completions
versus chat completions, Harmony encoding versus standard message format.
Integrating a new model requires implementing all of these details correctly,
from scratch, before the first useful result is available. This is a
paradigm-switching cost, not a mathematical reasoning cost.

The effect on the competition's collective intelligence is suppressive. The set
of competitors who can experiment with model selection, model mixing, inference
acceleration, and agent configuration is filtered by familiarity with
implementation details rather than by mathematical creativity or strategic
thinking. Competitors with strong ideas about what to try are blocked by the
work of building infrastructure to try it. The result is that the community's
creative capacity is effectively gated behind implementation skill, which is
orthogonal to the skill being measured by the competition.

This architecture is a direct response to that problem. Its central design
constraint is: **adding a new model family requires implementing two classes
and registering one factory line, nothing more.** The `ModelSuiteProtocol`
abstraction (Block E) isolates all model-specific logic behind eight abstract
methods. The `EngineConfig` hierarchy (Block D) isolates all hardware and
server configuration. Everything between the model boundary and the solver
output - agent dispatch, stop conditions, voting, sandbox management, timing,
JSON output - is shared infrastructure that works identically regardless of
which model is running.

A competitor using this notebook can focus on the questions that actually
affect score: which model performs best on competition problems, whether mixing
two model families outperforms either alone, whether a fine-tuned variant
produces better TIR traces than its base model, whether inference acceleration
via speculative decoding is worth the configuration cost, how many agents is
the right number for a given model size and GPU budget, and what prompting
strategy produces the most reliable answers. The infrastructure does not answer
these questions. It removes the obstacles between thinking about them and
testing them empirically.

---

### What This Architecture Solves

The architecture addresses five problems simultaneously.

**Multi-model support without reimplementation.** Running `gpt-oss-20b` and
`Qwen35-27B-FP8` in the same session, or `gpt-oss-20b` alongside a fine-tuned
`gpt-oss-20b-math` variant, requires no changes outside Block I (which model
path and port each engine uses) and Block K (which prompt variant each suite
receives). The shared `Blackboard`, `Controller`, `aggregate_answers()`,
`JupyterSandbox` pool, and JSON output work identically across all
combinations.

The MetaAgent paradigm extends this further. Because a MetaAgent is
structurally identical to a SolverAgent - same TIR loop, same protocol, same
sandbox - it can run on a different model family from the primary solvers.
A configuration where `gpt-oss-20b` serves as the primary population and
`Qwen35-27B-FP8` serves as the MetaAgent engine is a one-line change in
Block I (`META_AGENT_ENGINE_IDX`). This enables experiments in which the
secondary solver brings an independent approach shaped by a different model's
training, rather than merely reinforcing the primary population's reasoning
paths. The MetaAgent receives the competing answers and reasoning excerpts
from the primary solvers as context, then solves from scratch - aggregating
the exploratory work of the primary population into a conflict-resolution pass
that is informed by all prior attempts without being anchored to any of them.

**Time allocation under uncertainty.** The adaptive deadline formula allocates
time per problem based on a rolling average of recent solve times,
automatically expanding budgets after fast problems and contracting them after
slow ones. No fixed per-problem timeout is required. The formula is
self-correcting across the full 50-problem run: a cluster of hard early
problems tightens subsequent budgets; a cluster of easy problems releases
capacity for harder ones that follow.

**Answer selection from noisy multi-agent output.** Classical majority voting
penalises depth and rewards speed - the agent that answers first in 300 tokens
carries equal weight to the agent that reasoned for 6,000 tokens across eight
Python iterations. The effort-weighted aggregation mechanism weights each
answer by the total tokens generated in producing it, treating deeper
multi-turn TIR traces as stronger evidence. Three stop conditions - raw
majority (Stop-A), effort-weighted consensus (Stop-B), and MetaAgent
corroboration (Stop-C) - are evaluated on every incoming result, so easy
problems exit early and hard problems receive the full time budget without any
per-problem classification step.

**Hard problem resolution.** When the primary SolverAgent population splits
across competing answers and no stop condition fires, a MetaAgent is spawned
as an independent secondary solver. It receives the competing answers for
context but is instructed to solve from scratch - the *informed but not
anchored* protocol. This avoids the anchoring bias that would arise from
asking an agent to adjudicate existing traces, while still using the primary
population's work to narrow the search space.

**Inference acceleration.** The speculative decoding layer (`SpecConfig`, Block D)
supports Eagle3 draft models, n-gram prompt lookup, and the Qwen3.5 built-in
MTP head under a unified CLI interface (`--speculative-config '<json>'`).
Enabling speculative decoding requires one field change on the `EngineConfig`
instance. Five community-trained Eagle3 variants for `gpt-oss-120b` are
pre-catalogued in Block I; the Qwen3.5 MTP implementation is stable and
delivers approximately 150-200% throughput improvement at no configuration
cost beyond setting `spec_config = mtp_spec(n=5)`. 

**Unlocking more complex architectures.** Throughput is not only a speed
concern. It is a structural constraint on what reasoning architectures are
feasible within a fixed session budget. Under baseline generation rates, the
token budget is the binding limit on agent count, TIR depth, and orchestration
complexity. Inference acceleration directly expands that envelope. Architectures
whose cost was previously prohibitive - Tree-of-Thought, Graph-of-Thought, dense
inter-agent communication, complex multi-stage orchestration with verification
passes - become feasible when each token costs less wall-clock time. The
speculative decoding infrastructure in this notebook is therefore not only a
performance optimisation. It is an enabler for more capable solution
architectures that were previously excluded from the competition's compute
envelope.

---

### Architecture Overview

```
AIMO3Solver                          top-level orchestrator, public facade
├── Blackboard                       shared state, thread-safe, two-level
│   └── RunTracker                   rolling statistics, adaptive deadline
├── Controller                       stop conditions, MetaAgent dispatch,
│   │                                final answer fallback chain
│   ├── SolverAgent  x N             primary TIR loop per problem
│   │   └── ModelSuiteProtocol       model-specific conversation protocol (abstract)
│   │       ├── GptOssModelSuite     Harmony TIR, token-ID completions  [stable]
│   │       ├── Qwen35ModelSuite     chat completions, AutoTokenizer     [stable]
│   │       └── (4 experimental)     DeepSeek-R1 Qwen, DeepSeek-R1 Llama,
│   │                                OpenReasoning-Nemotron, Gemma3
│   └── MetaAgent    x M             secondary solver on answer divergence
├── JupyterSandbox   x W             persistent kernels, pre-warmed pool
└── InferenceEngineManager x E       vLLM server lifecycle, OpenAI client
```

The architecture follows the **Blackboard pattern**, a coordination scheme
originating in the Hearsay-II speech understanding system (CMU, 1976) and
formalised in Hayes-Roth's BB1 system (1985). A passive shared workspace
accumulates contributions from independent specialist agents. A controller
monitors the workspace and commits to a solution when accumulated evidence
crosses a threshold. Agents do not communicate with each other directly; the
Blackboard is the only shared mutable state between them.

`SolverAgent` instances run concurrently in a `ThreadPoolExecutor`, each
operating an independent TIR loop. They write `AttemptResult` records to the
`Blackboard` without any knowledge of what other agents have produced. The
`Controller` reads snapshots, calls `aggregate_answers()` after each completed
future, evaluates stop conditions, and cancels remaining work when sufficient
evidence accumulates. The two-level state structure separates problem-level
state (reset between problems) from run-level state (persistent across all
50 problems), which is how the adaptive deadline can learn from earlier
problems to budget later ones.

---

### Tool-Integrated Reasoning

TIR is a multi-turn reasoning protocol in which a language model alternates
between generating natural language reasoning and executing Python code in a
persistent interpreter. The model submits code; the interpreter returns output;
the output is injected back into the conversation as a tool response; the model
continues reasoning with the new information. The loop repeats until the model
produces a final answer or exhausts its token budget.

TIR is appropriate for competition mathematics because many olympiad problems
require computation that is feasible for Python but not for pure text
reasoning: enumeration over combinatorial spaces, symbolic algebra with many
terms, high-precision arithmetic, and numerical verification of closed-form
results. A model that can write and observe code is not limited by the
precision of mental arithmetic or the tractability of manual symbolic
manipulation.

Each `SolverAgent` runs its TIR loop inside a persistent `JupyterSandbox` - a
separate kernel process whose state accumulates across turns within one
problem. Variables defined in turn $k$ remain accessible in turn $k+n$.
Functions written in turn 2 can be called in turn 7. This mirrors the
behaviour of an interactive Python session, allowing agents to build up
computation incrementally rather than restarting from scratch on each turn.
Between problems, the sandbox is fully reset. Sandboxes are pre-warmed at
solver startup so kernel initialisation latency is never on the problem hot
path.

The TIR contract is enforced at the model suite level (Block E). Two
implementations are stable: `GptOssModelSuite` uses the Harmony token-ID
completions protocol specific to the `gpt-oss` model family; `Qwen35ModelSuite`
uses standard chat completions with markdown code fences. The remaining four
suites extend `Qwen35ModelSuite` with model-specific answer detection overrides.
The TIR loop in `SolverAgent` is identical regardless of which suite is running.

---

### Agent Diversity

Running $N$ identical agents on the same problem with the same prompt and the
same temperature produces convergence, not diversity. The value of a multi-agent
population depends on the probability that at least one agent explores a
reasoning path that leads to a correct answer - which increases with variance
across agents, not with agent count alone.

Two mechanisms introduce controlled variance without affecting capability:

**Mathematical spirits.** Each agent is assigned a one-sentence persona prefix
drawn from a pool of 16 historical mathematicians (Euler, Gauss, Ramanujan,
Hilbert, and others) at the start of each problem. The spirit is prepended to
the system prompt, nudging the agent toward a different initial orientation.
Two agents with different spirits may approach the same problem via different
mathematical frameworks - one reaching for algebraic manipulation, another for
combinatorial enumeration.

**Temperature sampling.** Each agent's temperature is sampled independently
from a configurable range (`[temp_agent_min, temp_agent_max]` in `CFG`).
Different temperatures produce different sampling paths through the model's
probability distribution. At the default setting (`min == max == 1.0`) all
agents sample identically; widening to `[0.8, 1.2]` increases path diversity
at the cost of occasionally less coherent reasoning in high-temperature agents.

These two sources of diversity are orthogonal and independent. They are
designed to increase variance without introducing capability inequality between
agents - the equal-capacity assumption that underlies the effort-weighted
voting rationale.

---

### Voting Mechanism

When multiple agents have produced answers, the system selects one via
effort-weighted aggregation:

$$\text{weight}(X) = \sum_{\substack{\text{agents with} \\ \text{answer} = X}} \text{response\_length}$$

$$\text{effort\_pct}(X) = \frac{\text{weight}(X)}{\displaystyle\sum_{\text{all answers}} \text{weight}(a)}$$

`response_length` is the total tokens generated across all TIR turns for one
attempt. An agent that produced 6,000 tokens through multiple Python iterations
contributes proportionally more evidence than one that answered in 300 tokens.

The rationale draws on an analogy to human deliberation: when a panel of
equal-capability experts works on a hard problem, the expert who invests more
time and produces a more detailed analysis is generally more trustworthy than
one who answers immediately. Token count is the computational analogue of that
investment. This is a proxy for reasoning depth, not a proof of correctness.
The divergence from classical majority voting is most meaningful on hard
problems, where agents vary widely in how much computation they commit before
producing an answer.

Five stop conditions are evaluated on every incoming result:

| Condition | Trigger | Purpose |
|---|---|---|
| Stop-A | $\geq k_2$ agents agree (raw majority) | Fast exit on easy problems where all agents converge quickly |
| Stop-B | $\geq k_1$ valid answers AND effort share $\geq 0.50$ | Quality gate: sufficient evidence count plus dominant effort concentration |
| Stop-C | $\geq 3$ MetaAgents independently agree | Hard conflict resolution via independent secondary solvers |
| Stop-D | Deadline exceeded | Time budget exhausted; commit best available answer |
| Stop-E | All futures complete naturally | Full population exhaustion without earlier stop |

On easy problems, Stop-A typically fires before Stop-B has accumulated enough
answers to evaluate. On hard problems where the population splits, Stop-C
provides a resolution path that does not depend on the primary population
converging. Stop-D and Stop-E are safety nets rather than expected paths.

---

### Shared Data Structures

Three dataclasses define the data contract between all components. Every piece
of data that crosses a component boundary passes through one of these types.
No ad-hoc dictionaries or loosely typed payloads cross component boundaries.

- **`AttemptResult`**: one record per agent attempt. Carries the agent's
  answer, total tokens generated (`response_length`), Python call and error
  counts, wall-clock duration, and the full serialised conversation
  (BENCHMARK mode only). The `response_length` field is the sole input to
  the effort-weighted voting mechanism; all other fields are diagnostic.

- **`ProblemSummary`**: one record per solved problem. Carries the final answer,
  raw vote distribution, effort distribution, winning effort share, stop reason,
  and solve duration. Written to `problems/{pb_id}.json` in BENCHMARK mode and used
  by `RunTracker` to maintain the rolling average for adaptive deadline
  computation.

- **`RunStats`**: one accumulator across the full 50-problem run. Carries
  cumulative and per-problem timing statistics. Printed as a run summary at the
  end of a BENCHMARK run.

These types are defined in Block C before any solver code and are the only
shared state format used across the `Blackboard`, `Controller`, `SolverAgent`,
`MetaAgent`, and JSON output pipeline.

---

### Dual-Mode Operation

```python
MODE = 'SUBMISSION'   # Kaggle evaluation harness; no conversation traces stored
MODE = 'BENCHMARK'    # reference dataset; correctness labeling; JSON output
```

All solver logic is identical between modes. The mode flag controls three
things only: whether conversation traces are serialised into `AttemptResult.messages`,
whether `AnswerChecker` is instantiated to label each attempt as correct or
incorrect, and which entry point Block N dispatches to.

Three JSON files written in BENCHMARK mode provide the data for empirical
validation:

| Directory | Unit | Key uses |
|---|---|---|
| `attempts/{pb_id}__{agent}__{idx}.json` | Per agent attempt | Per-suite accuracy, effort vs. correctness correlation, `python_errors` distribution |
| `meta/{pb_id}__{META-N}.json` | Per MetaAgent run | MetaAgent contribution to final answer, cross-suite secondary solver performance |
| `problems/{pb_id}.json` | Per problem | Stop reason distribution, `weighted_pct_winner` vs. correctness, timing distribution |

Each file is written atomically as a standalone JSON object, so interrupted
runs produce usable partial output. Load with `json.load()` per file or
glob the directory for batch analysis.

---

### Adaptive Time Budget

$$\text{budget} = \text{clamp}\!\left(\text{time\_left} - (n_{\text{remaining}} - 1) \times \bar{t}_{\text{recent}},\; t_{\min},\; t_{\max}\right)$$

$\bar{t}_{\text{recent}}$ is the rolling average of the five most recent
problem durations. Reserving $(n_{\text{remaining}} - 1) \times \bar{t}_{\text{recent}}$
ensures the current problem cannot consume time that would leave later problems
with less than $t_{\min}$ each.

The self-correcting property: slow problems raise $\bar{t}_{\text{recent}}$,
contracting future budgets and protecting the remaining run. Fast problems
lower it, expanding future budgets and releasing capacity for harder problems
that follow. The floor $t_{\min} = 276\text{s}$ ($276 \times 50 = 13{,}800\text{s}$,
leaving $\approx 3{,}600\text{s}$ of slack against the 17,400s limit) and the
ceiling $t_{\max} = 895\text{s}$ prevent pathological allocation at both ends.

---

### Block Map

| Block | Contents | Primary edit target |
|---|---|---|
| A | Environment setup, package removal, offline wheel installation | Wheel paths, vLLM version |
| B | All imports (post-environment) | - |
| C | `AttemptResult`, `ProblemSummary`, `RunStats`, `JupyterSandbox` | Sandbox timeout, pre-loaded libraries |
| D | `SpecConfig`, `EngineConfig` hierarchy, `InferenceEngineManager` | GPU memory, speculative decoding config |
| E | `ModelSuiteProtocol` (8 abstract methods), six model suites, `make_model_suite()` | New model suite implementation |
| F | `Blackboard`, `RunTracker`, `aggregate_answers()`, `AnswerChecker` | Stop condition thresholds |
| G | `Controller`, `SolverAgent`, `MetaAgent` | Agent loop depth, fallback chain |
| H | `AIMO3Solver`, `build_and_start()`, JSON record builders | - |
| I | `EngineConfig` instances, `ACTIVE_ENGINE_CONFIGS`, agent identity pools | **Model paths, active engines, n_agents** |
| J | `CFG` dataclass | **Timeouts, temperatures, vote thresholds, context budgets** |
| K | Prompt strings, injection cell | **System prompts, preference prompts** |
| L | System initialisation (`set_seed`, `build_and_start`) | - |
| M | `predict()`, `predict_and_benchmark()` | - |
| N | Evaluation gateway (SUBMISSION / BENCHMARK dispatch) | - |

Between experiments, Blocks I, J, and K are the primary edit targets. All
other blocks are read-only unless implementing a structural change to the
architecture.

---

### Score and Architecture

This architecture does not guarantee that competitors will improve scores directly. It removes the implementation
obstacles between a competitor's reasoning about what to try and their ability
to try it. A higher leaderboard score requires better model selection, better
prompting, better stop condition calibration, or better inference acceleration -
work that depends entirely on the skill, effort, and creativity of the
competitor. The architecture provides the infrastructure for that work. It
provides nothing else.

---

### Prerequisites

Four areas of background knowledge are required to read, modify, and extend
this notebook effectively.

**Python concurrency**
The solver runs agents in a `ThreadPoolExecutor`. The Controller's monitoring
loop uses `as_completed()`. The `Blackboard` uses `threading.Lock` and
`threading.Event`. The sandbox pool uses `queue.Queue`. Understanding these
primitives is necessary to reason about thread safety, stop condition timing,
and the ordering of incremental versus final Blackboard posts.
- [Python Threading documentation](https://docs.python.org/3/library/threading.html)
- [Python Concurrency with Futures - Real Python](https://realpython.com/python-concurrency/)
- Coursera: *Python for Everybody* - University of Michigan: https://www.coursera.org/specializations/python
- Python Tutorials: https://www.tutorialspoint.com/python/index.htm

**LLM inference and vLLM**
The inference layer runs on vLLM. Understanding KV cache allocation, continuous
batching, speculative decoding mechanics, and the OpenAI-compatible REST server
is necessary for Block D and Block I configuration. Specific flags that require
calibration per deployment: `--max-num-seqs`, `--gpu-memory-utilization`,
`--kv-cache-dtype`, `--max-model-len`, and `--speculative-config`.
- [vLLM documentation](https://docs.vllm.ai/en/latest/)
- [NVIDIA Deep Learning Institute](https://www.nvidia.com/en-us/training/)
- [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course)
- [Speculative Decoding - vLLM](https://docs.vllm.ai/en/latest/features/spec_decode.html)

**Systems engineering and system design**
The notebook applies a layered component architecture with explicit interface
contracts between components (`ModelSuiteProtocol`, `EngineConfig`,
`AttemptResult`). Understanding abstraction boundaries, separation of concerns,
and the failure modes of concurrent systems is necessary to extend the
architecture without introducing regressions. The SEBOK and NASA SEH provide
a formal vocabulary for reasoning about component interfaces and system
decomposition.
- [NASA Systems Engineering Handbook SP-2016-6105](https://www.nasa.gov/reference/systems-engineering-handbook/)
- [SEBOK - Systems Engineering Body of Knowledge](https://sebokwiki.org/wiki/Guide_to_the_Systems_Engineering_Body_of_Knowledge_(SEBoK))
- [Designing Data-Intensive Applications - Kleppmann](https://www.oreilly.com/library/view/designing-data-intensive-applications/9781491903063/)

**Competition mathematics**
The prompts, TIR protocol, spirit diversity pool, and agent configuration are
calibrated for olympiad-style problems. Understanding the structure of
competition problems - modular arithmetic, combinatorics, algebraic
manipulation, number theory, geometric reasoning - is necessary to interpret
BENCHMARK output meaningfully and to tune prompts toward the problem types
where the current configuration underperforms.
- [Art of Problem Solving](https://artofproblemsolving.com/)
- [MIT OpenCourseWare - Mathematics for Computer Science](https://ocw.mit.edu/courses/6-042j-mathematics-for-computer-science-fall-2010/)
- [Project Euler](https://projecteuler.net/) - computational problems with known answers, useful for local benchmarking before submitting to the competition harness

---

*Architecture v5.0 - Beyond gpt-oss-120b + Harmony TIR - Multi-Agent Blackboard Solver*
*Kaggle AIMO3 - March 2026*


## Block A - Environment Setup

Kaggle GPU notebooks start from a base Docker image built for general-purpose
data science. That image ships with TensorFlow, Keras, scikit-learn, and
Matplotlib pre-installed. These packages are irrelevant to LLM inference, and
several of them pin versions of shared C libraries (`libstdc++`, `protobuf`,
`grpcio`) that vLLM requires at different, incompatible versions.

The consequence is not a clean error. It is a silent installation where pip
reports success, but the running process loads the wrong shared library at
import time and fails mid-execution with a message that points nowhere near
the actual cause.

The first cell strips those packages before any installation runs. Package
removal must precede `set_env()` because pip's dependency resolver reads the
currently installed environment at the start of an install pass. Packages
present at that moment influence what pip decides to install, downgrade, or
leave unchanged. Removing conflicts after the fact does not undo decisions
already made.


### Offline Package Installation

Kaggle inference sessions run without outbound network access. The Python
Package Index is unreachable. `pip install vllm` will fail with a connection
timeout, not a meaningful error message.

The standard pattern for this constraint is to pre-download all required wheel
files into a Kaggle Dataset, mount that dataset as a notebook input, and pass
its local path to pip via `--find-links`. The `--no-index` flag tells pip to
ignore PyPI entirely and resolve packages only from that local directory.
The `set_env()` function in the following cell encodes this pattern.
See: [pip local archives guide](https://pip.pypa.io/en/stable/topics/local-project-installs/).

Three packages require explicit installation.

**vLLM** is the inference server. It manages model weight loading, KV cache
allocation, request batching, and continuous batching scheduling. It exposes an
OpenAI-compatible HTTP API - the same `/v1/chat/completions` and
`/v1/completions` endpoints that the OpenAI Python client speaks - so the rest
of the solver code is decoupled from vLLM internals. The model runs inside
the vLLM subprocess; the solver communicates with it over localhost HTTP.
See: [vLLM documentation](https://docs.vllm.ai/en/latest/) and the
[NVIDIA deep learning institute](https://www.nvidia.com/en-us/training/).

**openai_harmony** is the Harmony encoding layer for the `gpt-oss` model
family. The `gpt-oss` models do not use the standard chat completions format.
They consume conversations as flat token-ID sequences submitted to the
`/completions` endpoint, with routing metadata embedded in the token stream
itself. Harmony handles the rendering of multi-turn conversations into that
format and the parsing of response token IDs back into typed messages. This
package is required only by `GptOssModelSuite`; every other model suite in
this notebook uses standard chat completions and has no dependency on it.

**tiktoken** is the BPE tokenizer used internally by the Harmony encoding
layer. It is a transitive dependency of `openai_harmony` but must be installed
explicitly for one reason: at import time, tiktoken looks for its encoding
cache files on disk. If those files are absent and no cache path is configured,
it attempts a network fetch, which fails in the offline environment. The
`TIKTOKEN_ENCODINGS_BASE` variable in cell A.3 redirects that lookup to
pre-downloaded files shipped alongside the wheel archive. That variable must
be set before tiktoken is imported, which means the encoding files must be
confirmed present beforehand - which is what the verification step in A.3
performs.
See: [tiktoken on PyPI](https://pypi.org/project/tiktoken/).


### Path Configuration and Environment Variables

#### Wheel directory

`WHEEL_DIR` is the mount path of the Kaggle Dataset containing the vLLM wheel
archive. Two versions are provided:

- `vllm-wheel-0-15-1` - the stable release. Default for all `gpt-oss` and
  standard Qwen3.5 experiments.
- `vllm-v16` - a nightly build. Required for Qwen3.5 MTP speculative decoding,
  which depends on internal vLLM APIs introduced after 0.15.1. Treat as
  unstable; nightly builds are not regression-tested and may introduce breaking
  changes between dataset refreshes.

#### Tiktoken verification

The `ls` call on `TIKTOKEN_PATH` is a pre-flight check. If the path resolves
to an empty directory or does not exist, `TIKTOKEN_ENCODINGS_BASE` is about to
be set to a broken value. The import error that follows will not mention the
path variable. Confirming the files are present at this step surfaces the
misconfiguration where it is straightforward to fix.

#### Environment variables

All variables must be set before any import that depends on them. Once Python
imports a module, its initialisation code has already run; setting an
environment variable that the module reads at init time has no retroactive
effect.
See: [Python `os.environ` reference](https://docs.python.org/3/library/os.html#os.environ).

| Variable | Value | Effect |
|---|---|---|
| `TRANSFORMERS_NO_TF` | `'1'` | Suppresses TensorFlow backend initialisation in `transformers`. Without this, `transformers` attempts to import TF at load time and either raises an error or prints deprecation warnings that clutter output. |
| `TRANSFORMERS_NO_FLAX` | `'1'` | Same for the JAX/Flax backend. |
| `TOKENIZERS_PARALLELISM` | `'false'` | The HuggingFace `tokenizers` library uses Rust-based thread parallelism. Inside a `multiprocessing`-based server process like vLLM, forking a process that already has active Rust threads is undefined behavior under POSIX and causes deadlocks in practice. See: [HuggingFace tokenizers docs](https://huggingface.co/docs/tokenizers). |
| `CUDA_VISIBLE_DEVICES` | `'0'` | Pins all CUDA operations to physical GPU 0. Kaggle allocates one GPU per session; explicit restriction avoids driver-level enumeration issues and makes GPU memory accounting unambiguous. |
| `TRITON_PTXAS_PATH` | `/usr/local/cuda/bin/ptxas` | Points the Triton JIT compiler to the CUDA assembler binary. Flash Attention and several vLLM kernels are compiled by Triton on first use. Without this path, Triton searches `PATH` for `ptxas`, which is not guaranteed present in the Kaggle CUDA environment. A missing binary produces kernel compilation failures that surface as opaque CUDA errors mid-inference. See: [Triton documentation](https://triton-lang.org/main/index.html). |
| `TIKTOKEN_ENCODINGS_BASE` | `TIKTOKEN_PATH` | Redirects tiktoken's encoding cache to the pre-downloaded files in the wheel dataset. Without this, tiktoken falls back to a network fetch on first use, which fails silently and only surfaces as an encoding error when the Harmony loader first processes a conversation. |
| `PYTORCH_ALLOC_CONF` | `'expandable_segments:True'` | Switches PyTorch's CUDA memory allocator to expandable segments. The default allocator carves the GPU pool into fixed blocks at startup. Over a run of 50 problems where KV caches are repeatedly allocated and freed, those blocks fragment: the allocator holds memory it cannot reuse for new allocations of different sizes. Expandable segments grow and shrink with actual demand. See: [PyTorch CUDA memory management](https://pytorch.org/docs/stable/notes/cuda.html#memory-management). |
| `VLLM_DISABLE_PAD_FOR_CUDAGRAPH` | `'1'` | Disables vLLM's automatic batch size padding for CUDAGraph compatibility. By default, vLLM rounds every batch up to the nearest power of two so that CUDAGraphs - compiled for fixed batch sizes - can be reused. With 8 concurrent agents generating at different speeds, most actual batch sizes are small and irregular. Rounding up wastes GPU memory proportional to how far each batch falls below the next power of two. |

In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
!sudo ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/local/cuda/lib64/libcuda.so

In [ ]:
import warnings
warnings.simplefilter('ignore')

In [ ]:
import os
import sys
import subprocess

### Offline Package Installation

All packages are installed from a local wheel archive (no internet at inference time).
`vllm`, `openai_harmony`, and `tiktoken` are the three required packages.
`tiktoken` must be installed explicitly so that the `TIKTOKEN_ENCODINGS_BASE` path
is honoured at import time.

In [ ]:
def set_env(wheel_dir: str) -> None:
    """
    Install required packages from an offline wheel archive.

    Args:
        wheel_dir: path to the dataset directory containing a 'wheels/' subdirectory
    """
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install',
            '--no-index',
            '--find-links', f'{wheel_dir}/wheels',
            'vllm',
            'openai_harmony',
            'tiktoken',
        ],
        check=True,
    )

### Path Configuration

Adjust `WHEEL_DIR` to point to your vLLM wheel dataset mount point.

In [ ]:
# ── Wheel archive ────────────────────────────────────────────────────────────
WHEEL_DIR      = '/kaggle/input/datasets/khoinguyennguyen/vllm-wheel-0-15-1'
WHEEL_DIR      = '/kaggle/input/datasets/khoinguyennguyen/vllm-v16' # nighly, unstable, use it if want to try Qwen3.5


TMP_SETUP_DIR  = '/kaggle/working/tmp/setup'
TIKTOKEN_PATH  = f'{WHEEL_DIR}/tiktoken_encodings'

set_env(WHEEL_DIR)

In [ ]:
# Verify tiktoken encoding files are present before setting the env var
subprocess.run(['ls', TIKTOKEN_PATH])

In [ ]:
# ── Transformer / tokenizer flags ────────────────────────────────────────────
os.environ['TRANSFORMERS_NO_TF']          = '1'
os.environ['TRANSFORMERS_NO_FLAX']        = '1'
os.environ['TOKENIZERS_PARALLELISM']      = 'false'

# ── CUDA ──────────────────────────────────────────────────────────────────────
os.environ['CUDA_VISIBLE_DEVICES']        = '0'
os.environ['TRITON_PTXAS_PATH']           = '/usr/local/cuda/bin/ptxas'

# ── tiktoken encoding cache ───────────────────────────────────────────────────
os.environ['TIKTOKEN_ENCODINGS_BASE']     = TIKTOKEN_PATH

# ── PyTorch memory allocator ──────────────────────────────────────────────────
os.environ['PYTORCH_ALLOC_CONF']          = 'expandable_segments:True'

# ── vLLM CUDAGraph padding disable ───────────────────────────────────────────
os.environ['VLLM_DISABLE_PAD_FOR_CUDAGRAPH'] = '1'

# ── FLASHINFER - vllm[flashinfer] required - EXPERIMENTAL ───────────────────────────────────────────
#os.environ["VLLM_ATTENTION_BACKEND"] = "FLASHINFER" 

print('Environment variables set.')


## Block B - Imports

All imports that depend on the environment variables set in Block A are loaded
here, after `set_env()` completes and before any class or function definitions.
The ordering is a hard requirement, not a convention: `openai_harmony` and
`vllm`-adjacent packages are installed by `set_env()` into the running Python
process. Importing them before that call completes raises `ModuleNotFoundError`
because the packages do not yet exist on the path.

The imports are organised into six groups.

#### Standard library

The Python standard library requires no installation. These modules cover the
concurrency model, data structures, and utilities used throughout the solver.

| Module | Role in this notebook |
|---|---|
| `gc` | Manual garbage collection control. Called in `predict()` to suppress GC pauses during inference, where an untimely collection can interrupt a streaming response. |
| `re` | Regular expressions. Used in `Qwen35ModelSuite` and its subclasses to detect ` ```python ` code fences and `\boxed{}` answer markers in streaming text. |
| `math` | Mathematical constants and functions available to agent-generated code via the pre-loaded kernel environment. |
| `time` | Wall-clock timing for deadline enforcement and per-problem duration tracking in `RunTracker`. |
| `queue` | `queue.Queue` is the thread-safe pool that holds pre-warmed `JupyterSandbox` instances. Agents acquire a sandbox with `get(timeout=...)` and return it in a `finally` block. |
| `json` | Serialisation for BENCHMARK mode JSON output and for the `--speculative-config` CLI argument passed to vLLM. |
| `random` | Temperature sampling and spirit assignment per agent. A `random.Random` instance seeded from `cfg.seed` ensures reproducibility across runs with the same configuration. |
| `threading` | `threading.Event` is the stop signal broadcast to all worker threads when a stop condition fires. `threading.Lock` protects the Blackboard and the `meta_answers` list. |
| `contextlib` | `contextlib.suppress` is used in sandbox cleanup to silently swallow `queue.Empty` and kernel shutdown exceptions that are expected during teardown. |
| `dataclasses` | `@dataclass` and `field()` define all typed containers: `AttemptResult`, `ProblemSummary`, `RunStats`, `CFG`, and all `EngineConfig` subclasses. |
| `typing` | `Optional` and `Any` type annotations. Used throughout for static analysis compatibility and to make the abstract interface of `ModelSuiteProtocol` explicit. |
| `collections` | `Counter` and `defaultdict` in `aggregate_answers()` for effort-weighted vote accumulation. |
| `concurrent.futures` | `ThreadPoolExecutor` runs `SolverAgent` and `MetaAgent` instances concurrently. `as_completed()` drives the Controller's monitoring loop, triggering stop condition evaluation as each future resolves. |

See: [Python standard library reference](https://docs.python.org/3/library/).

#### Data libraries

`pandas` and `polars` are both present because the notebook straddles two
interfaces. The Kaggle evaluation harness delivers problems and expects answers
as `polars.DataFrame` objects - `polars` is the harness's native format.
Internally, `pandas` is used for reading the reference CSV in BENCHMARK mode
and for the results DataFrame returned by `predict_and_benchmark()`. Both
libraries are pre-installed in the Kaggle base image and require no wheel
installation.

See: [Polars documentation](https://docs.pola.rs/) and
[Pandas documentation](https://pandas.pydata.org/docs/).

#### OpenAI client

`openai.OpenAI` is the HTTP client used to communicate with the vLLM server
over localhost. vLLM exposes an OpenAI-compatible REST API; the `OpenAI` client
speaks to it identically to how it would speak to the OpenAI cloud API, with
the `base_url` pointed at `http://localhost:{port}/v1`. Both
`GptOssModelSuite` (via `/v1/completions`) and `Qwen35ModelSuite` (via
`/v1/chat/completions`) use this client. No API key is required for a local
vLLM server.

See: [OpenAI Python client](https://github.com/openai/openai-python) and
[vLLM OpenAI-compatible server documentation](https://docs.vllm.ai/en/latest/serving/openai_compatible_server.html).

#### Harmony TIR protocol

The `openai_harmony` symbols are imported at the module level here rather than
inside `GptOssModelSuite.__init__()` for clarity, but only the suite class
itself uses them. The relevant symbols and their roles:

| Symbol | Role |
|---|---|
| `HarmonyEncodingName` | Enum identifying which encoding variant to load. `HARMONY_GPT_OSS` is the variant for the `gpt-oss` model family. |
| `load_harmony_encoding` | Loads the encoding object that handles tokenisation, conversation rendering, and response parsing. |
| `SystemContent` | Builder for the system message, carrying model identity text, reasoning effort level, and tool namespace configuration. |
| `ReasoningEffort` | Enum controlling the model's reasoning depth. Set to `HIGH` for all solver agents. |
| `ToolNamespaceConfig` | Declares the Python tool namespace to the model: its name (`'python'`), a description of the available libraries, and an empty tool list. |
| `Author`, `Message`, `Role`, `TextContent`, `Conversation` | Core message types used to construct and extend the conversation object passed to the encoding layer. |

See: the `openai_harmony` package documentation is not publicly hosted;
consult the wheel archive's bundled README for the full API reference.

#### Transformers seed utility

`set_seed` from the HuggingFace `transformers` library sets the random seed
across Python's `random` module, NumPy, and PyTorch simultaneously. It is
called once in Block L before `build_and_start()` to make agent temperature
sampling and spirit assignment reproducible.

See: [HuggingFace `set_seed` reference](https://huggingface.co/docs/transformers/main_classes/utilities#transformers.set_seed).

#### Jupyter kernel manager

`jupyter_client.KernelManager` manages the lifecycle of an individual Jupyter
kernel process: spawning it, connecting to its ZMQ message sockets, and
shutting it down. `JupyterSandbox` (Block C) wraps one `KernelManager` per
agent slot and uses it to execute Python code generated by the model during
TIR turns. The kernel runs as a separate OS process; code execution is
isolated from the notebook kernel and from other agent sandboxes.

See: [jupyter_client documentation](https://jupyter-client.readthedocs.io/en/stable/).

#### Kaggle evaluation gateway

`kaggle_evaluation.aimo_3_inference_server` provides `AIMO3InferenceServer`,
the Kaggle-supplied harness that delivers problems to `predict()` and collects
answers during a submission run. It is imported here to ensure any import-time
side effects (server socket initialisation, signal handler registration) occur
before the model weights are loaded in Block L. In BENCHMARK mode this import
is unused but harmless.

See: [Kaggle evaluation utilities](https://github.com/Kaggle/kaggle-api).

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import gc
import re
import math
import time
import queue
import json
import random
import threading
import contextlib
from dataclasses import dataclass, field
from typing import Optional, Any
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import concurrent

from __future__ import annotations

import json
import os
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional

# ── Third-party: data ─────────────────────────────────────────────────────────
import pandas as pd
import polars as pl

# ── OpenAI client ─────────────────────────────────────────────────────────────
from openai import OpenAI

# ── Harmony TIR protocol (requires openai_harmony installed by set_env) ───────
from openai_harmony import (
    HarmonyEncodingName,
    load_harmony_encoding,
    SystemContent,
    ReasoningEffort,
    ToolNamespaceConfig,
    Author,
    Message,
    Role,
    TextContent,
    Conversation,
)

# ── Transformers seed utility ─────────────────────────────────────────────────
from transformers import set_seed

# ── Jupyter kernel manager (sandbox pool) ────────────────────────────────────
from jupyter_client import KernelManager

# ── Kaggle evaluation gateway ─────────────────────────────────────────────────
import kaggle_evaluation.aimo_3_inference_server

print('All imports successful.')


## Block C - Data Structures and Execution Sandbox

This block defines two independent components. Neither has any dependency on
model-specific code, on vLLM, or on the Harmony protocol. Both can be read,
tested, and reasoned about in isolation from the rest of the notebook.

#### Data structures

Three dataclasses serve as the typed containers through which all information
flows between components. No logic lives in these classes; they are pure records.
Every piece of data that crosses a component boundary - from a `SolverAgent`
to the `Blackboard`, from the `Blackboard` to `aggregate_answers()`, from the
solver to the JSON output files - is carried in one of these three types.

**`AttemptResult`** is one record per agent attempt. It is posted to the
`Blackboard` twice: once incrementally when an answer is first detected
mid-stream (`is_incremental=True`), and once on attempt completion with the
full accumulated state (`is_incremental=False`). The `Controller` deduplicates
by `(agent_name, attempt_index)`, keeping only the final post. Fields are
grouped into five concerns: identity (which agent, which problem), outcome
(answer integer, raw final text), statistics (total tokens generated, Python
call count, error count, wall duration), conversation trace (populated only in
BENCHMARK mode), and correctness label (populated only in BENCHMARK mode by
`AnswerChecker`).

The `response_length` field - total tokens generated across all TIR turns for
this attempt - is the effort proxy used by `aggregate_answers()` for
token-investment weighted voting. It is the only field in the data layer with
a direct bearing on the voting mechanism.

**`ProblemSummary`** is one record per solved problem. It captures the
aggregate outcome: final answer, vote distribution (`answer_distribution`),
effort distribution (`effort_distribution`), winning effort share
(`weighted_pct_winner`), stop reason, solve duration, and correctness count.
`RunTracker` in Block F accumulates these summaries across the full 50-problem
run and uses `solve_duration_sec` to maintain the rolling average that drives
adaptive deadline computation.

**`RunStats`** is a single accumulator across the entire run. It tracks total
problems seen, total correct (BENCHMARK only), cumulative and average solve
time, and fastest/slowest problem durations. It is updated by `RunTracker` on
each `record()` call and printed as a summary at the end of a BENCHMARK run.

#### Execution sandbox

`JupyterSandbox` wraps a persistent Jupyter kernel process that agents use to
execute Python code during TIR loops. A pre-warmed pool of these sandboxes is
initialised at solver startup in `build_and_start()` - one per agent slot plus
one per possible concurrent MetaAgent - so that kernel startup latency is never
on the hot path during inference.

See: [jupyter_client kernel management documentation](https://jupyter-client.readthedocs.io/en/stable/kernels.html).


### JupyterSandbox - Persistent Jupyter Kernel Wrapper

An agent acquires a `JupyterSandbox` from the shared pool at the start of an
attempt, uses it for all Python code executions across that attempt's TIR turns,
then resets and returns it to the pool in a `finally` block. Thread safety is
enforced by the pool itself: `queue.Queue` ensures each sandbox is owned by
exactly one thread at any given time.

#### Statefulness within a problem

Kernel state accumulates across TIR turns within a single attempt. A variable
defined in turn 2 is accessible in turn 5. A helper function written in turn 1
can be called in turn 8. This mirrors the way a mathematician uses an
interactive Python session: intermediate results are built up incrementally,
not recomputed from scratch on each turn.

This statefulness is a deliberate design choice. It allows agents to express
multi-step computations naturally across turns without compressing the entire
computation into a single code block - a compression that would make the
reasoning trace harder to follow and the code more error-prone.

#### Reset between problems

`reset()` executes `%reset -f` in the kernel (clearing all user-defined
variables, functions, and module aliases) and then reimports the standard
mathematical libraries. This ensures that one agent's computation on problem
$n$ cannot contaminate another agent's kernel state on problem $n+1$. The
reset is called in the `finally` block of each agent's attempt loop, before
the sandbox is returned to the pool.

#### Port allocation

Each kernel requires five ZMQ sockets: shell, IOPub, stdin, heartbeat, and
control. The class-level `_next_port` counter, protected by a `threading.Lock`,
allocates five consecutive ports per kernel starting at 50000. The lock
prevents port collisions when kernels are initialised in parallel during
`_init_sandbox_pool()`.

See: [ZMQ messaging in Jupyter](https://jupyter-client.readthedocs.io/en/stable/messaging.html).

#### Pre-loaded libraries

Every kernel is initialised with the following libraries imported and ready,
so that agent-generated code can use them without an explicit import statement:

| Library | Version note | Purpose in agent code |
|---|---|---|
| `math` | standard library | Trigonometric functions, logarithms, integer operations (`math.gcd`, `math.comb`). |
| `numpy` | third-party | Numerical arrays, linear algebra, fast vectorised computation. |
| `sympy` | third-party | Symbolic algebra: polynomial factorisation, equation solving, combinatorics, number theory. |
| `mpmath` | third-party, `dps=64` | Arbitrary-precision arithmetic. `mpmath.mp.dps = 64` sets 64 decimal places of precision, sufficient for competition problems that require high-precision floating-point intermediate results. |
| `itertools` | standard library | Combinatorial iterators: `combinations`, `permutations`, `product`. |
| `collections` | standard library | `Counter`, `defaultdict`, `deque` for combinatorial enumeration and graph problems. |

Agents may import additional libraries within their code; these six are
pre-loaded because they cover the vast majority of AIMO3 problem types and
their absence would force agents to write boilerplate imports at the start of
every code block.

#### Execution model

`execute(code, timeout)` submits the code string to the kernel via the
`shell` channel, then drains the `IOPub` channel for `stream`, `error`,
`execute_result`, and `display_data` messages until the `execute_reply` arrives.
`stdout` and `stderr` (including formatted tracebacks) are concatenated and
returned as a plain string to the calling agent. On timeout, the kernel is
interrupted via `interrupt_kernel()` and an `[ERROR]` sentinel is returned.

The returned string is injected into the agent's conversation as a tool
response message, closing the think-code-observe loop of one TIR turn.

See: [Jupyter messaging protocol](https://jupyter-client.readthedocs.io/en/stable/messaging.html)
and [IPython kernel architecture](https://ipython.readthedocs.io/en/stable/overview.html).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA STRUCTURES
# Pure typed containers. No logic. No side effects.
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class AttemptResult:
    """
    One record per agent attempt.

    Created by SolverAgent._process_attempt() and posted to the Blackboard
    twice per attempt: once incrementally when an answer is first detected
    mid-stream (is_incremental=True), and once on attempt completion
    (is_incremental=False) with the final accumulated state.

    The Controller always reads from Blackboard snapshots and deduplicates
    by (agent_name, attempt_index), keeping the last (final) post.
    """

    # ── Identity ──────────────────────────────────────────────────────────────
    agent_name:            str
    agent_suite:           str            # 'gpt-oss' | 'qwen35' | 'deepseek'
    attempt_index:         int
    pb_id:                 str

    # ── Outcome ───────────────────────────────────────────────────────────────
    answer:                Optional[object]  # int (0–99999) or str (BENCHMARK non-integer); None if not found
    answer_raw:            Optional[str]  # raw boxed content string (BENCHMARK non-integer)
    answer_complete:       bool           # True iff answer is not None
    final_text:            str            # raw last assistant turn text

    # ── Statistics ────────────────────────────────────────────────────────────
    response_length:       int            # total tokens generated (effort proxy)
    python_calls:          int
    python_errors:         int
    attempt_duration_sec:  float

    # ── Conversation trace ────────────────────────────────────────────────────
    # Populated in BENCHMARK mode only. Empty list in SUBMISSION mode.
    messages:              list           # [{role, content, channel}, ...]

    # ── Correctness (BENCHMARK only) ──────────────────────────────────────────
    correct:               Optional[bool] # None = SUBMISSION or not yet evaluated
    eval_method:           Optional[str]  # 'exact_match'|'llm_judge'|'no_answer'|...

    # ── Post type ─────────────────────────────────────────────────────────────
    # True  = posted mid-loop when answer first detected (partial state)
    # False = posted on attempt completion (full final state)
    is_incremental:        bool = False


@dataclass
class ProblemSummary:
    """
    One record per solved problem. Written to RunTracker and to problems/{pb_id}.json.

    answer_distribution captures the raw vote count per answer.
    effort_distribution captures total tokens generated per answer.
    weighted_pct_winner is the effort share of the winning answer (0.0–1.0).
    stop_reason records which of the five stop conditions terminated the solve loop.
    """

    pb_id:                 str
    pb_text_len:           int            # character count of problem text
    n_agents_fired:        int
    n_valid_answers:       int            # agents that returned a parsable answer
    n_correct:             Optional[int]  # BENCHMARK only; None in SUBMISSION
    final_answer:          object            # int or str (BENCHMARK non-integer)
    expected_answer:       Optional[str]
    solve_duration_sec:    float
    stop_reason:           str            # 'stop_A'|'stop_B'|'stop_C'|'stop_D'|'stop_E'|'fallback_*'
    answer_distribution:   dict           # {answer_int: raw_vote_count}
    effort_distribution:   dict           # {answer_int: total_tokens}
    weighted_pct_winner:   float          # effort share of final_answer
    timestamp:             float          # wall time at completion


@dataclass
class RunStats:
    """
    Single accumulator across the full dataset run.

    avg_solve_time_sec is continuously updated by RunTracker and used by
    AIMO3Solver._compute_deadline() for adaptive per-problem time allocation.
    """

    total_problems_seen:    int   = 0
    total_problems_correct: int   = 0    # BENCHMARK only
    total_solve_time_sec:   float = 0.0
    avg_solve_time_sec:     float = 0.0
    fastest_problem_sec:    float = float('inf')
    slowest_problem_sec:    float = 0.0
    run_start_time:         float = 0.0  # set at Blackboard initialisation


# ══════════════════════════════════════════════════════════════════════════════
# SOLVER CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════

# Returned by solve_problem() on error/timeout instead of 0, which is a valid
# mathematical answer.  Clamped to 0 at the submission boundary (predict()).
ERROR_SENTINEL = -1

# Number of extra Jupyter sandboxes beyond the theoretical minimum
# (total_agents + n_meta).  Absorbs timing jitter during sandbox release.
SANDBOX_POOL_MARGIN = 2


def _elapsed(blackboard) -> str:
    """Format seconds elapsed since current problem started, for log prefixes."""
    return f'[{time.time() - blackboard.pb_start:6.1f}s]'
    
def _elapsed_duration_nb(blackboard) -> str:
    """Format seconds elapsed since current problem started, for log prefixes."""
    return time.time() - blackboard.pb_start

### JupyterSandbox — Persistent Jupyter Kernel Wrapper

Each agent acquires a `JupyterSandbox` from the shared pool at the start of an
attempt, uses it for all Python code executions across that attempt's TIR turns,
then resets and returns it to the pool in a `finally` block.

**Statefulness within a problem.** Kernel state is preserved across TIR turns within
a single attempt: variables defined in earlier turns remain accessible in later ones.
This allows agents to compute intermediate values, store them, and reference them in
subsequent reasoning steps — mirroring how a human mathematician would use an
interactive Python session.

**Reset between attempts.** `reset()` calls `%reset -f` and reimports the standard
mathematical libraries, ensuring that one agent's computation cannot contaminate
another agent's kernel state.

**Port allocation.** The class-level `_next_port` counter assigns five consecutive
ZMQ ports per kernel starting at 50000, using a thread lock to prevent collisions
when kernels are initialised in parallel.

**Pre-loaded libraries.** Every kernel starts with `math`, `numpy`, `sympy`,
`mpmath` (precision set to 64 decimal places), `itertools`, and `collections`
available without an explicit import in agent-generated code.

In [ ]:
# Number of extra Jupyter sandboxes beyond the theoretical minimum
# (total_agents + n_meta).  Absorbs timing jitter during sandbox release.
SANDBOX_POOL_MARGIN = 2

class JupyterSandbox:
    """
    Persistent Jupyter kernel. One instance per worker slot.
    Model-suite-agnostic — used identically by SolverAgent and MetaAgent.

    Thread safety: each sandbox is owned by exactly one thread at a time.
    The pool (queue.Queue) in AIMO3Solver enforces single-owner access.
    """

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list:
        """Allocate `count` consecutive ZMQ ports. Thread-safe."""
        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count
            return ports

    def __init__(self, timeout: float) -> None:
        self._default_timeout = timeout
        self._owns_kernel     = False
        self._client          = None
        self._km              = None

        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION']   = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT']   = '0'
        env['JUPYTER_PLATFORM_DIRS']            = '1'
        env['PYTHONWARNINGS']                   = 'ignore'
        env['MPLBACKEND']                       = 'Agg'

        self._km              = KernelManager()
        self._km.shell_port   = ports[0]
        self._km.iopub_port   = ports[1]
        self._km.stdin_port   = ports[2]
        self._km.hb_port      = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(
            env=env,
            extra_arguments=['--Application.log_level=CRITICAL'],
        )

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=timeout)
        self._owns_kernel = True

        # Pre-load standard mathematical libraries in every kernel
        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import mpmath\n'
            'import itertools\n'
            'import collections\n'
            'mpmath.mp.dps = 64\n'
        )

    # ── Core execution ────────────────────────────────────────────────────────

    def execute(self, code: str, timeout: float = None) -> str:
        """
        Execute `code` in the kernel. Return stdout as a plain string.

        On timeout: interrupt the kernel and return an [ERROR] string.
        stderr (including tracebacks) is appended to stdout when present.
        """
        client            = self._client
        effective_timeout = timeout or self._default_timeout

        msg_id = client.execute(
            code,
            store_history=True,
            allow_stdin=False,
            stop_on_error=False,
        )

        stdout_parts: list = []
        stderr_parts: list = []
        start_time = time.time()

        while True:
            if time.time() - start_time > effective_timeout:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {effective_timeout}s'

            try:
                msg = client.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content  = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')
                if content.get('name') == 'stdout':
                    stdout_parts.append(text)
                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                stderr_parts.append(
                    self._format_error(content.get('traceback', []))
                )

            elif msg_type in ('execute_result', 'display_data'):
                text = content.get('data', {}).get('text/plain')
                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to display results.'

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _format_error(self, traceback: list) -> str:
        """Strip ANSI escape codes. Drop non-user ipython-input frames."""
        clean = []
        for frame in traceback:
            frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)
            if 'File \"' in frame and 'ipython-input' not in frame:
                continue
            clean.append(frame)
        return ''.join(clean)

    def reset(self) -> None:
        """Clear all kernel variables and reimport standard mathematical libraries."""
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import mpmath\n'
            'import itertools\n'
            'import collections\n'
            'mpmath.mp.dps = 64\n'
        )

    def close(self) -> None:
        """Stop ZMQ channels and shut down the kernel process."""
        _suppress = getattr(contextlib, 'suppress', None)
        if _suppress is None:
            return
        with _suppress(Exception):
            if self._client:
                self._client.stop_channels()
        if self._owns_kernel and self._km is not None:
            with _suppress(Exception):
                self._km.shutdown_kernel(now=True)
            with _suppress(Exception):
                self._km.cleanup_resources()

    def __del__(self) -> None:
        self.close()


## Block D - Inference Engine Layer

This block defines the hardware boundary of the solver. It encapsulates
everything that varies between model deployments: which model is loaded, on
which port, with what GPU memory budget, and with what speculative decoding
configuration. Nothing here describes how a model reasons or constructs a
conversation. That is the sole domain of Block E.

The separation is a deliberate constraint. A practitioner tuning stop
conditions, temperatures, or context budgets in Block J should never need to
open this block. A practitioner changing the GPU memory split for a dual-suite
experiment should never need to open Block J. The two configuration domains are
orthogonal by design.

Three classes constitute this layer.

**`SpecConfig`** is a dataclass that encodes a speculative decoding
configuration and serialises it to the single JSON flag that vLLM accepts since
version 0.15: `--speculative-config '<json>'`. All speculative decoding
methods - Eagle3 draft model, simpler draft model, n-gram prompt lookup, and
Qwen3.5 built-in MTP head - are expressed as `SpecConfig` instances and produce
the same CLI flag. Four named factory functions (`eagle3_spec`, `draft_model_spec`,
`ngram_spec`, `mtp_spec`) cover the configurations used in this notebook and
mirror the experiment presets in the reference benchmark notebook.

**`EngineConfig`** is an abstract base class. One concrete subclass instance
describes one vLLM server process. Six subclasses are defined, two stable and
four experimental, corresponding to the six model suites in Block E. The two
abstract methods `build_server_cmd()` and `build_extra_body()` are the only
points where subclasses diverge; all other server management logic is shared
and lives in `InferenceEngineManager`.

**`InferenceEngineManager`** owns the complete lifecycle of one vLLM server
process: weight preloading into OS page cache, subprocess launch, readiness
polling, and graceful shutdown. It exposes a single thread-safe OpenAI client
shared by all agents on that engine.

See: [vLLM engine arguments reference](https://docs.vllm.ai/en/latest/serving/engine_args.html).


### EngineConfig - Per-Server Configuration Hierarchy

`EngineConfig` is an abstract base class decorated with `@dataclass`. One
concrete subclass instance describes one vLLM server process. The two abstract
methods `build_server_cmd()` and `build_extra_body()` are the only points where
subclasses diverge; all server lifecycle logic is shared and lives in
`InferenceEngineManager`.

Fields are grouped into four concerns: identity, server parameters, sampling
defaults, and prompts.

#### Identity

`suite_id` is the label used in log output and JSON records.
`served_model_name` is passed to `--served-model-name` and used as the `model`
parameter in every API call.

#### Server parameters

| Field | vLLM flag | Notes |
|---|---|---|
| `model_path` | `--model` | Absolute path to the model directory on the Kaggle dataset mount. |
| `port` | `--port` | Localhost port. Multi-suite configurations use consecutive ports (8000, 8001, ...). |
| `n_agents` | - | Number of concurrent `SolverAgent` instances for this engine. Not a vLLM flag; used by `build_and_start()` for sandbox pool sizing. |
| `batch_size` | `--max-num-seqs` | Maximum sequences per vLLM scheduling step. Reduce to 4-16 in speculative decoding mode to avoid GPU OOM. |
| `kv_cache_dtype` | `--kv-cache-dtype` | FP8 KV cache halves memory footprint versus FP16. Set to `auto` for Gemma3, where FP8 is numerically unstable. |
| `dtype` | `--dtype` | Weight dtype. Override to `bfloat16` for Gemma3. |
| `gpu_memory_utilization` | `--gpu-memory-utilization` | Fraction of GPU memory reserved for vLLM. In multi-suite configurations, the sum across all active engines must leave headroom for CUDA kernels and system overhead. |
| `context_tokens` | `--max-model-len` | Maximum sequence length (prompt + generation). Sets KV cache allocation size. |
| `tensor_parallel` | `--tensor-parallel-size` | Fixed at 1 for single-GPU Kaggle sessions. |
| `enable_prefix_cache` | `--enable-prefix-caching` | Reuses KV cache across requests sharing a common prefix (e.g. identical system prompts). |
| `async_scheduling` | `--async-scheduling` | Enables vLLM's async continuous batching scheduler. Recommended for streaming workloads. |
| `trust_remote_code` | `--trust-remote-code` | Required for Qwen2.5/3 and DeepSeek-R1-Qwen tokenizers. Not needed for Llama or Gemma3. |
| `language_model_only` | `--language-model-only` | Required for Qwen3.5 with vLLM; disables unused multimodal components. |

See: [vLLM engine arguments reference](https://docs.vllm.ai/en/latest/serving/engine_args.html).

#### Sampling defaults and logprobs

`min_p`, `top_p`, and `top_logprobs` are forwarded to every API call via
`build_extra_body()`. Keeping them on `EngineConfig` rather than in `CFG`
allows different model suites to use different sampling defaults without
sharing a global configuration.

See: [vLLM sampling parameters](https://docs.vllm.ai/en/latest/dev/sampling_params.html).

#### Speculative decoding

`spec_config` accepts a `SpecConfig` instance or `None`. When present,
`_base_vllm_cmd()` appends `--speculative-config '<json>'` to the server
command. The `use_spec` property exposes the boolean.

Speculative decoding generates $n$ draft tokens in parallel and verifies them
in a single target model forward pass. When the draft is accurate, throughput
scales near-linearly with $n$ at no change to the output distribution.

| Method | Factory | Draft source | Stability |
|---|---|---|---|
| `eagle3` | `eagle3_spec(path, n)` | Separately trained Eagle3 model; 5 community variants available for `gpt-oss-120b` | Experimental |
| `draft_model` | `draft_model_spec(path, n)` | Smaller model of the same family | Experimental |
| `ngram` | `ngram_spec(n)` | N-gram lookup over the prompt; no external model | Available |
| `qwen3_next_mtp` | `mtp_spec(n)` | Built-in MTP head in Qwen3.5; no external model | Stable |

See: [speculative decoding in vLLM](https://docs.vllm.ai/en/latest/features/spec_decode.html)
and [github.com/juemifuji/eagle3-aimo3](https://github.com/juemifuji/eagle3-aimo3).

#### Prompts

`system_prompt`, `tool_prompt`, and `preference_prompt` are fields on
`EngineConfig`, initialised to empty strings. They are populated in the prompt
injection cell of Block K. Separating prompt content from class definition
means prompts can be edited without touching the class hierarchy, and different
engine configs in the same run can carry different prompts independently.

#### The six concrete subclasses

| Class | Backbone | `trust_remote_code` | `language_model_only` | `kv_cache_dtype` | `dtype` | Extra flag |
|---|---|---|---|---|---|---|
| `GptOssEngineConfig` | GPT-OSS | False | False | `fp8_e4m3` | `auto` | - |
| `Qwen35EngineConfig` | Qwen3.5 | True | True | `fp8_e4m3` | `auto` | - |
| `DeepSeekR1QwenEngineConfig` | Qwen2.5/3 | True | False | `fp8_e4m3` | `auto` | - |
| `DeepSeekR1LlamaEngineConfig` | Llama3.1/3.3 | False | False | `fp8_e4m3` | `auto` | - |
| `OpenReasoningEngineConfig` | Qwen2.5 | True | False | `fp8_e4m3` | `auto` | - |
| `Gemma3EngineConfig` | Gemma3 | False | False | `auto` | `bfloat16` | `--enforce-eager` |

`Gemma3EngineConfig` appends `--enforce-eager` to disable CUDAGraph compilation,
which produces graph capture errors on the Gemma3 architecture.

Only GptOssEngineConfig and Qwen35EngineConfig are stable, other EngineConfig subclasses are experimental.


### InferenceEngineManager - Server Lifecycle

`InferenceEngineManager` manages one vLLM server process from startup to
shutdown. One instance is created per entry in `ACTIVE_ENGINE_CONFIGS` by
`build_and_start()` in Block H.

#### Weight preloading

Before launching the vLLM subprocess, `_preload_weights()` reads all model
files into the OS page cache by walking the model directory and reading each
file in 1 GB chunks. If a `SpecConfig` with a draft model path is present, the
draft model directory is preloaded as well.

The motivation is latency reduction at the GPU level. vLLM loads model weights
from disk during server startup. If the files are cold in the page cache, this
loading is bounded by disk read throughput. If the files are warm in the page
cache (already in RAM), loading is bounded by the much faster RAM-to-GPU PCIe
transfer. On Kaggle's NVMe storage, the difference between cold and warm load
times for a 120B-parameter model can exceed two minutes.

Preloading uses a `ThreadPoolExecutor` with `cfg.workers` threads for parallel
file reads, amortising the I/O wait across multiple files simultaneously.

#### Server launch and readiness

`_start_server()` constructs the full vLLM command via `ec.build_server_cmd(sys.executable)`
and launches it as a `subprocess.Popen` with `start_new_session=True`, so that
the vLLM process survives independent of the notebook kernel's process group.
All server stdout and stderr are redirected to `vllm_server_{port}.log` for
post-hoc inspection.

`_wait_for_ready()` polls the server's `/v1/models` endpoint at one-second
intervals until it returns HTTP 200 or the `cfg.server_timeout` limit is
reached. If the process exits before returning 200, the poll loop detects the
dead process and raises immediately rather than waiting for the full timeout.

#### Shared OpenAI client

The `OpenAI` client is created once in `_create_client()` with `base_url`
pointing to `http://0.0.0.0:{port}/v1` and `timeout=cfg.session_timeout`.
The Python `openai` client is thread-safe; all `SolverAgent` and `MetaAgent`
instances that use this engine share the same client object without any locking
at the application level.

See: [OpenAI Python client thread safety note](https://github.com/openai/openai-python#async-usage).

#### Shutdown

`stop()` calls `server_process.terminate()` (SIGTERM), waits up to 10 seconds
for a clean exit, and falls back to `kill()` (SIGKILL) if the process does not
respond. The log file handle is closed after the process exits. `stop()` is
registered with `atexit` in `build_and_start()` to ensure it runs even if the
notebook kernel is interrupted.

If zombie vLLM processes persist after `stop()` - a known failure mode when
vLLM worker subprocesses hold CUDA contexts independently of the parent process -
call `nuke_gpu_memory()` in Block L before re-running Block L. See Block L
for the full remediation procedure.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SPEC CONFIG — Unified speculative decoding configuration
# All methods produce: --speculative-config '<json_string>'
# Source: benchmark.ipynb EXPERIMENT_CONFIGS + _start_server()
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class SpecConfig:
    """
    Unified speculative decoding configuration.

    All speculative methods in vLLM ≥ 0.15 use a single CLI flag pair:
        --speculative-config '<json>'
    This class builds that JSON payload regardless of method.

    Methods supported:
        'eagle3'          — Eagle3 draft model (separate model directory)
        'draft_model'     — smaller model as simple draft (e.g. gpt-oss-20b)
        'ngram'           — n-gram prompt lookup table, no separate model
        'qwen3_next_mtp'  — Qwen3.5 built-in MTP head, no separate model
    """

    method:                     str
    num_speculative_tokens:     int            = 5
    # Eagle3 / draft_model fields
    model:                      Optional[str]  = None   # draft model path
    draft_tensor_parallel_size: int            = 1
    parallel_drafting:          Optional[bool] = None   # eagle3 amazon_p variant
    # ngram fields
    prompt_lookup_max:          Optional[int]  = None
    prompt_lookup_min:          Optional[int]  = None

    def _payload(self) -> dict:
        d: dict = {
            'method':                 self.method,
            'num_speculative_tokens': self.num_speculative_tokens,
        }
        if self.model is not None:
            d['model']                      = self.model
            d['draft_tensor_parallel_size'] = self.draft_tensor_parallel_size
        if self.parallel_drafting is not None:
            d['parallel_drafting'] = self.parallel_drafting
        if self.prompt_lookup_max is not None:
            d['prompt_lookup_max'] = self.prompt_lookup_max
        if self.prompt_lookup_min is not None:
            d['prompt_lookup_min'] = self.prompt_lookup_min
        return d

    def to_cli_flags(self) -> list:
        """Return ['--speculative-config', '<json>'] — identical for all methods."""
        return ['--speculative-config', json.dumps(self._payload())]

    def draft_model_path(self) -> Optional[str]:
        """
        Return the draft model directory path for methods that require a
        separate model to preload (eagle3, draft_model).
        Returns None for ngram and qwen3_next_mtp.
        """
        return self.model if self.method in ('eagle3', 'draft_model') else None


# ── Named factory helpers ─────────────────────────────────────────────────────
# Mirror benchmark.ipynb EXPERIMENT_CONFIGS for drop-in compatibility.

def eagle3_spec(model_path: str, n: int = 5, parallel: bool = False) -> SpecConfig:
    """Eagle3 draft model. parallel=True for the amazon_p parallel-drafting variant."""
    return SpecConfig(
        method                     = 'eagle3',
        num_speculative_tokens     = n,
        model                      = model_path,
        draft_tensor_parallel_size = 1,
        parallel_drafting          = True if parallel else None,        
    )

def draft_model_spec(model_path: str, n: int = 5) -> SpecConfig:
    """Smaller model (e.g. gpt-oss-20b) as draft model."""
    return SpecConfig(
        method                     = 'draft_model',
        num_speculative_tokens     = n,
        model                      = model_path,
        draft_tensor_parallel_size = 1,
    )

def ngram_spec(n: int = 5, lookup_max: int = 4, lookup_min: int = 2) -> SpecConfig:
    """N-gram prompt lookup — no draft model required."""
    return SpecConfig(
        method                 = 'ngram',
        num_speculative_tokens = n,
        prompt_lookup_max      = lookup_max,
        prompt_lookup_min      = lookup_min,
    )

def mtp_spec(n: int = 5) -> SpecConfig:
    """Qwen3.5 built-in MTP head — no separate model files required."""
    return SpecConfig(method='qwen3_next_mtp', num_speculative_tokens=n)

### EngineConfig — Per-Server Configuration Hierarchy

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from typing import Optional


# ══════════════════════════════════════════════════════════════════════════════
# ENGINE CONFIG — MINIMAL BASE
# Only fields the rest of the system actually reads.
# All vLLM server flags live in each suite's build_server_cmd().
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class EngineConfig:
    """
    Minimal base for all engine configs.

    The rest of the system (Controller, Solver, InferenceEngineManager) reads
    ONLY these fields. Everything else (dtype, kv_cache_dtype, context_tokens,
    trust_remote_code, etc.) is internal to each suite's build_server_cmd().
    """

    # ── Identity ──────────────────────────────────────────────────────────────
    suite_id:               str   = ''
    served_model_name:      str   = ''

    # ── Paths + network ───────────────────────────────────────────────────────
    model_path:             str   = ''
    port:                   int   = 8000
    n_agents:               int   = 8
    batch_size:             int   = 8       # safe default — was 64, caused OOM on missed override
    gpu_memory_utilization: float = 0.96

    # ── Speculative decoding (read by restart_if_dead, preload) ───────────────
    spec_config:            Optional[SpecConfig] = None

    # ── Prompts (injected by Block I, read by suites + MetaAgent) ─────────────
    system_prompt:          str   = ''
    tool_prompt:            str   = ''
    preference_prompt:      str   = ''

    @property
    def use_spec(self) -> bool:
        return self.spec_config is not None

    def build_server_cmd(self, python_exe: str) -> list:
        raise NotImplementedError(f'{type(self).__name__}.build_server_cmd()')

    def build_extra_body(self, stop_token_ids: Optional[list] = None) -> dict:
        raise NotImplementedError(f'{type(self).__name__}.build_extra_body()')

    def build_env(self) -> dict:
        """Default: inherit current environment."""
        return os.environ.copy()


# ══════════════════════════════════════════════════════════════════════════════
# GPT-OSS ENGINE CONFIG
# gpt-oss-120b (and fine-tuned variants) — Harmony TIR protocol
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class GptOssEngineConfig(EngineConfig):
    """
    gpt-oss-120b via Harmony TIR protocol (token-ID completions endpoint).

    All vLLM flags are explicit in build_server_cmd() — no hidden inheritance.
    """

    suite_id:          str   = 'gpt-oss'
    served_model_name: str   = 'gpt-oss'

    # ── gpt-oss specific ──────────────────────────────────────────────────
    dtype:             str   = 'auto'
    kv_cache_dtype:    str   = 'fp8_e4m3'
    context_tokens:    int   = 65536
    min_p:             float = 0.02
    top_p:             float = 0.95
    top_logprobs:      int   = 5
    harmony_encoding_name: str = 'HARMONY_GPT_OSS'

    # ── Memory tuning ─────────────────────────────────────────────────────
    flashinfer_attention:   bool = True
    max_num_batched_tokens: Optional[int] = None
    cudagraph_capture_sizes: Optional[list] = None

    def build_env(self) -> dict:
        env = os.environ.copy()
        if self.flashinfer_attention:
            env["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
        return env

    def build_server_cmd(self, python_exe: str) -> list:
        cmd = [
            python_exe, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed',                   '42',
            '--model',                  self.model_path,
            '--served-model-name',      self.served_model_name,
            '--tensor-parallel-size',   '1',
            '--max-num-seqs',           str(self.batch_size),
            '--gpu-memory-utilization', str(self.gpu_memory_utilization),
            '--host',                   '0.0.0.0',
            '--port',                   str(self.port),
            '--dtype',                  self.dtype,
            '--kv-cache-dtype',         self.kv_cache_dtype,
            '--max-model-len',          str(self.context_tokens),
            '--stream-interval',        '200',
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
        ]
        if self.cudagraph_capture_sizes:
            cmd += ['--compilation-config', json.dumps({
                "cudagraph_capture_sizes": sorted(self.cudagraph_capture_sizes)
            })]
        if self.max_num_batched_tokens is not None:
            cmd += ['--max-num-batched-tokens', str(self.max_num_batched_tokens)]
        if self.use_spec:
            cmd += self.spec_config.to_cli_flags()
        return cmd

    def build_extra_body(self, stop_token_ids=None) -> dict:
        body = {'top_p': self.top_p, 'return_token_ids': True}
        if stop_token_ids:
            body['stop_token_ids'] = stop_token_ids
        return body


# ══════════════════════════════════════════════════════════════════════════════
# QWEN 3.5 ENGINE CONFIG
# Qwen3.5-27B-FP8 / Qwen3.5-35B-A3B-FP8 — chat completions
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Qwen35EngineConfig(EngineConfig):
    """
    Qwen3.5 via /v1/completions + client-side tokenization with native tool calling.

    Requires: --trust-remote-code, --language-model-only
    All flags explicit in build_server_cmd().
    """

    suite_id:          str   = 'qwen35'
    served_model_name: str   = 'qwen35'

    # ── Qwen specific ─────────────────────────────────────────────────────
    dtype:             str   = 'auto'
    kv_cache_dtype:    str   = 'fp8_e4m3'
    context_tokens:    int   = 65536
    min_p:             float = 0.05
    top_p:             float = 0.95
    enable_thinking:   bool  = False

    # ── FIX: native tool calling (Qwen3.5 <parameter> protocol) ──────────
    use_native_tools:  bool  = True

    # ── FIX: sampling params matching working notebook ────────────────────
    presence_penalty:  float = 1.5
    top_logprobs:      int   = 5

    # ── Memory tuning ─────────────────────────────────────────────────────
    flashinfer_attention:   bool = True
    max_num_batched_tokens: Optional[int] = None

    def build_env(self) -> dict:
        env = os.environ.copy()
        if self.flashinfer_attention:
            env["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
        return env

    def build_server_cmd(self, python_exe: str) -> list:
        cmd = [
            python_exe, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed',                   '42',
            '--model',                  self.model_path,
            '--served-model-name',      self.served_model_name,
            '--tensor-parallel-size',   '1',
            '--max-num-seqs',           str(self.batch_size),
            '--gpu-memory-utilization', str(self.gpu_memory_utilization),
            '--host',                   '0.0.0.0',
            '--port',                   str(self.port),
            '--dtype',                  self.dtype,
            '--kv-cache-dtype',         self.kv_cache_dtype,
            '--max-model-len',          str(self.context_tokens),
            '--stream-interval',        '200',
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
            '--trust-remote-code',
            '--language-model-only',
        ]
        if self.max_num_batched_tokens is not None:
            cmd += ['--max-num-batched-tokens', str(self.max_num_batched_tokens)]
        if self.use_spec:
            cmd += self.spec_config.to_cli_flags()
        return cmd

    def build_extra_body(self, stop_token_ids=None) -> dict:
        body = {'top_p': self.top_p, 'return_token_ids': True}
        if stop_token_ids:
            body['stop_token_ids'] = stop_token_ids
        return body


# ══════════════════════════════════════════════════════════════════════════════
# DEEPSEEK-R1 QWEN ENGINE CONFIG
# DeepSeek-R1-Distill (Qwen2.5 / Qwen3 backbone)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class DeepSeekR1QwenEngineConfig(EngineConfig):

    suite_id:          str   = 'deepseek_r1_qwen'
    served_model_name: str   = 'deepseek-r1-qwen'

    dtype:             str   = 'auto'
    kv_cache_dtype:    str   = 'fp8_e4m3'
    context_tokens:    int   = 65536
    min_p:             float = 0.02          # ← WAS 0.01, match working v8
    top_p:             float = 0.95
    enable_thinking:   bool  = False # If set True => overthinking
    top_logprobs:      int   = 5             # ← ADD: needed for entropy voting

    flashinfer_attention:   bool = True
    max_num_batched_tokens: Optional[int] = None

    def build_env(self) -> dict:
        env = os.environ.copy()
        if self.flashinfer_attention:
            env["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
        return env

    def build_server_cmd(self, python_exe: str) -> list:
        cmd = [
            python_exe, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed',                   '42',
            '--model',                  self.model_path,
            '--served-model-name',      self.served_model_name,
            '--tensor-parallel-size',   '1',
            '--max-num-seqs',           str(self.batch_size),
            '--gpu-memory-utilization', str(self.gpu_memory_utilization),
            '--host',                   '0.0.0.0',
            '--port',                   str(self.port),
            '--dtype',                  self.dtype,
            '--kv-cache-dtype',         self.kv_cache_dtype,
            '--max-model-len',          str(self.context_tokens),
            '--stream-interval',        '200',
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
            '--trust-remote-code',
            # ← REMOVED: '--reasoning-parser', 'deepseek_r1'
            # Client-side tokenization + /completions = no parser needed
        ]
        if self.max_num_batched_tokens is not None:
            cmd += ['--max-num-batched-tokens', str(self.max_num_batched_tokens)]
        if self.use_spec:
            cmd += self.spec_config.to_cli_flags()
        return cmd

    def build_extra_body(self, stop_token_ids=None) -> dict:
        body = {
            'top_p': self.top_p,
            'min_p': self.min_p,             # ← ADD: was missing
            'return_token_ids': True,        # ← ADD: for consistency
        }
        if stop_token_ids:
            body['stop_token_ids'] = stop_token_ids
        return body

# ══════════════════════════════════════════════════════════════════════════════
# DEEPSEEK-R1 LLAMA ENGINE CONFIG
# DeepSeek-R1-Distill-Llama-8B / 70B — Llama backbone
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class DeepSeekR1LlamaEngineConfig(EngineConfig):
    """
    DeepSeek-R1-Distill-Llama-8B (Llama3.1) + Llama-70B (Llama3.3).

    Llama backbone: no --trust-remote-code, no --language-model-only.
    """

    suite_id:          str   = 'deepseek_r1_llama'
    served_model_name: str   = 'deepseek-r1-llama'

    dtype:             str   = 'auto'
    kv_cache_dtype:    str   = 'fp8_e4m3'
    context_tokens:    int   = 65536
    min_p:             float = 0.01
    top_p:             float = 0.95

    flashinfer_attention: bool = True

    def build_env(self) -> dict:
        env = os.environ.copy()
        if self.flashinfer_attention:
            env["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
        return env

    def build_server_cmd(self, python_exe: str) -> list:
        return [
            python_exe, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed',                   '42',
            '--model',                  self.model_path,
            '--served-model-name',      self.served_model_name,
            '--tensor-parallel-size',   '1',
            '--max-num-seqs',           str(self.batch_size),
            '--gpu-memory-utilization', str(self.gpu_memory_utilization),
            '--host',                   '0.0.0.0',
            '--port',                   str(self.port),
            '--dtype',                  self.dtype,
            '--kv-cache-dtype',         self.kv_cache_dtype,
            '--max-model-len',          str(self.context_tokens),
            '--stream-interval',        '200',
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
            '--reasoning-parser',       'deepseek_r1',
        ]

    def build_extra_body(self, stop_token_ids=None) -> dict:
        return {'min_p': self.min_p, 'top_p': self.top_p}


# ══════════════════════════════════════════════════════════════════════════════
# OPENREASONING-NEMOTRON ENGINE CONFIG
# nvidia/OpenReasoning-Nemotron-7B / 14B / 32B — Qwen2.5 backbone
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class OpenReasoningEngineConfig(EngineConfig):
    """
    nvidia/OpenReasoning-Nemotron (Qwen2.5 backbone, AIMO-2 winners).

    Same vLLM flags as DeepSeek-R1-Qwen: --trust-remote-code, --reasoning-parser deepseek_r1.
    """

    suite_id:          str   = 'openreasoning'
    served_model_name: str   = 'openreasoning'

    dtype:             str   = 'auto'
    kv_cache_dtype:    str   = 'fp8_e4m3'
    context_tokens:    int   = 65536
    min_p:             float = 0.01
    top_p:             float = 0.95
    enable_thinking:   bool  = True

    flashinfer_attention: bool = True

    def build_env(self) -> dict:
        env = os.environ.copy()
        if self.flashinfer_attention:
            env["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
        return env

    def build_server_cmd(self, python_exe: str) -> list:
        return [
            python_exe, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed',                   '42',
            '--model',                  self.model_path,
            '--served-model-name',      self.served_model_name,
            '--tensor-parallel-size',   '1',
            '--max-num-seqs',           str(self.batch_size),
            '--gpu-memory-utilization', str(self.gpu_memory_utilization),
            '--host',                   '0.0.0.0',
            '--port',                   str(self.port),
            '--dtype',                  self.dtype,
            '--kv-cache-dtype',         self.kv_cache_dtype,
            '--max-model-len',          str(self.context_tokens),
            '--stream-interval',        '200',
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
            '--trust-remote-code',
            '--reasoning-parser',       'deepseek_r1',
        ]

    def build_extra_body(self, stop_token_ids=None) -> dict:
        return {'min_p': self.min_p, 'top_p': self.top_p}


# ══════════════════════════════════════════════════════════════════════════════
# GEMMA3 ENGINE CONFIG
# google/gemma-3-4b-it / 12b-it / 27b-it — bfloat16, enforce-eager
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Gemma3EngineConfig(EngineConfig):
    """
    Google Gemma3 Instruct.

    Hardcoded: dtype=bfloat16, kv_cache_dtype=auto, --enforce-eager.
    No --trust-remote-code, no --reasoning-parser, no spec decoding.
    """

    suite_id:          str   = 'gemma3'
    served_model_name: str   = 'gemma3'

    context_tokens:    int   = 65536
    min_p:             float = 0.01
    top_p:             float = 0.95

    flashinfer_attention: bool = True

    def build_env(self) -> dict:
        env = os.environ.copy()
        if self.flashinfer_attention:
            env["VLLM_ATTENTION_BACKEND"] = "FLASHINFER"
        return env

    def build_server_cmd(self, python_exe: str) -> list:
        return [
            python_exe, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed',                   '42',
            '--model',                  self.model_path,
            '--served-model-name',      self.served_model_name,
            '--tensor-parallel-size',   '1',
            '--max-num-seqs',           str(self.batch_size),
            '--gpu-memory-utilization', str(self.gpu_memory_utilization),
            '--host',                   '0.0.0.0',
            '--port',                   str(self.port),
            '--dtype',                  'bfloat16',
            '--kv-cache-dtype',         'auto',
            '--max-model-len',          str(self.context_tokens),
            '--stream-interval',        '200',
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
            '--enforce-eager',
        ]

    def build_extra_body(self, stop_token_ids=None) -> dict:
        return {'min_p': self.min_p, 'top_p': self.top_p}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE ENGINE MANAGER
# Manages one vLLM server process and its OpenAI-compatible client.
# One instance per entry in ACTIVE_ENGINE_CONFIGS.
# ══════════════════════════════════════════════════════════════════════════════

class InferenceEngineManager:
    """
    Lifecycle manager for one vLLM server.

    Responsibilities:
      - Preload model weights into OS page cache before server launch
      - Launch vLLM as a subprocess and redirect output to a log file
      - Poll until the server is ready to accept requests
      - Expose a single OpenAI client for use by ModelSuiteProtocol instances
      - Gracefully shut down on solver teardown

    The client exposed here is the only network connection to the model.
    All agents on this engine share it — the OpenAI client is thread-safe.
    """

    def __init__(self, ec: EngineConfig, cfg) -> None:
        self.ec             = ec
        self.cfg            = cfg
        self.client         = None    # OpenAI — set in start()
        self.server_process = None    # subprocess.Popen
        self.log_file       = None    # open file handle for server stdout

    # ── Public lifecycle ──────────────────────────────────────────────────────

    def start(self) -> None:
        """Preload weights → start server → wait for readiness."""
        self._preload_weights()
        self.server_process = self._start_server()
        self._create_client()
        self._wait_for_ready()
        spec_note = (
            f' [spec:{self.ec.spec_config.method}]' if self.ec.use_spec else ''
        )
        print(f'[{self.ec.suite_id}:{self.ec.port}] Server ready{spec_note}\n')

    def stop(self) -> None:
        """Terminate server process and close log file."""
        if self.server_process is not None:
            self.server_process.terminate()
            try:
                self.server_process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                self.server_process.kill()
        if self.log_file is not None:
            self.log_file.close()

    # ── Properties ────────────────────────────────────────────────────────────

    @property
    def base_url(self) -> str:
        return f'http://0.0.0.0:{self.ec.port}/v1'

    @property
    def suite_id(self) -> str:
        return self.ec.suite_id

    @property
    def model_name(self) -> str:
        return self.ec.served_model_name

    # ── Internal ──────────────────────────────────────────────────────────────

    def _preload_weights(self) -> None:
        """
        Read model files into OS page cache before vLLM starts.
        Also preloads the draft model if spec_config requires one.
        Uses ThreadPoolExecutor for parallel file reads.
        """
        dirs_to_load = [self.ec.model_path]
        if self.ec.use_spec:
            draft_path = self.ec.spec_config.draft_model_path()
            if draft_path:
                dirs_to_load.append(draft_path)

        for model_dir in dirs_to_load:
            if not os.path.isdir(model_dir):
                print(f'  [WARN] Model dir not found, skipping preload: {model_dir}')
                continue

            files, total_bytes = [], 0
            for root, _, fnames in os.walk(model_dir):
                for fn in fnames:
                    fp = os.path.join(root, fn)
                    if os.path.isfile(fp):
                        files.append(fp)
                        total_bytes += os.path.getsize(fp)

            print(f'Preloading {model_dir} ({len(files)} files, {total_bytes/1e9:.2f} GB) …')

            def _read_file(path: str) -> None:
                with open(path, 'rb') as f:
                    while f.read(1 << 30):  # 1 GB chunks
                        pass

            with ThreadPoolExecutor(max_workers=self.cfg.workers) as ex:
                list(ex.map(_read_file, files))

            print(f'  Preload complete.')

    def _start_server(self) -> subprocess.Popen:
        """Launch vLLM server subprocess. stdout + stderr → log file."""
        cmd      = self.ec.build_server_cmd(sys.executable)
        log_path = f'vllm_server_{self.ec.port}.log'
        self.log_file = open(log_path, 'w')
        print(f'Starting vLLM [{self.ec.suite_id}] on port {self.ec.port} …')
        print(f'  Log: {log_path}')
        return subprocess.Popen(
            cmd,
            stdout            = self.log_file,
            stderr            = subprocess.STDOUT,
            start_new_session = True,
        )

    def _create_client(self) -> None:
        self.client = OpenAI(
            base_url = self.base_url,
            api_key  = 'sk-local',
            timeout  = self.cfg.session_timeout,
        )

    def _wait_for_ready(self) -> None:
        """Poll until the server responds or the process dies / timeout occurs."""
        print(f'Waiting for server [{self.ec.suite_id}:{self.ec.port}] …')
        start = time.time()

        for _ in range(self.cfg.server_timeout):
            if self.server_process.poll() is not None:
                self.log_file.flush()
                log_path = f'vllm_server_{self.ec.port}.log'
                try:
                    with open(log_path) as lf:
                        tail = lf.read()[-2000:]
                except Exception:
                    tail = '(log unreadable)'
                raise RuntimeError(
                    f'[{self.ec.suite_id}] Server process died '
                    f'(exit {self.server_process.returncode}).\n'
                    f'Last 2000 chars of log:\n{tail}'
                )
            try:
                self.client.models.list()
                elapsed = time.time() - start
                print(f'  Ready in {elapsed:.1f}s')
                return
            except Exception:
                time.sleep(1)

        raise RuntimeError(
            f'[{self.ec.suite_id}] Server did not become ready '
            f'within {self.cfg.server_timeout}s'
        )


## Block E - Model Suite Layer

This block defines the abstraction boundary between the generic TIR loop and
every model-specific implementation detail. A `SolverAgent` or `MetaAgent`
holds one `ModelSuiteProtocol` instance and delegates all conversation work to
it: message construction, API call submission, response streaming, code
detection, answer scanning, and token counting. The TIR loop in both agent
classes is identical regardless of which suite is in use.

Adding support for a new model requires implementing eight abstract methods in
one new class and registering one line in the `make_model_suite()` factory.
No changes to `SolverAgent`, `Controller`, `Blackboard`, `CFG`, or any other
component are required.

This boundary is the primary mechanism by which the notebook lowers the cost of
model experimentation. The technical differences between model families -
tokenization conventions, chat template structure, stop token semantics,
reasoning block delimiters, endpoint types - are fully contained within the
suite class. A competitor experimenting with a new model family works only in
this block and in Block D; everything else remains unchanged.

#### The eight abstract methods

| Method | Responsibility |
|---|---|
| `build_initial_messages(pb_text, system_prompt, spirit)` | Constructs the model-native initial conversation container. Returns a Harmony `Conversation` for `gpt-oss`, or `list[dict]` for all chat-completions suites. The `spirit` argument (one sentence) is prepended to the system prompt when provided. |
| `build_tool_response_message(code_output)` | Wraps sandbox stdout as the message that delivers execution results back to the model. For `gpt-oss`: a Harmony TOOL message. For chat-completions suites: a user dict with a `Code output:` prefix. |
| `create_stream(messages, seed, max_tokens, temperature)` | Submits the current conversation to vLLM and returns a streaming response object. |
| `parse_stream(stream, stop_event, deadline, mode)` | Consumes the streaming response with inline answer detection and mode-aware stream termination. See ARCH-001 below. |
| `is_final_turn(full_text, new_messages)` | Returns `True` if this turn carries a final answer and no further tool calls are expected. Detection criterion varies by suite. |
| `scan_for_answer(text)` | Extracts an integer answer from text. Returns an integer in $[0, 99999]$ or `None`. |
| `count_prompt_tokens(messages)` | Returns the token count of the current conversation, used to derive `max_new_tokens` before each API call. |
| `handle_final_non_parsable(last_text, messages)` | Called when `is_final_turn` is `True` but `scan_for_answer` returns `None`. Attempts a broader extraction pass. Returns an integer or `None`. |

See: [Python `abc` module documentation](https://docs.python.org/3/library/abc.html).

#### ARCH-001 - Mode-aware streaming

`parse_stream` is the operationally most sensitive method in the protocol.
Its contract is:

```
parse_stream(stream, stop_event, deadline, mode)
    -> (full_text, code, new_messages, mid_stream_answer)
```

On every chunk, two checks execute unconditionally before any other processing:
- `stop_event.is_set()` - a stop condition fired on another agent; abort this stream.
- `time.time() > deadline` - the per-problem budget is exhausted; abort.

When a `}` character appears in the newly decoded chunk, a sliding window of
`cfg.search_tokens` recent chunks is scanned for a `\boxed{N}` pattern.
On a match, behaviour splits by mode:

**SUBMISSION** - the stream closes immediately. No further tokens are generated
or billed. The answer is returned as `mid_stream_answer` and posted to the
`Blackboard` as an incremental `AttemptResult` before the agent's TIR turn
completes. The `Controller` can therefore evaluate stop conditions and cancel
remaining agents before any of them finishes generating.

**BENCHMARK** - the answer is recorded in `mid_stream_answer` but the stream
continues draining to natural completion. The full reasoning trace is preserved
in `full_text` and written to `attempts/{pb_id}__{agent}__{idx}.json`.

A second call to `scan_for_answer(full_text)` runs after stream close in
`SolverAgent`, covering answers that span chunk boundaries or where the `}`
trigger fell on a different chunk from the opening `\boxed{`.

Two helper functions shared across all suites are defined alongside the
protocol. `_append_messages()` extends a conversation in place for both Harmony
`Conversation` objects and plain `list[dict]`. `_serialize()` converts either
format to a uniform `list[dict]` for JSON output in BENCHMARK mode.

#### Two active suites

| Suite | Endpoint | Answer format | Token counting | Status |
|---|---|---|---|---|
| `GptOssModelSuite` | `/v1/completions` (Harmony token-ID) | `\boxed{N}` | Harmony encoding render | Stable |
| `Qwen35ModelSuite` | `/v1/chat/completions` | `\boxed{N}` | AutoTokenizer `apply_chat_template` | Stable |

The remaining four suites are experimental. Their descriptions are in E.2.


### Model Suites

Six model suites are implemented. Two carry the `# STABLE` label; four carry
`# EXPERIMENTAL`. Stability refers to the expected reliability of the suite
under competition conditions on the Kaggle hardware allocation. Experimental
suites are functional but have not been validated across the full range of
vLLM versions and hardware configurations.

#### GptOssModelSuite - Stable

The `gpt-oss` model family does not use the standard chat completions format.
Conversations are rendered as flat token-ID sequences by the Harmony encoding
layer and submitted to the `/completions` endpoint. The response token IDs are
parsed back into typed messages carrying routing metadata: `recipient='python'`
signals that the model requests code execution; `channel='final'` marks the
answer turn.

`is_final_turn()` checks `channel == 'final'` on the last parsed message -
a deterministic signal from the model's own output structure, not a heuristic.
`count_prompt_tokens()` calls the Harmony encoding's `render()` method and
counts the resulting token-ID sequence. The Harmony encoding and stop token IDs
are loaded once in `__init__` and reused across all agent calls.

`ReasoningEffort.HIGH` is set in the system content, directing the model toward
deeper multi-step reasoning.

See: the `openai_harmony` package README bundled with the wheel archive.

#### Qwen35ModelSuite - Stable

`Qwen35ModelSuite` uses the standard `/chat/completions` endpoint with plain
`list[dict]` message format. Code blocks are detected by the regex
`` ` ``python\n...\n`` ` `` ``. The `AutoTokenizer` is loaded at init via
`from_pretrained(ec.model_path)` and used in `count_prompt_tokens()` by
calling `apply_chat_template(messages, tokenize=True)`.

`is_final_turn()` returns `True` when the text contains no ` ```python ` fence
and `scan_for_answer()` returns a non-None value. Tool responses are injected
as user-role messages with a `Code output:` prefix, following standard
multi-turn chat conventions.

`Qwen35ModelSuite` is the base class for three of the four experimental suites.
Its `parse_stream`, `build_initial_messages`, `build_tool_response_message`,
and `create_stream` implementations are inherited unchanged by all subclasses.

See: [Qwen3.5 model card on HuggingFace](https://huggingface.co/Qwen).

#### DeepSeekR1QwenModelSuite - Experimental

Subclass of `Qwen35ModelSuite`. Covers DeepSeek-R1-Distill-Qwen-1.5B,
7B, 14B, 32B, and DeepSeek-R1-0528-Qwen3-8B. Inherits all conversation
and streaming logic unchanged. Three overrides are added.

`_extract_post_think(text)` returns the substring after the last `</think>` tag.
DeepSeek-R1 models regularly write a provisional answer inside the `<think>`
block and then revise it after closing the tag. `scan_for_answer()` scans the
post-think region first and falls back to the full text only if no answer is
found there, avoiding the acceptance of a revised-away provisional answer.

`is_final_turn()` adds `has_think_end AND has_answer` as a valid final-turn
condition alongside the Qwen35 baseline of `no_code AND has_answer`.

No `__init__` override is needed: the Qwen2.5/3 AutoTokenizer is correct for
all Qwen-backbone DeepSeek-R1 variants.

See: [DeepSeek-R1 technical report](https://arxiv.org/abs/2501.12948).

#### DeepSeekR1LlamaModelSuite - Experimental

Same behavioural overrides as `DeepSeekR1QwenModelSuite`. Covers
DeepSeek-R1-Distill-Llama-8B (Llama3.1) and Llama-70B (Llama3.3).

`__init__` calls `ModelSuiteProtocol.__init__` directly, bypassing
`Qwen35ModelSuite.__init__`, and then loads the Llama `AutoTokenizer`. This
is the only case in the codebase where a subclass bypasses its parent's
`__init__`; it is necessary because `Qwen35ModelSuite.__init__` would
unconditionally load the Qwen tokenizer, which would fail against a Llama
model path.

#### OpenReasoningModelSuite - Experimental

Functionally identical to `DeepSeekR1QwenModelSuite`. Covers
nvidia/OpenReasoning-Nemotron-1.5B, 7B, 14B, and 32B. All four sizes use
Qwen2.5 as the backbone - not Llama, despite the NVIDIA branding. Trained on
5M reasoning traces distilled from DeepSeek-R1-0528. These models formed the
basis of the AIMO-2 winning approach.

Maintained as a separate class rather than an alias for two reasons: independent
`suite_id` labelling in JSON output enables per-model accuracy analysis across
suites in BENCHMARK mode; and prompt variants for OpenReasoning can be tuned
independently of the DeepSeek-R1 configs without risk of cross-contamination.

See: [OpenReasoning-Nemotron on HuggingFace](https://huggingface.co/nvidia/OpenReasoning-Nemotron-32B).

#### Gemma3ModelSuite - Experimental

Direct subclass of `ModelSuiteProtocol` (not of `Qwen35ModelSuite`). Covers
google/gemma-3-4b-it, 12b-it, and 27b-it.

The Gemma3 instruct chat template does not reliably handle the `system` role
across vLLM versions. `build_initial_messages()` merges the system prompt,
spirit prefix, problem text, and preference prompt into a single user-role
message. The Gemma3 `AutoTokenizer` is loaded for `count_prompt_tokens()`.

Gemma3-27b-it emits `<think>...</think>` blocks; `scan_for_answer()` applies
the same post-think region priority as the DeepSeek suites.

`parse_stream` is identical to `Qwen35ModelSuite` - standard chat completions
delta streaming with the same ARCH-001 mode-aware loop.

See: [Gemma3 model documentation](https://ai.google.dev/gemma/docs).

#### make_model_suite() - Factory

```python
make_model_suite(engine, ec, cfg) -> ModelSuiteProtocol
```

Dispatches on `EngineConfig` type via `isinstance` checks and returns the
corresponding suite. Subclasses are checked before their parent classes;
`DeepSeekR1QwenEngineConfig` is checked before `Qwen35EngineConfig` because
the former would pass an `isinstance` check against the latter's parent.

An unregistered `EngineConfig` type raises `ValueError`, ensuring that a
missing factory registration fails loudly rather than silently falling through
to an incorrect suite.

To register a new suite: implement a concrete `EngineConfig` subclass in Block D,
implement a concrete `ModelSuiteProtocol` subclass in this block, and add one
`isinstance` branch at the correct position in `make_model_suite()`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL SUITE PROTOCOL — ABSTRACT BASE
# One instance per engine. Shared across all agents on that engine.
# Stateless: all conversation state lives in the per-call messages container.
# Thread-safe: no mutable instance state accessed during a call.
# ══════════════════════════════════════════════════════════════════════════════

class ModelSuiteProtocol(ABC):
    """
    Conversation protocol for one model suite.

    Concrete subclasses implement all model-specific logic:
      - Message formatting (system/user/tool structure)
      - Streaming API call parameters
      - Response parsing and code detection
      - Answer scanning regex patterns
      - Token counting method

    The TIR loop in SolverAgent and MetaAgent is identical for all suites.
    """

    def __init__(self, engine, ec: EngineConfig, cfg) -> None:
        self.engine = engine   # InferenceEngineManager
        self.ec     = ec       # EngineConfig subclass
        self.cfg    = cfg      # CFG instance (for search_tokens, buffer_tokens, etc.)

    # ── Build messages ────────────────────────────────────────────────────────

    @abstractmethod
    def build_initial_messages(
        self,
        pb_text:       str,
        system_prompt: str,
        spirit:        Optional[str] = None,
    ):
        """
        Build the model-native initial message container.
        spirit (one sentence) is prepended to system_prompt when provided.
        Returns a Harmony Conversation or a list[dict] depending on suite.
        """
        ...

    @abstractmethod
    def build_tool_response_message(self, code_output: str):
        """
        Build the message that delivers sandbox stdout back to the model.
        gpt-oss:  Harmony TOOL message with recipient='assistant'
        Qwen/DS:  plain user dict with 'Code output:' prefix
        """
        ...

    # ── API call ──────────────────────────────────────────────────────────────

    @abstractmethod
    def create_stream(
        self,
        messages,
        seed:        int,
        max_tokens:  int,
        temperature: float,
    ):
        """Submit conversation to vLLM and return a streaming response object."""
        ...

    # ── Response parsing — ARCH-001 ───────────────────────────────────────────

    @abstractmethod
    def parse_stream(
        self,
        stream,
        stop_event: threading.Event,
        deadline:   float,
        mode:       str,
    ) -> tuple:
        """
        Consume the streaming response with inline answer detection.

        Args:
            stream:     streaming API response object
            stop_event: shared threading.Event — break if set
            deadline:   wall-clock deadline — break if exceeded
            mode:       'SUBMISSION' | 'BENCHMARK'

        Returns:
            full_text         — complete assistant output as plain string
            code              — extracted Python code block (str) or None
            new_messages      — model-native messages to append to conversation
            mid_stream_answer — integer answer detected during streaming, or None

        Behaviour:
            Both modes:   break on stop_event or deadline (safety)
            SUBMISSION:   also break on first answer found (save tokens)
            BENCHMARK:    never break for answer (preserve full reasoning trace)
            Always:       stream.close() in finally block
        """
        ...

    # ── Turn logic ────────────────────────────────────────────────────────────

    @abstractmethod
    def is_final_turn(self, full_text: str, new_messages: list) -> bool:
        """
        True if this is the answer turn — no further tool calls expected.
        gpt-oss:     last message channel == 'final'
        Qwen/DS:     no ```python``` block AND answer marker present
        """
        ...

    # ── BENCHMARK helpers (non-integer answer support) ────────────────────────

    def _extract_any_boxed(self, text: str) -> Optional[str]:
        """Extract raw content from last \\boxed{...}, handling nested braces."""
        idx = text.rfind('\\boxed')
        if idx == -1:
            return None
        brace_start = text.find('{', idx)
        if brace_start == -1:
            return None
        depth = 0
        for i in range(brace_start, len(text)):
            if text[i] == '{': depth += 1
            elif text[i] == '}':
                depth -= 1
                if depth == 0:
                    content = text[brace_start+1:i].strip()
                    return content if content else None
        return None

    def _benchmark_boxed_fallback(self, text: str):
        """BENCHMARK only: accept any boxed content as answer (int or str).
        SUBMISSION mode: returns None (zero regression).
        Returns: int | str | None"""
        if MODE != 'BENCHMARK':
            return None
        raw = self._extract_any_boxed(text)
        if raw is None:
            return None
        # Integer content: parse normally
        try:
            v = int(raw.replace(',', '').strip())
            if -999999999999 <= v <= 999999999999:
                return v
        except (ValueError, OverflowError):
            pass
        # Non-integer: return normalized raw boxed content as string answer
        # This preserves the symbolic answer for LLM judge evaluation
        # and allows identical symbolic answers to aggregate in voting
        return raw.strip()

    # ── Abstract interface ────────────────────────────────────────────────

    @abstractmethod
    def scan_for_answer(self, text: str) -> Optional[int]:
        """Extract integer answer from text. Returns int in [0, 99999] or None."""
        ...

    @abstractmethod
    def count_prompt_tokens(self, messages) -> int:
        """Return token count of the current conversation."""
        ...

    @abstractmethod
    def handle_final_non_parsable(
        self,
        last_text: str,
        messages,
    ) -> Optional[int]:
        """
        Called when is_final_turn=True but scan_for_answer=None.
        Placeholder: tries broader regex, optionally an LLM extraction call.
        Returns extracted integer or None.
        """
        ...


# ══════════════════════════════════════════════════════════════════════════════
# SHARED HELPERS
# Used by SolverAgent, MetaAgent, and _serialize (BENCHMARK mode).
# ══════════════════════════════════════════════════════════════════════════════

def _append_messages(messages, new_msgs: list) -> None:
    """
    Append new_msgs to the conversation container in-place.
    Works for both list[dict] (Qwen) and Harmony Conversation objects.
    """
    if isinstance(messages, list):
        messages.extend(new_msgs)
    else:
        messages.messages.extend(new_msgs)


def _serialize(messages, suite: ModelSuiteProtocol) -> list:
    """
    Convert model-native conversation to plain list[dict] for JSON storage.
    Always returns [{role, content, channel}, ...].
    Called in BENCHMARK mode only — not on the hot path.

    Harmony conversations contain heterogeneous content types:
      - TextContent       (assistant / user / tool turns) → .text attribute
      - SystemContent     (system turn)                   → no .text; use str()
      - Other types       (future-proofing)               → fallback to str()
    """
    raw = messages if isinstance(messages, list) else messages.messages
    out = []
    for m in raw:
        if isinstance(m, dict):
            out.append({
                'role':    m.get('role', 'unknown'),
                'content': m.get('content', ''),
                'channel': m.get('channel', 'none'),
            })
        else:
            author  = getattr(m, 'author', None)
            role    = getattr(author, 'role', None)
            role_s  = role.value if hasattr(role, 'value') else str(role)
            content = getattr(m, 'content', None)
            channel = getattr(m, 'channel', 'none') or 'none'

            text = ''
            if content:
                item = content[0]
                if hasattr(item, 'text'):
                    text = item.text or ''
                else:
                    try:
                        text = str(item)
                    except Exception:
                        text = '[non-serialisable content]'

            out.append({'role': role_s, 'content': text, 'channel': channel})
    return out

In [ ]:
# STABLE
# ══════════════════════════════════════════════════════════════════════════════
# GPT-OSS MODEL SUITE — Harmony TIR Protocol
# Token-ID completions endpoint. openai_harmony encoding handles all
# prompt rendering and response parsing.
# ══════════════════════════════════════════════════════════════════════════════

class GptOssModelSuite(ModelSuiteProtocol):
    """
    Harmony TIR protocol for gpt-oss-120b (and fine-tuned variants).

    The Harmony encoding renders the full conversation to a flat sequence of
    token IDs, which is submitted to the /completions endpoint (not /chat).
    The response token IDs are parsed back into typed messages that carry
    routing metadata: recipient='python' triggers code execution,
    channel='final' marks the answer turn.
    """

    def __init__(self, engine, ec: GptOssEngineConfig, cfg) -> None:
        super().__init__(engine, ec, cfg)

        # Late imports — only available after Block A wheel installation
        from openai_harmony import (
            HarmonyEncodingName, load_harmony_encoding,
            SystemContent, ReasoningEffort, ToolNamespaceConfig,
            Author, Message, Role, TextContent, Conversation,
        )
        self._H = {
            'SystemContent':       SystemContent,
            'ReasoningEffort':     ReasoningEffort,
            'ToolNamespaceConfig': ToolNamespaceConfig,
            'Author':              Author,
            'Message':             Message,
            'Role':                Role,
            'TextContent':         TextContent,
            'Conversation':        Conversation,
        }
        self.encoding       = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _tool_cfg(self):
        H = self._H
        return H['ToolNamespaceConfig'](
            name        = 'python',
            description = self.ec.tool_prompt,
            tools       = [],
        )

    # ── Build messages ────────────────────────────────────────────────────────

    def build_initial_messages(self, pb_text: str, system_prompt: str, spirit=None):
        H   = self._H
        sys_text = f'{spirit}\n{system_prompt}' if spirit else system_prompt
        sc  = (
            H['SystemContent'].new()
            .with_model_identity(sys_text)
            .with_reasoning_effort(H['ReasoningEffort'].HIGH)
            .with_tools(self._tool_cfg())
        )
        sys_msg = H['Message'].from_role_and_content(H['Role'].SYSTEM, sc)
        usr_msg = H['Message'].from_role_and_content(
            H['Role'].USER,
            f'{pb_text}\n\n{self.ec.preference_prompt}',
        )
        return H['Conversation'].from_messages([sys_msg, usr_msg])

    def build_tool_response_message(self, code_output: str):
        H = self._H
        return (
            H['Message'](
                author  = H['Author'](role=H['Role'].TOOL, name='python'),
                content = [H['TextContent'](text=code_output)],
            ).with_recipient('assistant')
        )

    # ── API call ──────────────────────────────────────────────────────────────

    def create_stream(self, conversation, seed: int, max_tokens: int, temperature: float):
        prompt_ids = self.encoding.render_conversation_for_completion(
            conversation, self._H['Role'].ASSISTANT
        )
        return self.engine.client.completions.create(
            model       = self.ec.served_model_name,
            prompt      = prompt_ids,
            temperature = temperature,
            logprobs    = self.ec.top_logprobs,
            max_tokens  = max_tokens,
            seed        = seed,
            stream      = True,
            extra_body  = self.ec.build_extra_body(self.stop_token_ids),
        )

    # ── ARCH-001: MODE-aware streaming ────────────────────────────────────────

    def parse_stream(self, stream, stop_event, deadline, mode) -> tuple:
        """
        Drain the Harmony completions stream with inline answer detection.

        Token IDs are accumulated for Harmony message parsing.
        Text chunks are accumulated for sliding-window answer scanning.

        SUBMISSION: stream breaks immediately on first \\boxed{} found.
        BENCHMARK:  stream runs to completion — full reasoning trace preserved.
        """
        token_buf:   list = []
        text_chunks: list = []
        mid_answer:  Optional[int] = None

        try:
            for chunk in stream:
                # Safety breaks — both modes
                if stop_event.is_set() or time.time() > deadline:
                    break

                choice = chunk.choices[0]
                toks   = getattr(choice, 'token_ids', None) or []
                txt    = getattr(choice, 'text', '') or ''

                if toks:
                    token_buf.extend(toks)
                if txt:
                    text_chunks.append(txt)

                # Scan for answer on every '}' — \boxed{} always ends with '}'
                if '}' in txt and mid_answer is None:
                    window    = ''.join(text_chunks[-self.cfg.search_tokens:])
                    candidate = self.scan_for_answer(window)
                    if candidate is not None:
                        mid_answer = candidate
                        if mode == 'SUBMISSION':
                            break   # save remaining generation tokens

        finally:
            stream.close()

        full_text    = ''.join(text_chunks)
        new_messages = self.encoding.parse_messages_from_completion_tokens(
            token_buf, self._H['Role'].ASSISTANT
        )

        # Detect tool call: last message has recipient == 'python'
        # content[0] is normally TextContent but can be SystemContent on edge
        # cases (empty token buffer, early stream break). Guard with hasattr.
        code: Optional[str] = None
        if new_messages:
            last = new_messages[-1]
            if getattr(last, 'recipient', None) == 'python':
                content = getattr(last, 'content', None)
                if content:
                    item = content[0]
                    code = item.text if hasattr(item, 'text') else None

        return full_text, code, new_messages, mid_answer

    # ── Turn logic ────────────────────────────────────────────────────────────

    def is_final_turn(self, full_text: str, new_messages: list) -> bool:
        if not new_messages:
            return True
        return getattr(new_messages[-1], 'channel', None) == 'final'

    def scan_for_answer(self, text: str) -> Optional[int]:
        for pat in [
            r'\\boxed\s*\{\s*([0-9,]+)\s*\}',
            r'final\s+answer\s+is\s*([0-9,]+)',
        ]:
            for m in re.findall(pat, text, re.IGNORECASE):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(text)

    def count_prompt_tokens(self, conversation) -> int:
        return len(
            self.encoding.render_conversation_for_completion(
                conversation, self._H['Role'].ASSISTANT
            )
        )

    def handle_final_non_parsable(self, last_text: str, messages) -> Optional[int]:
        # Placeholder — broader regex attempts before giving up
        for pat in [
            r'(?:answer|result)\s*[=:]\s*([0-9,]+)',
            r'\b([0-9]{1,5})\b(?=\s*$)',
        ]:
            for m in re.findall(pat, last_text.strip()[-500:], re.IGNORECASE):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(last_text)

In [ ]:
# STABLE
# ══════════════════════════════════════════════════════════════════════════════
# QWEN 3.5 MODEL SUITE — /v1/completions + Client-Side Tokenization
# ══════════════════════════════════════════════════════════════════════════════

class Qwen35ModelSuite(ModelSuiteProtocol):
    """
    Qwen3.5-27B-FP8 / Qwen3.5-35B-A3B-FP8 via /v1/completions.

    Architecture:
      - AutoTokenizer loaded eagerly at init
      - Client-side tokenization via tokenizer.apply_chat_template()
      - Raw token IDs submitted to /v1/completions (NOT /chat/completions)
      - Server runs without --reasoning-parser (no hang risk)
      - Qwen3.5: native tool calling with <parameter> tags
      - Subclasses (DeepSeek, OpenReasoning): markdown code fences, no tools

    Subclass contract:
      - DeepSeekR1Qwen, OpenReasoning: inherit __init__ → use_native_tools=False
      - DeepSeekR1Llama: skips this __init__, loads own Llama tokenizer
      - All inherit create_stream, parse_stream, _tokenize without changes
    """

    def __init__(self, engine, ec, cfg) -> None:
        super().__init__(engine, ec, cfg)

        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            ec.model_path,
            trust_remote_code=True,
        )

        # ── FIX: detect Qwen3.5 native tool calling ──────────────────────
        self._use_native_tools = getattr(ec, 'use_native_tools', False)

        if self._use_native_tools:
            # Qwen3.5 outputs code in <parameter=code>...</parameter> tags
            self._code_re = re.compile(
                r'<parameter=[^>]+>(.*?)</parameter>', re.DOTALL
            )
            # Tool definition list passed to apply_chat_template
            # Uses ec.tool_prompt if set, otherwise a sensible default
            _tool_desc = ec.tool_prompt if ec.tool_prompt else (
                'Use this tool to execute Python code for:\n'
                '- Complex calculations that would be error-prone by hand\n'
                '- Numerical verification of analytical results\n'
                '- Generating examples or testing conjectures\n'
                '- Brute-force verification for small cases\n\n'
                'The environment is a stateful Jupyter notebook. '
                'Code persists between executions.\n'
                'Always use print() to display results.'
            )
            self._tools = [
                {
                    "type": "function",
                    "function": {
                        "name": "python",
                        "description": _tool_desc,
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "The Python code to execute."
                                }
                            },
                            "required": ["code"]
                        },
                    },
                },
            ]
        else:
            # Subclass path: DeepSeek / OpenReasoning use markdown code fences
            self._code_re = re.compile(r'```python\n(.*?)\n```', re.DOTALL)
            self._tools = None

    # ── Client-side tokenization ──────────────────────────────────────────────

    def _tokenize(self, messages: list) -> list:
        """
        Tokenize conversation via AutoTokenizer.apply_chat_template.

        Returns list[int] — raw token IDs sent to /v1/completions.

        FIX: passes tools= when use_native_tools is True.
        This tells the tokenizer to format tool definitions into the prompt,
        which is how the working notebook operates.
        """
        kwargs = dict(
            tokenize=True,
            add_generation_prompt=True,
        )
        if hasattr(self.ec, 'enable_thinking'):
            kwargs['enable_thinking'] = self.ec.enable_thinking
        # FIX: pass tool definitions so Qwen3.5 knows about the python tool
        if self._tools:
            kwargs['tools'] = self._tools
        try:
            return self.tokenizer.apply_chat_template(messages, **kwargs)
        except TypeError:
            # Tokenizer doesn't support enable_thinking or tools — retry without
            kwargs.pop('enable_thinking', None)
            kwargs.pop('tools', None)
            return self.tokenizer.apply_chat_template(messages, **kwargs)

    # ── Build messages ────────────────────────────────────────────────────────

    def build_initial_messages(self, pb_text: str, system_prompt: str, spirit=None):
        sys_text = f'{spirit}\n{system_prompt}' if spirit else system_prompt
        return [
            {'role': 'system', 'content': sys_text},
            {'role': 'user',   'content': f'{pb_text}\n\n{self.ec.preference_prompt}'},
        ]

    def build_tool_response_message(self, code_output: str) -> dict:
        """
        Build the message that delivers sandbox stdout back to the model.

        FIX: Qwen3.5 with native tools uses role='tool' (matching working notebook).
        Nudge messages (no preceding tool call) use role='user' to avoid
        apply_chat_template errors when there's no tool_call in the assistant turn.
        Subclasses (DeepSeek, OpenReasoning) always use role='user'.
        """
        if self._use_native_tools:
            # Nudge messages from SolverAgent start with 'No Python code'
            # They MUST use role='user' because there was no tool_call in the
            # preceding assistant message — role='tool' would crash the template.
            is_nudge = code_output.startswith('No Python code')
            if not is_nudge:
                return {'role': 'tool', 'content': code_output}

        # Fallback: subclasses + nudge messages
        return {
            'role':    'user',
            'content': (
                f'Code output:\n{code_output}\n\n'
                r'Continue solving. Place your final answer in \boxed{}.'
            ),
        }

    # ── API call ──────────────────────────────────────────────────────────────

    def create_stream(self, messages: list, seed: int, max_tokens: int, temperature: float):
        """
        Tokenize client-side, submit token IDs to /v1/completions.

        FIX: adds presence_penalty and logprobs from EngineConfig,
        matching the working notebook's completions.create() call.
        Safe for subclasses: getattr returns 0.0 / None if field absent.
        """
        prompt_ids = self._tokenize(messages)

        # FIX: presence_penalty + logprobs from config (0.0/None for subclasses)
        presence_penalty = getattr(self.ec, 'presence_penalty', 0.0)
        logprobs_val     = getattr(self.ec, 'top_logprobs', None)

        return self.engine.client.completions.create(
            model            = self.ec.served_model_name,
            prompt           = prompt_ids,
            temperature      = temperature,
            max_tokens       = max_tokens,
            seed             = seed,
            stream           = True,
            presence_penalty = presence_penalty,
            logprobs         = logprobs_val,
            extra_body       = self.ec.build_extra_body(),
        )

    # ── ARCH-001: MODE-aware streaming ────────────────────────────────────────

    def parse_stream(self, stream, stop_event, deadline, mode) -> tuple:
        """
        Drain the /v1/completions stream with inline answer detection.

        FIX: code extraction uses findall (all matches joined) instead of
        search (first match only), matching the working notebook's
        extract_code_text() which joins all regex matches.
        """
        chunks:     list = []
        mid_answer: Optional[int] = None

        try:
            for chunk in stream:
                if stop_event.is_set() or time.time() > deadline:
                    break

                txt = chunk.choices[0].text or ''
                if txt:
                    chunks.append(txt)

                if '}' in txt and mid_answer is None:
                    window    = ''.join(chunks[-self.cfg.search_tokens:])
                    candidate = self.scan_for_answer(window)
                    if candidate is not None:
                        mid_answer = candidate
                        if mode == 'SUBMISSION':
                            break

        finally:
            stream.close()

        full_text = ''.join(chunks)

        # FIX: use findall to extract ALL code blocks and join them
        # Working notebook does: "\n".join(match.strip() for match in matches)
        matches = self._code_re.findall(full_text)
        code    = '\n'.join(m.strip() for m in matches) if matches else None

        return full_text, code, [{'role': 'assistant', 'content': full_text}], mid_answer

    # ── Turn logic ────────────────────────────────────────────────────────────

    def is_final_turn(self, full_text: str, new_messages: list) -> bool:
        """
        FIX: uses self._code_re to detect code instead of hardcoded ```python```.
        Qwen3.5 outputs <parameter> tags, not markdown fences.
        """
        has_code   = bool(self._code_re.search(full_text))
        has_answer = self.scan_for_answer(full_text) is not None
        return (not has_code) and has_answer

    def scan_for_answer(self, text: str) -> Optional[int]:
        for m in re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', text):
            try:
                v = int(m.replace(',', ''))
                if 0 <= v <= 99999:
                    return v
            except ValueError:
                pass
        return self._benchmark_boxed_fallback(text)

    def count_prompt_tokens(self, messages: list) -> int:
        """
        Token count via _tokenize() — same path as create_stream().
        One tokenization path, no count mismatch.
        """
        try:
            return len(self._tokenize(messages))
        except Exception:
            return sum(len(m.get('content', '')) for m in messages) // 3

    def handle_final_non_parsable(self, last_text: str, messages) -> Optional[int]:
        for pat in [
            r'(?:answer|result)\s*[=:]\s*([0-9,]+)',
            r'\b([0-9]{1,5})\b(?=\s*$)',
        ]:
            for m in re.findall(pat, last_text.strip()[-500:], re.IGNORECASE):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(last_text)







In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DEEPSEEK-R1 QWEN MODEL SUITE — subclass of Qwen35ModelSuite
# Covers: DeepSeek-R1-Distill-Qwen-1.5B/7B/14B/32B + R1-0528-Qwen3-8B
# ══════════════════════════════════════════════════════════════════════════════

class DeepSeekR1QwenModelSuite(Qwen35ModelSuite):
    """
    DeepSeek-R1-Distill (Qwen backbone) model suite.

    Inherits EVERYTHING from Qwen35ModelSuite — same tokenizer (Qwen2.5/3),
    same chat completions endpoint, same parse_stream ARCH-001 loop,
    same build_initial_messages, same build_tool_response_message.

    Three overrides vs Qwen35ModelSuite:

    1. _extract_post_think(): new helper — returns text after last </think> tag.

    2. scan_for_answer(): scans post-think region FIRST, then falls back to
       full text. Rationale: DeepSeek-R1 often writes a provisional answer
       inside <think>...</think> then corrects it after </think>. Scanning
       only the post-think region avoids accepting the provisional answer.

    3. is_final_turn(): treats </think> + answer as a final turn signal,
       in addition to the Qwen35 baseline condition (no code + answer).

    No __init__ override — Qwen AutoTokenizer works for all Qwen2.5/3 variants.
    """

    def _extract_post_think(self, text: str) -> str:
        """Return text after last </think> tag, or '' if no tag found."""
        idx = text.rfind('</think>')
        return text[idx + len('</think>'):] if idx != -1 else ''

    def scan_for_answer(self, text: str) -> Optional[int]:
        """Scan post-think region first, fallback to full text."""
        for region in [self._extract_post_think(text), text]:
            for m in re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', region):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(text)

    def is_final_turn(self, full_text: str, new_messages: list) -> bool:
        has_code      = bool(re.search(r'```python', full_text))
        has_answer    = self.scan_for_answer(full_text) is not None
        has_think_end = '</think>' in full_text
        # Final if: thinking complete + answer found, OR no code + answer found
        return (has_think_end and has_answer) or ((not has_code) and has_answer)


In [ ]:
# EXPERIMENTAL
# ══════════════════════════════════════════════════════════════════════════════
# DEEPSEEK-R1 LLAMA MODEL SUITE — subclass of Qwen35ModelSuite
# Covers: DeepSeek-R1-Distill-Llama-8B (Llama3.1) + Llama-70B (Llama3.3)
# ══════════════════════════════════════════════════════════════════════════════

class DeepSeekR1LlamaModelSuite(Qwen35ModelSuite):
    """
    DeepSeek-R1-Distill (Llama backbone) model suite.

    Same behavioural overrides as DeepSeekR1QwenModelSuite.
    Key difference: __init__ loads Llama AutoTokenizer instead of Qwen tokenizer.

    Calls ModelSuiteProtocol.__init__ directly to skip the Qwen tokenizer load
    in Qwen35ModelSuite.__init__, then loads the Llama tokenizer instead.

    Llama3 chat template supports system role correctly — no merge needed.
    build_initial_messages is inherited from Qwen35ModelSuite (list[dict]).
    """

    def __init__(self, engine, ec: DeepSeekR1LlamaEngineConfig, cfg) -> None:
        # Skip Qwen35ModelSuite.__init__ — it would load the wrong tokenizer
        ModelSuiteProtocol.__init__(self, engine, ec, cfg)
        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            ec.model_path,
            trust_remote_code=False,
        )
        self._code_re = re.compile(r'```python\n(.*?)\n```', re.DOTALL)

    def _extract_post_think(self, text: str) -> str:
        idx = text.rfind('</think>')
        return text[idx + len('</think>'):] if idx != -1 else ''

    def scan_for_answer(self, text: str) -> Optional[int]:
        for region in [self._extract_post_think(text), text]:
            for m in re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', region):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(text)

    def is_final_turn(self, full_text: str, new_messages: list) -> bool:
        has_code      = bool(re.search(r'```python', full_text))
        has_answer    = self.scan_for_answer(full_text) is not None
        has_think_end = '</think>' in full_text
        return (has_think_end and has_answer) or ((not has_code) and has_answer)


In [ ]:
# EXPERIMENTAL
# ══════════════════════════════════════════════════════════════════════════════
# OPENREASONING-NEMOTRON MODEL SUITE — subclass of Qwen35ModelSuite
# Covers: nvidia/OpenReasoning-Nemotron-1.5B / 7B / 14B / 32B
# AIMO-2 competition winning approach (Qwen2.5 backbone, NOT Llama)
# ══════════════════════════════════════════════════════════════════════════════

class OpenReasoningModelSuite(Qwen35ModelSuite):
    """
    OpenReasoning-Nemotron model suite.

    All four sizes (1.5B, 7B, 14B, 32B) use Qwen2.5 as the backbone.
    Trained on 5M reasoning traces distilled from DeepSeek-R1-0528.
    AIMO-2 winning solution was built on this model family.

    Identical implementation to DeepSeekR1QwenModelSuite.
    Kept as a separate class for:
      - Independent suite_id labelling in logs/JSON output
      - Independent prompt tuning without affecting DS-R1 configs
      - Clear intent in ACTIVE_ENGINE_CONFIGS

    No __init__ override — Qwen2.5 tokenizer is correct for all OR variants.
    """

    def _extract_post_think(self, text: str) -> str:
        idx = text.rfind('</think>')
        return text[idx + len('</think>'):] if idx != -1 else ''

    def scan_for_answer(self, text: str) -> Optional[int]:
        """Scan post-think region first, fallback to full text."""
        for region in [self._extract_post_think(text), text]:
            for m in re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', region):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(text)

    def is_final_turn(self, full_text: str, new_messages: list) -> bool:
        has_code      = bool(re.search(r'```python', full_text))
        has_answer    = self.scan_for_answer(full_text) is not None
        has_think_end = '</think>' in full_text
        return (has_think_end and has_answer) or ((not has_code) and has_answer)


In [ ]:
# EXPERIMENTAL
# ══════════════════════════════════════════════════════════════════════════════
# GEMMA3 MODEL SUITE — independent (subclass of ModelSuiteProtocol)
# Covers: google/gemma-3-4b-it / 12b-it / 27b-it
# ══════════════════════════════════════════════════════════════════════════════

class Gemma3ModelSuite(ModelSuiteProtocol):
    """
    Google Gemma3 Instruct model suite.

    Cannot subclass Qwen35ModelSuite because:
      1. Different tokenizer (Gemma3, not Qwen2.5)
      2. System role NOT reliably supported in Gemma3 instruct chat template
         — build_initial_messages merges system + user into one user message
      3. count_prompt_tokens must use the Gemma AutoTokenizer

    Gemma3-27b-it emits <think>...</think> reasoning blocks.
    scan_for_answer prioritises the post-think region (same as DS-R1 suites).

    parse_stream is identical to Qwen35ModelSuite — standard chat completions
    delta streaming, no token-ID buffer, same ARCH-001 MODE-aware loop.
    """

    def __init__(self, engine, ec: Gemma3EngineConfig, cfg) -> None:
        super().__init__(engine, ec, cfg)
        from transformers import AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            ec.model_path,
            trust_remote_code=False,
        )
        self._code_re = re.compile(r'```python\n(.*?)\n```', re.DOTALL)

    # ── Build messages ────────────────────────────────────────────────────────

    def build_initial_messages(self, pb_text: str, system_prompt: str, spirit=None):
        """
        Merge system + spirit + problem into a single user message.

        Gemma3 instruct chat template does not reliably handle the 'system'
        role. Merging avoids template incompatibilities across vLLM versions.
        The preference prompt is appended inline.
        """
        spirit_line = f'{spirit}\n' if spirit else ''
        merged = (
            f'{spirit_line}{system_prompt}\n\n'
            f'{pb_text}\n\n{self.ec.preference_prompt}'
        )
        return [{'role': 'user', 'content': merged}]

    def build_tool_response_message(self, code_output: str) -> dict:
        return {
            'role':    'user',
            'content': (
                f'Code output:\n{code_output}\n\n'
                r'Continue solving. Place your final answer in \boxed{}.'
            ),
        }

    # ── API call ──────────────────────────────────────────────────────────────

    def create_stream(self, messages: list, seed: int, max_tokens: int, temperature: float):
        #self.engine.client.chat.completions.create
        return self.engine.client.completions.create(
            model       = self.ec.served_model_name,
            messages    = messages,
            temperature = temperature,
            max_tokens  = max_tokens,
            seed        = seed,
            stream      = True,
            extra_body  = self.ec.build_extra_body(),
        )

    # ── ARCH-001: MODE-aware streaming ────────────────────────────────────────

    def parse_stream(self, stream, stop_event, deadline, mode) -> tuple:
        """Identical ARCH-001 loop to Qwen35ModelSuite."""
        chunks:     list = []
        mid_answer: Optional[int] = None

        try:
            for chunk in stream:
                if stop_event.is_set() or time.time() > deadline:
                    break
                delta = chunk.choices[0].delta
                txt   = delta.content or ''
                if txt:
                    chunks.append(txt)
                if '}' in txt and mid_answer is None:
                    window    = ''.join(chunks[-self.cfg.search_tokens:])
                    candidate = self.scan_for_answer(window)
                    if candidate is not None:
                        mid_answer = candidate
                        if mode == 'SUBMISSION':
                            break
        finally:
            stream.close()

        full_text = ''.join(chunks)
        # code detection for tool response
        m = self._code_re.search(full_text)
        code = m.group(1) if m else None
        return full_text, code, [], mid_answer

    # ── Turn logic ────────────────────────────────────────────────────────────

    def is_final_turn(self, full_text: str, new_messages: list) -> bool:
        has_code      = bool(re.search(r'```python', full_text))
        has_answer    = self.scan_for_answer(full_text) is not None
        has_think_end = '</think>' in full_text
        return (has_think_end and has_answer) or ((not has_code) and has_answer)

    def scan_for_answer(self, text: str) -> Optional[int]:
        # Post-think region first, fallback to full text
        post = ''
        idx  = text.rfind('</think>')
        if idx != -1:
            post = text[idx + len('</think>'):]
        for region in [post, text]:
            for m in re.findall(r'\\boxed\s*\{\s*([0-9,]+)\s*\}', region):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(text)

    def count_prompt_tokens(self, messages: list) -> int:
        try:
            toks = self.tokenizer.apply_chat_template(
                messages,
                tokenize             = True,
                add_generation_prompt = True,
            )
            return len(toks)
        except Exception:
            return sum(len(m.get('content', '')) for m in messages) // 3

    def handle_final_non_parsable(self, last_text: str, messages) -> Optional[int]:
        for pat in [
            r'(?:answer|result)\s*[=:]\s*([0-9,]+)',
            r'\b([0-9]{1,5})\b(?=\s*$)',
        ]:
            for m in re.findall(pat, last_text.strip()[-500:], re.IGNORECASE):
                try:
                    v = int(m.replace(',', ''))
                    if 0 <= v <= 99999:
                        return v
                except ValueError:
                    pass
        return self._benchmark_boxed_fallback(last_text)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL SUITE FACTORY
# ══════════════════════════════════════════════════════════════════════════════

def make_model_suite(engine, ec: EngineConfig, cfg) -> ModelSuiteProtocol:
    """
    Dispatch on EngineConfig type to return the correct ModelSuiteProtocol.

    ORDER MATTERS: specific subclasses must be checked before their parents.
    All four new suite types are registered here.

    To add a new suite:
      1. Implement a concrete EngineConfig subclass in Block D
      2. Implement a concrete ModelSuiteProtocol subclass in Block E
      3. Add an isinstance branch here (before any parent-class branches)

    Raises ValueError for unregistered EngineConfig types.
    """
    if isinstance(ec, DeepSeekR1QwenEngineConfig):
        return DeepSeekR1QwenModelSuite(engine, ec, cfg)
    if isinstance(ec, DeepSeekR1LlamaEngineConfig):
        return DeepSeekR1LlamaModelSuite(engine, ec, cfg)
    if isinstance(ec, OpenReasoningEngineConfig):
        return OpenReasoningModelSuite(engine, ec, cfg)
    if isinstance(ec, Gemma3EngineConfig):
        return Gemma3ModelSuite(engine, ec, cfg)
    if isinstance(ec, GptOssEngineConfig):
        return GptOssModelSuite(engine, ec, cfg)
    if isinstance(ec, Qwen35EngineConfig):
        return Qwen35ModelSuite(engine, ec, cfg)
    raise ValueError(
        f'No ModelSuiteProtocol registered for {type(ec).__name__}. '
        'Implement a subclass and add a case to make_model_suite().'
    )



## Block F - Blackboard, Voting, and Answer Checker

This block implements the shared state layer and the two functions that act on it:
the effort-weighted voting aggregation and the correctness checker.

#### The Blackboard pattern

The Blackboard is a coordination architecture with roots in classical AI
problem solving. The original formulation - articulated in the Hearsay-II
speech understanding system at Carnegie Mellon in the mid-1970s and formalised
by Hayes-Roth in the BB1 system (1985) - describes a shared global workspace
to which independent specialist agents (called Knowledge Sources) contribute
partial solutions. A controller monitors the workspace and decides when
sufficient evidence has accumulated to commit to a solution.

The appeal of the pattern is precisely its passivity: the Blackboard does not
instruct agents, schedule work, or impose a particular reasoning strategy. It
accumulates evidence. The intelligence of the system emerges from the
combination of independent contributions, not from coordination.

This notebook applies that pattern directly. `SolverAgent` and `MetaAgent`
instances run concurrently and independently. Each posts results to the shared
`Blackboard` without any knowledge of what other agents have produced. The
`Controller` reads snapshots of the Blackboard, calls `aggregate_answers()` on
each snapshot, and fires a stop condition when the accumulated evidence crosses
a threshold. No agent-to-agent communication occurs.

See: Hayes-Roth, B. (1985). *A Blackboard Architecture for Control*. Artificial
Intelligence, 26(3), 251-321. Also:
[Hearsay-II overview, CMU](https://www.cs.cmu.edu/Groups/AI/html/cltl/cltl2.html).

#### Blackboard structure

The `Blackboard` class holds two distinct levels of state with different
lifetimes and reset semantics.

**Problem-level state** is cleared at the start of each problem by
`reset_for_problem()`. It accumulates:
- `all_results`: every `AttemptResult` posted by any agent, in posting order.
- `answers`: every `AnswerRecord` derived from non-`None` answers. This is the
  primary input to `aggregate_answers()`.
- `meta_answers`: integer answers produced by MetaAgent instances, stored in a
  separate list under its own lock, never mixed into the main answer pool.

**Run-level state** persists across all 50 problems in a `RunTracker` instance.
It accumulates one `ProblemSummary` per solved problem and maintains
`RunStats`. The most operationally important statistic is the rolling average
solve time returned by `last_n_avg(5)`, which drives the adaptive deadline
formula in `AIMO3Solver._compute_deadline()`:

```
reserved = (problems_remaining - 1) x last_n_avg(5)
budget   = clamp(time_left - reserved,
                 cfg.base_problem_timeout,
                 cfg.high_problem_timeout)
```

A sequence of fast problems lowers the rolling average, expanding the budget
available for subsequent problems. A sequence of slow problems raises it,
contracting the budget and protecting the remaining run from time starvation.

#### Thread safety

A single `threading.Lock` protects all writes. All read methods (`snapshot()`,
`get_all_results()`, `get_answers()`) return copies, not references. Callers
hold a consistent snapshot even as agents continue posting concurrently.

Deduplication of incremental versus final posts is handled in
`get_all_results()` at read time, by keying on `(agent_name, attempt_index)`
and retaining only the later post. Performing deduplication at read time rather
than write time keeps the write path - and the lock-hold duration - minimal.

#### AnswerRecord

`AnswerRecord` is the lightweight entry posted to `Blackboard.answers` whenever
an agent produces a non-`None` answer. Its `response_length` field - total
tokens generated across all TIR turns for this attempt - is the effort proxy
used by `aggregate_answers()`. Its `last_3_turns` field carries a short
serialised excerpt of the conversation, included in the context block passed
to MetaAgent instances for informed-but-not-anchored reconciliation.

#### aggregate_answers() - pure voting function

`aggregate_answers()` is a pure function with no side effects and no shared
state. It accepts a list of `AnswerRecord` instances and parameters for all
three stop conditions, and returns a dictionary of results. The `Controller`
calls it after every posted result to evaluate whether any stop condition has
been met.

The effort-weighted distribution is computed as:

```
weight(X)     = sum of response_length for all records where answer == X
effort_pct(X) = weight(X) / sum of weight(all answers)
winner        = argmax weight(X)
```

Three stop conditions are evaluated against this distribution:

**Stop-A** (`stop_A`): `raw_votes[winner] >= k2_raw_vote` (default: 4).
Fast path. Fires when four or more agents independently produce the same
answer. On easy problems where all agents converge quickly, this condition
fires before enough answers accumulate for effort weighting to be meaningful.
It releases problem budget without waiting for the quality gate.

**Stop-B** (`stop_B`): `len(answers) >= k1_min_valid` AND
`effort_pct(winner) >= effort_threshold` (defaults: 6 and 0.50).
Quality gate. Requires a minimum evidence count before evaluating effort
concentration, preventing premature firing when only one or two agents have
completed. An effort share of 0.50 means the leading answer has accumulated
more tokens than all other answers combined.

**Stop-C** (`stop_meta`): `Counter(meta_answers).most_common(1)[0][1] >= meta_vote_stop`
(default: 3). Independent reconciliation consensus. Fires only when MetaAgents
are enabled and at least three have independently produced the same answer.
MetaAgent answers are never mixed into the main answer pool and do not
contribute to Stop-A or Stop-B.

The function returns `stop_any = stop_A or stop_B or stop_meta`. The
`Controller` checks `stop_any` on every result posting.


### AnswerChecker - BENCHMARK Mode Only

`AnswerChecker` assigns a correctness label to each `AttemptResult` for
post-hoc analysis. It is never instantiated in SUBMISSION mode.

Labelling proceeds on a two-path cascade, tried in order.

**Path 1 - integer exact match**: if both the agent answer and the expected
answer parse cleanly as integers, a direct equality comparison is performed.
This path requires no API call, executes in $O(1)$, and handles the entire
AIMO3 problem set, since all competition answers are integers in $[0, 99999]$.

**Path 2 - LLM judge**: invoked when integer parsing fails on either side,
which occurs when the reference CSV contains a symbolic expression or a
non-integer string as the expected answer. The judge receives the problem text,
the expected answer, and the agent answer, and is instructed to respond with
exactly one word: `CORRECT` or `INCORRECT`. Any other response is recorded as
`llm_judge_parse_fail` with `correct=None`.

The judge uses the inference engine at index `JUDGE_ENGINE_IDX` (default: 0,
the first active engine) via a non-streaming `/chat/completions` call with
`temperature=0.0` and `max_tokens=8`. No separate model or endpoint is required.

Six `eval_method` values are recorded in each `AttemptResult`:

| Value | Meaning |
|---|---|
| `no_answer` | `result.answer` is `None`; agent produced no parsable answer. |
| `exact_match` | Integer equality check succeeded (correct or incorrect). |
| `llm_judge` | LLM returned `CORRECT`. |
| `llm_judge_incorrect` | LLM returned `INCORRECT`. |
| `llm_judge_parse_fail` | LLM response was not `CORRECT` or `INCORRECT`. |
| `llm_judge_error` | API call raised an exception. |

These labels are written to the per-attempt JSON files and enable per-suite, per-model,
and per-stop-condition accuracy analysis in BENCHMARK mode.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANSWER RECORD — Lightweight answer entry posted to the Blackboard
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class AnswerRecord:
    """
    One record per valid (non-None) answer, posted to Blackboard.answers.

    response_length is the total tokens generated by this agent across all
    TIR turns — the effort proxy used for weighted voting in aggregate_answers().

    attempt_index is used by _deduplicated_answers() to keep only the last-
    posted record per (agent_name, attempt_index) pair — ensuring incremental
    and final posts from the same agent count as a single vote.

    last_3_turns is a short serialised excerpt of the conversation, included
    in the context block passed to MetaAgent for informed reconciliation.
    """
    agent_name:      str
    agent_suite:     str
    attempt_index:   int
    answer:          object  # int or str (BENCHMARK non-integer)
    response_length: int
    timestamp:       float
    last_3_turns:    list  = field(default_factory=list)


# ══════════════════════════════════════════════════════════════════════════════
# RUN TRACKER — Accumulates ProblemSummary records across the full 50-problem run
# ══════════════════════════════════════════════════════════════════════════════

class RunTracker:
    """
    Persistent across all problems. Updated by Blackboard.finalize_problem().

    The rolling average solve time drives adaptive deadline computation in
    AIMO3Solver._compute_deadline():

        reserved = (problems_remaining - 1) × last_n_avg(5)
        budget   = clamp(time_left - reserved,
                         cfg.base_problem_timeout,
                         cfg.high_problem_timeout)

    This ensures that slow problems do not starve later problems, and that
    fast early problems release time budget for harder later ones.
    """

    def __init__(self) -> None:
        self.summaries:  list = []
        self.run_stats:  RunStats = RunStats(run_start_time=time.time())

    def record(self, summary: ProblemSummary) -> None:
        """Append summary and update all RunStats fields atomically."""
        self.summaries.append(summary)
        rs = self.run_stats
        t  = summary.solve_duration_sec

        rs.total_problems_seen    += 1
        rs.total_solve_time_sec   += t
        rs.avg_solve_time_sec      = rs.total_solve_time_sec / rs.total_problems_seen
        rs.fastest_problem_sec     = min(rs.fastest_problem_sec, t)
        rs.slowest_problem_sec     = max(rs.slowest_problem_sec, t)

        if summary.n_correct is not None:
            rs.total_problems_correct += summary.n_correct

    def last_n_avg(self, n: int = 5) -> float:
        """Average solve time of the most recent n problems. Used for adaptive deadline."""
        recent = self.summaries[-n:]
        if not recent:
            return 0.0
        return sum(s.solve_duration_sec for s in recent) / len(recent)

    def print_run_summary(self) -> None:
        rs  = self.run_stats
        n   = rs.total_problems_seen
        elapsed = time.time() - rs.run_start_time

        print('\n' + '═' * 60)
        print(f'  Run Summary — {n} problems')
        print('═' * 60)
        print(f'  Total wall time  : {elapsed/60:.1f} min')
        print(f'  Avg solve time   : {rs.avg_solve_time_sec:.1f}s')
        print(f'  Fastest problem  : {rs.fastest_problem_sec:.1f}s')
        print(f'  Slowest problem  : {rs.slowest_problem_sec:.1f}s')
        if rs.total_problems_correct > 0:
            acc = rs.total_problems_correct / n * 100
            print(f'  Accuracy         : {rs.total_problems_correct}/{n} = {acc:.1f}%')
        print('═' * 60 + '\n')


# ══════════════════════════════════════════════════════════════════════════════
# BLACKBOARD — Shared state for one problem solve
# ══════════════════════════════════════════════════════════════════════════════

class Blackboard:
    """
    Thread-safe shared state store. One instance per AIMO3Solver.

    Two levels of state:
      Problem-level: reset by reset_for_problem() at the start of each problem.
      Run-level:     RunTracker, never reset, persists for the full 50-problem run.

    Write path (post_result) holds the lock for the minimum time required.
    Read path returns copies — callers see a consistent snapshot.
    """

    def __init__(self) -> None:
        self._lock        = threading.Lock()
        self.run_tracker  = RunTracker()

        # Problem-level state (set by reset_for_problem)
        self.pb_id        = ''
        self.pb_text      = ''
        self.pb_start     = 0.0
        self.answers:     list = []   # list[AnswerRecord] — valid answers only
        self.all_results: list = []   # list[AttemptResult] — all attempts

    # ── Problem lifecycle ─────────────────────────────────────────────────────

    def reset_for_problem(self, pb_id: str, pb_text: str) -> None:
        """Clear problem-level state. Must be called before spawning agents."""
        with self._lock:
            self.pb_id              = pb_id
            self.pb_text            = pb_text
            self.pb_start           = time.time()
            self.answers            = []
            self.all_results        = []
            self.meta_reasoning_traces: list = []   # BENCHMARK: one entry per MetaAgent run

    def finalize_problem(
        self,
        final_answer:    object,  # int or str (BENCHMARK non-integer)
        stop_reason:     str,
        expected_answer: Optional[str] = None,
    ) -> ProblemSummary:
        """
        Build and record a ProblemSummary from current state.
        Called by AIMO3Solver after the solve loop completes.
        """
        with self._lock:
            results = self._deduplicated_results()
            answers = list(self.answers)

        valid      = [r for r in results if r.answer_complete]
        n_correct  = sum(1 for r in valid if r.correct) if any(
            r.correct is not None for r in valid
        ) else None

        # Vote distributions
        raw_votes    = Counter(r.answer for r in valid)
        effort_by    = defaultdict(int)
        for r in valid:
            effort_by[r.answer] += r.response_length
        total_effort  = sum(effort_by.values()) or 1
        effort_pct    = effort_by.get(final_answer, 0) / total_effort

        summary = ProblemSummary(
            pb_id               = self.pb_id,
            pb_text_len         = len(self.pb_text),
            n_agents_fired      = len(results),
            n_valid_answers     = len(valid),
            n_correct           = n_correct,
            final_answer        = final_answer,
            expected_answer     = expected_answer,
            solve_duration_sec  = time.time() - self.pb_start,
            stop_reason         = stop_reason,
            answer_distribution = dict(raw_votes),
            effort_distribution = dict(effort_by),
            weighted_pct_winner = effort_pct,
            timestamp           = time.time(),
        )
        self.run_tracker.record(summary)
        return summary

    # ── Write path ────────────────────────────────────────────────────────────

    def post_result(self, result: AttemptResult, suite: 'ModelSuiteProtocol') -> None:
        """
        Post one AttemptResult from an agent.

        If the result carries a valid answer:
          - build an AnswerRecord (lightweight) and append to self.answers
          - last_3_turns is populated from the serialised conversation tail
        Always append to self.all_results regardless of answer presence.

        Both writes are done under a single lock acquisition.

        Stale-result guard: agents from a previous problem may still be running
        after timeout (shutdown(wait=False)). Reject any result whose pb_id
        does not match the current problem to prevent cross-problem contamination.
        """
        if result.pb_id != self.pb_id:
            return  # stale agent from a previous problem — discard
        record = None
        if result.answer is not None:
            # Serialize last 3 turns for MetaAgent context (BENCHMARK: use messages)
            turns = []
            if result.messages:
                serialized = _serialize(result.messages, suite) if suite else result.messages
                turns = serialized[-3:] if len(serialized) >= 3 else serialized
            record = AnswerRecord(
                agent_name      = result.agent_name,
                agent_suite     = result.agent_suite,
                attempt_index   = result.attempt_index,
                answer          = result.answer,
                response_length = result.response_length,
                timestamp       = time.time(),
                last_3_turns    = turns,
            )

        with self._lock:
            self.all_results.append(result)
            if record is not None:
                self.answers.append(record)

    # ── Read path (snapshot copies) ───────────────────────────────────────────

    def get_answers_snapshot(self) -> list:
        """
        Thread-safe deduplicated copy of valid AnswerRecords.

        Deduplication mirrors get_all_results(): for each (agent_name,
        attempt_index) pair keep only the last-posted AnswerRecord.

        Without this, incremental posts (is_incremental=True) and final
        posts (is_incremental=False) from the same agent both land in
        self.answers, making one agent count as TWO votes in
        aggregate_answers() and halving the effective k2_raw_vote_stop
        threshold.
        """
        with self._lock:
            return self._deduplicated_answers()

    def get_all_results(self) -> list:
        """
        Thread-safe deduplicated copy of all AttemptResults.

        Deduplication: for each (agent_name, attempt_index) pair, keep the
        last-posted entry. This ensures that a final post (is_incremental=False)
        always supersedes an earlier incremental post (is_incremental=True)
        from the same attempt.
        """
        with self._lock:
            return self._deduplicated_results()

    def get_run_stats(self) -> RunStats:
        """Thread-safe copy of current RunStats."""
        with self._lock:
            rs = self.run_tracker.run_stats
            return RunStats(
                total_problems_seen    = rs.total_problems_seen,
                total_problems_correct = rs.total_problems_correct,
                total_solve_time_sec   = rs.total_solve_time_sec,
                avg_solve_time_sec     = rs.avg_solve_time_sec,
                fastest_problem_sec    = rs.fastest_problem_sec,
                slowest_problem_sec    = rs.slowest_problem_sec,
                run_start_time         = rs.run_start_time,
            )

    def post_meta_trace(self, trace: dict) -> None:
        """
        Store a MetaAgent reasoning trace (BENCHMARK mode only).
        trace = {pb_id: str, agent_name: str, answer: int|None, messages: list[dict]}

        Stale-result guard: same as post_result — reject traces from previous problems.
        """
        if trace.get('pb_id') != self.pb_id:
            return  # stale MetaAgent from a previous problem — discard
        with self._lock:
            self.meta_reasoning_traces.append(trace)

    def get_meta_traces(self) -> list:
        """Thread-safe copy of all MetaAgent reasoning traces."""
        with self._lock:
            return list(self.meta_reasoning_traces)

    # ── Internal ──────────────────────────────────────────────────────────────

    def _deduplicated_results(self) -> list:
        """
        Keep last-posted AttemptResult per (agent_name, attempt_index).
        Called under self._lock — must not acquire lock again.
        """
        seen:   dict = {}
        for r in self.all_results:
            seen[(r.agent_name, r.attempt_index)] = r
        return list(seen.values())

    def _deduplicated_answers(self) -> list:
        """
        Keep last-posted AnswerRecord per (agent_name, attempt_index).
        Called under self._lock — must not acquire lock again.

        AnswerRecords carry attempt_index via the AttemptResult they are
        built from. We store it on AnswerRecord at post time so dedup
        can use the same key as _deduplicated_results().
        """
        seen: dict = {}
        for rec in self.answers:
            seen[(rec.agent_name, rec.attempt_index)] = rec
        return list(seen.values())


def _elapsed(blackboard) -> str:
    """Format seconds elapsed since current problem started, for log prefixes."""
    return f'[{time.time() - blackboard.pb_start:6.1f}s]'

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AGGREGATE ANSWERS — Pure voting function
# No side effects. Safe to call from any thread at any frequency.
# ══════════════════════════════════════════════════════════════════════════════

def aggregate_answers(
    answers:          list,
    k1_min_valid:     int   = 4,
    effort_threshold: float = 0.5,
    k2_raw_vote:      int   = 4,
    meta_answers:     Optional[list] = None,
    meta_vote_stop:   int   = 3,
) -> dict:
    """
    Compute effort-weighted vote distribution and evaluate all stop conditions.

    Args:
        answers:          list[AnswerRecord] — snapshot from Blackboard
        k1_min_valid:     Stop-B minimum valid answer count
        effort_threshold: Stop-B minimum effort share for the leading answer
        k2_raw_vote:      Stop-A raw vote majority threshold
        meta_answers:     list[int] from MetaAgent posts, or None
        meta_vote_stop:   Stop-C meta consensus threshold

    Returns:
        dict with keys:
          winner          — answer with highest effort weight (int or None)
          weighted_pct    — effort share of winner (0.0–1.0)
          raw_votes       — Counter({answer: count})
          stop_A          — raw majority threshold reached
          stop_B          — quality gate (count + effort) reached
          stop_meta       — meta-agent consensus reached
          stop_any        — any stop condition is True

    Voting mechanism:
        weight(X)        = Σ response_length for all AnswerRecords with answer X
        effort_pct(X)    = weight(X) / Σ weight(all answers)

    An agent that generated more tokens contributes proportionally more
    evidence. This is a proxy for reasoning depth — longer, iterative
    TIR traces are treated as stronger evidence than short single-pass answers.
    """
    if not answers:
        return dict(
            winner=None, weighted_pct=0.0,
            raw_votes=Counter(), stop_A=False,
            stop_B=False, stop_meta=False, stop_any=False,
        )

    # ── Effort-weighted vote distribution ─────────────────────────────────────
    effort_by:  dict = defaultdict(int)
    for rec in answers:
        effort_by[rec.answer] += rec.response_length

    total_effort = sum(effort_by.values()) or 1
    winner       = max(effort_by, key=effort_by.get)
    weighted_pct = effort_by[winner] / total_effort
    raw_votes    = Counter(rec.answer for rec in answers)

    # ── Stop conditions ───────────────────────────────────────────────────────
    # Stop-A: absolute raw majority — fast path for easy problems
    stop_A = raw_votes[winner] >= k2_raw_vote

    # Stop-B: sufficient evidence count AND effort concentration
    stop_B = (len(answers) >= k1_min_valid) and (weighted_pct >= effort_threshold)

    # Stop-C: independent MetaAgent consensus
    stop_meta = False
    if meta_answers:
        meta_counts = Counter(meta_answers)
        top_meta    = meta_counts.most_common(1)[0]
        stop_meta   = top_meta[1] >= meta_vote_stop

    stop_any = stop_A or stop_B or stop_meta

    return dict(
        winner       = winner,
        weighted_pct = weighted_pct,
        raw_votes    = raw_votes,
        stop_A       = stop_A,
        stop_B       = stop_B,
        stop_meta    = stop_meta,
        stop_any     = stop_any,
    )

### AnswerChecker (BENCHMARK Mode Only)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANSWER CHECKER — BENCHMARK mode only
# ══════════════════════════════════════════════════════════════════════════════

class AnswerChecker:
    """
    Labels AttemptResult objects with correctness information.

    Two-path cascade:
      1. Integer exact match (no API call, O(1))
      2. LLM judge call (fallback for symbolic/non-integer expected answers)

    Returns (correct: bool | None, eval_method: str) tuples.
    Thread-safe: stateless — all state is passed as arguments.
    """

    _JUDGE_SYSTEM = (
        'You are a mathematics answer checker. '
        'Compare the student answer to the expected answer for the given problem. '
        'Answers may be integers, fractions, radicals, symbolic expressions, '
        'sets, tuples, or other mathematical objects. '
        'Two answers are equivalent if they represent the same mathematical value or object, '
        'even if written in different notation (e.g. 1/2 vs 0.5 vs \\frac{1}{2}). '
        'Reply with exactly one word: CORRECT or INCORRECT.'
    )

    def __init__(self, engines: list) -> None:
        """
        Args:
            engines: list[InferenceEngineManager] — uses JUDGE_ENGINE_IDX
        """
        self._engines = engines

    # ── Public interface ──────────────────────────────────────────────────────

    def check(
        self,
        result:          'AttemptResult',
        expected:        str,
        pb_text:         str,
    ) -> tuple:
        """
        Check whether result.answer matches expected.

        Returns:
            (correct: bool | None, eval_method: str)

        eval_method values:
            'no_answer'           — result.answer is None
            'exact_match'         — integer equality
            'string_exact_match'  — normalized string equality
            'llm_judge'           — LLM verified as correct
            'llm_judge_incorrect' — LLM verified as incorrect
            'llm_judge_parse_fail'— LLM response not parseable
            'llm_judge_error'     — API call raised an exception
        """
        
        #return self._llm_judge(str(result), expected, pb_text)
        
        if result.answer is None:
            return None, 'no_answer'

        # Path 1 — integer exact match (both sides must parse as int)
        try:
            if int(result.answer) == int(expected.strip()):
                return True, 'exact_match'
            else:
                # Both are ints but different — definitively wrong
                return False, 'exact_match'
        except (ValueError, AttributeError, TypeError):
            pass

        # Path 1b — normalized string exact match (for symbolic answers)
        student_raw = result.answer_raw if result.answer_raw else str(result.answer)
        expected_clean = expected.strip()
        student_clean  = student_raw.strip()
        # Normalize whitespace and LaTeX spacing for comparison
        def _normalize(s):
            s = re.sub(r'\s+', ' ', s.strip())
            s = s.replace('\\,', '').replace('\\;', '').replace('\\!', '')
            s = s.replace('\\left', '').replace('\\right', '')
            s = s.replace(' ', '')
            return s.lower()
        if _normalize(student_clean) == _normalize(expected_clean):
            return True, 'string_exact_match'

        # Path 2 — LLM judge (use raw boxed string if available)
        return self._llm_judge(student_raw, expected_clean, pb_text)

    # ── Internal ──────────────────────────────────────────────────────────────

    def _llm_judge(
        self,
        student_answer:  str,
        expected:        str,
        pb_text:         str,
    ) -> tuple:
        """
        Ask the first available engine to verify correctness.
        Uses /chat/completions with a minimal prompt for reliability.
        """
        if not self._engines:
            return None, 'llm_judge_error'

        client = self._engines[JUDGE_ENGINE_IDX].client
        model  = self._engines[JUDGE_ENGINE_IDX].model_name
        print(f"Activate LLM Judge - Expected {expected} - Result {student_answer}")
        user_prompt = (
            f'Problem:\n{pb_text}\n\n'
            f'Expected answer: {expected}\n'
            f'Student answer:  {student_answer}\n\n'
            'Are they equivalent? Reply CORRECT or INCORRECT only.'
        )

        try:
            resp = client.chat.completions.create(
                model       = model,
                messages    = [
                    {'role': 'system', 'content': self._JUDGE_SYSTEM},
                    {'role': 'user',   'content': user_prompt},
                ],
                temperature = 0.5, # Increase Temp from 0 to 0.5 to avoid unexpected stop
                max_tokens  = 8192,
                stream      = False,
            )
            verdict = resp.choices[0].message.content.strip().upper()
            print(verdict)
            if verdict.startswith('CORRECT'):
                return True,  'llm_judge'
            if verdict.startswith('INCORRECT'):
                return False, 'llm_judge_incorrect'
            return None, 'llm_judge_parse_fail'

        except Exception as exc:
            print(f'  [AnswerChecker] LLM judge error: {exc}')
            return None, 'llm_judge_error'

## Block G - Controller, SolverAgent, and MetaAgent

This block implements the three actor classes that constitute the active
reasoning layer of the solver. All three share a `Blackboard` and a
`threading.Event` (the stop signal). Communication between actors is
exclusively through these two shared objects. No direct inter-actor calls
occur; no mutable state is shared outside the `Blackboard`.

#### Thread model

```
Main thread                          ThreadPoolExecutor workers
    |
    |  AIMO3Solver.solve_problem()
    |      |
    |      └- Controller.run()
    |              |
    |              ├- submit SolverAgent x N ──> SolverAgent.run()  [worker]
    |              |                                  └- TIR loop
    |              |   as_completed loop                   └- parse_stream (ARCH-001)
    |              |       |                               └- blackboard.post_result()
    |              |       ├- aggregate_answers()
    |              |       ├- stop condition check
    |              |       └- spawn MetaAgent (on divergence) ──> MetaAgent.run()  [worker]
    |              |                                                   └- TIR loop (capped at max_turns // 2)
    |              |   stop_event.set()                                └- meta_answers.append()
    |              └- _resolve_final_answer()
```

The `Controller` runs on the main thread. All `SolverAgent` and `MetaAgent`
instances run as `ThreadPoolExecutor` workers. The executor is shared and
sized to accommodate all primary solvers plus the maximum number of
concurrent MetaAgents.

#### SolverAgent - primary solver

`SolverAgent` is the primary reasoning unit. Each agent instance executes one
independent Tool-Integrated Reasoning (TIR) loop for a given problem. The
TIR loop is a think-code-execute cycle:

1. Submit the current conversation to the model via `create_stream()`.
2. Parse the streaming response (ARCH-001): detect `\boxed{}` mid-stream,
   respect the deadline and stop signal, break early in SUBMISSION mode.
3. If the response contains a Python code block, execute it in the agent's
   `JupyterSandbox` and inject the output as a tool response.
4. If the response signals a final answer (`is_final_turn()`), post to the
   `Blackboard` and exit.
5. Otherwise, repeat from step 1 up to `cfg.max_turns` times.

Each agent acquires one `JupyterSandbox` from the shared pool at the start
of its attempt and holds it for the duration of the TIR loop. Kernel state
accumulates across turns: variables defined in turn $k$ are accessible in
turn $k+n$. The sandbox is reset and returned to the pool in a `finally`
block regardless of how the attempt terminates.

An agent posts to the `Blackboard` twice per attempt:

- **Incremental post** (`is_incremental=True`): when `parse_stream` detects
  an answer mid-stream, the agent immediately posts a partial `AttemptResult`
  to the `Blackboard`. In SUBMISSION mode this happens before the agent has
  finished generating; the `Controller` can therefore evaluate stop conditions
  and cancel other agents before any of them completes its current turn.

- **Final post** (`is_incremental=False`): on attempt completion, the full
  `AttemptResult` (with complete `response_length`, `messages`, timing, and
  correctness fields) supersedes the incremental post via Blackboard
  deduplication on `(agent_name, attempt_index)`.

#### Agent diversity

Each agent is assigned an independent identity drawn at configuration time
from two sources of variation:

- **Mathematical spirit**: a one-sentence persona prefix drawn from a pool of
  16 historical mathematicians (Euler, Gauss, Ramanujan, Hilbert, and others).
  The spirit is prepended to the system prompt, nudging the agent toward a
  different initial orientation without altering its capability or the protocol.

- **Temperature**: sampled independently per agent from
  `[cfg.temp_agent_min, cfg.temp_agent_max]`. Different temperatures produce
  different sampling paths through the model's probability distribution,
  increasing the probability that at least one agent explores the reasoning
  path that leads to a correct answer.

These two sources of diversity are designed to increase the variance of the
agent population's approaches without introducing capability inequality or
inter-agent communication.


### MetaAgent - Secondary Solver for Hard Conflicts

The `SolverAgent` pool handles the majority of problems through direct
convergence: agents agree on an answer (Stop-A) or one answer accumulates
dominant effort weight (Stop-B). On hard problems, neither condition fires
before the agent population splits into two or more distinct answer groups.
Naive voting cannot resolve this situation reliably, because the split may
reflect a genuine difficulty - multiple plausible but incorrect approaches
each capturing some agents - rather than noise.

The `MetaAgent` is a secondary solver activated precisely for this case. It
is not an adjudicator. It does not read solver traces and select the most
plausible one. It is an independent problem solver that is given three things
in addition to the original problem statement:

1. The list of competing answers and their raw vote counts.
2. A brief excerpt (last 3 serialised turns) from the highest-effort agent
   in each competing group.

It is then instructed, via `_META_SYSTEM_SUFFIX` appended to the system
prompt, to solve the problem from scratch using its own reasoning and Python
tools, without echoing or voting on the proposals. This protocol is designed
to be *informed but not anchored*: the competing answers narrow the search
space without directing the MetaAgent toward any particular one.

The MetaAgent's TIR loop is structurally identical to `SolverAgent.run()`,
with two differences:

- Turn count is capped at `max_turns // 2`, limiting resource consumption.
- On finding an answer, the agent appends to the shared `meta_answers` list
  (under `meta_lock`) and exits immediately.

The `Controller` evaluates Stop-C via `aggregate_answers(meta_answers=...)` after
each MetaAgent future completes. When three MetaAgents independently produce the
same answer, Stop-C fires and that answer is selected.

MetaAgent spawning conditions:
- At least two distinct answers exist in the current Blackboard snapshot.
- No stop condition has yet fired.
- The total number of MetaAgents spawned for this problem is below `N_META_AGENTS_MAX`.
- `ENABLE_META_AGENT` is `True`.

The engine used for MetaAgent calls is the one at index `META_AGENT_ENGINE_IDX`
(default: 0). In a multi-suite configuration, this can be a different model family
from the primary solvers.

### Controller - Orchestrator

The `Controller` runs on the main thread and manages the complete lifecycle of
one problem solve.

**Agent dispatch**: for each active engine, the Controller creates `ec.n_agents`
`SolverAgent` instances, each with a unique name (drawn from `GAMER_ALPHABET`),
an independently drawn spirit and temperature, and an attempt index. All
agents are submitted to the `ThreadPoolExecutor` simultaneously.

**Monitoring loop**: the Controller drives an `as_completed()` loop over all
submitted futures. On each completed future, it takes a Blackboard snapshot and
calls `aggregate_answers()`. If `stop_any` is `True`, it sets `stop_event`,
cancels pending futures, and breaks.

If `ENABLE_META_AGENT` is `True` and the snapshot shows at least two distinct
answers with no stop condition fired, the Controller spawns a MetaAgent future
into the same executor.

**Deadline enforcement**: the monitoring loop checks the deadline on each
iteration. If the deadline is exceeded before any stop condition fires, the
loop exits with `stop_reason = 'stop_D'` and `stop_event` is set.

**Final answer fallback chain**: after the monitoring loop exits,
`_resolve_final_answer()` applies the following priority cascade:

| Priority | Source | Stop reason tag |
|---|---|---|
| 1 | Effort-weighted winner from `aggregate_answers()` | inherited from stop condition |
| 2 | Combined raw vote across SolverAgent + MetaAgent answers | `fallback_combined_vote` |
| 3 | Highest-effort valid answer from SolverAgents only | `fallback_aggregate` |
| 4 | `handle_final_non_parsable()` on the last result's text - BENCHMARK mode only | `fallback_nonparsable` |
| 5 | `0` | `fallback_zero` |

Priority 2 is the primary safety net when no stop condition fired. It pools
all available evidence - SolverAgent answers and any MetaAgent answers that
did not reach the `meta_vote_stop` threshold - into a single raw vote. A
scenario where SolverAgents split evenly (4 vs 4) and two MetaAgents agree on
one side will be resolved correctly here, whereas the original chain would have
discarded the MetaAgent evidence entirely.

Priority 4 is excluded from SUBMISSION mode. The LLM extraction call via
`handle_final_non_parsable()` adds latency that is not acceptable on the
competition hot path. It is available in BENCHMARK mode for diagnostic purposes.

The default of `0` at priority 5 is never a random guess. It is the explicit
signal that the solver produced no parsable evidence for any answer. All JSON
output still records the full vote distribution, stop reason, and timing for
post-hoc analysis.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SOLVER AGENT
# One independent TIR loop. Acquires one JupyterSandbox from the shared pool.
# Posts to Blackboard incrementally (mid-stream) and on completion (final).
# ══════════════════════════════════════════════════════════════════════════════

class SolverAgent:
    """
    Executes one Tool-Integrated Reasoning (TIR) attempt for a given problem.

    A single agent operates a think → code → execute → observe loop:
      1. Submit conversation to the model (streaming)
      2. Detect Python code blocks in the response
      3. Execute code in the persistent JupyterSandbox
      4. Inject the output back as a tool response message
      5. Repeat until the model signals a final answer or token budget exhausted

    Answer detection occurs *during* streaming (ARCH-001): when a \\boxed{}
    marker is found in the sliding window, the answer is posted to the Blackboard
    immediately as an incremental result. The stream is broken early in SUBMISSION
    mode; in BENCHMARK mode the full trace is preserved for JSON output.

    The final AttemptResult (is_incremental=False) posted on completion supersedes
    the earlier incremental post via Blackboard.get_all_results() deduplication.
    """

    def __init__(
        self,
        name:          str,
        spirit:        Optional[str],
        suite:         ModelSuiteProtocol,
        sandbox_pool:  queue.Queue,
        blackboard:    Blackboard,
        cfg,
        attempt_index: int,
        seed:          int,
        temperature:   float,
        mode:          str,
    ) -> None:
        self.name          = name
        self.spirit        = spirit
        self.suite         = suite
        self.sandbox_pool  = sandbox_pool
        self.blackboard    = blackboard
        self.cfg           = cfg
        self.attempt_index = attempt_index
        self.seed          = seed
        self.temperature   = temperature
        self.mode          = mode

    def run(
        self,
        pb_id:      str,
        pb_text:    str,
        stop_event: threading.Event,
        deadline:   float,
    ) -> AttemptResult:
        """Execute the TIR loop and return a completed AttemptResult."""
        t_start       = time.time()
        answer        = None
        total_tokens  = 0
        python_calls  = 0
        python_errors = 0
        posted_incr   = False
        final_text    = ''
        sandbox       = None
        ec            = self.suite.ec

        messages = self.suite.build_initial_messages(
            pb_text, ec.system_prompt, self.spirit
        )

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            for _turn in range(self.cfg.max_turns):

                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_tokens = self.suite.count_prompt_tokens(messages)
                max_new = (
                    self.cfg.agent_context_tokens
                    - prompt_tokens
                    - self.cfg.buffer_tokens
                )
                if max_new <= self.cfg.buffer_tokens:
                    break

                stream = self.suite.create_stream(
                    messages, seed=self.seed,
                    max_tokens=max_new, temperature=self.temperature,
                )
                full_text, code, new_msgs, mid_answer = self.suite.parse_stream(
                    stream, stop_event, deadline, self.mode
                )
                _append_messages(messages, new_msgs)
                total_tokens += max(1, len(full_text) // 4)
                final_text    = full_text

                # ── Mid-stream answer → incremental post ───────────────────
                if answer is None and mid_answer is not None:
                    answer = mid_answer
                    if not posted_incr:
                        self._post_incremental(
                            pb_id, answer, total_tokens,
                            python_calls, python_errors, t_start, messages,
                        )
                        posted_incr = True

                # ── Post-stream safety scan ────────────────────────────────
                if answer is None:
                    candidate = self.suite.scan_for_answer(full_text)
                    if candidate is not None:
                        answer = candidate
                        if not posted_incr:
                            self._post_incremental(
                                pb_id, answer, total_tokens,
                                python_calls, python_errors, t_start, messages,
                            )
                            posted_incr = True

                # ── Final turn ─────────────────────────────────────────────
                if self.suite.is_final_turn(full_text, new_msgs):
                    if answer is None:
                        answer = self.suite.handle_final_non_parsable(
                            final_text, messages
                        )
                    break

                # ── Code execution ─────────────────────────────────────────
                if code is not None:
                    python_calls += 1
                    
                    try:
                        print(f" [{self.name} - {pb_id} - {self.suite.ec.suite_id}]  | Turn {_turn} | {_elapsed(self.blackboard)} | Tokens Per Second Est {round((prompt_tokens+len(full_text)/4)/(_elapsed_duration_nb(self.blackboard)+0.1),1) } | running code",len(str(code)),"chars")
                    except Exception as ee: print(e)
                        
                    raw_output = sandbox.execute(code, timeout=self.cfg.jupyter_timeout)
                    if '[ERROR]' in raw_output:
                        python_errors += 1
                    _append_messages(
                        messages, [self.suite.build_tool_response_message(raw_output)]
                    )
                else:
                    _nudge = (
                        'No Python code was detected in your response. '
                        'Please use Python to verify your reasoning step-by-step. '
                        + (r'State your final answer as \boxed{N} where N is an integer.'
                           if self.mode != 'BENCHMARK' else
                           r'State your final answer as \boxed{answer} in exact mathematical form.')
                    )
                    _append_messages(messages, [self.suite.build_tool_response_message(_nudge)])

        except queue.Empty:
            print(f'{_elapsed(self.blackboard)}   [{self.name} - {pb_id}] No sandbox available — skipping')
        except Exception as exc:
            print(f'{_elapsed(self.blackboard)}   [{self.name} - {pb_id}] Error: {exc}')
        finally:
            if sandbox is not None:
                try:
                    sandbox.reset()
                except Exception:
                    pass
                self.sandbox_pool.put(sandbox)

        duration  = time.time() - t_start
        conv_msgs = _serialize(messages, self.suite) if self.mode == 'BENCHMARK' else []

        # BENCHMARK: extract raw boxed content for LLM judge (non-integer support)
        _answer_raw = None
        if answer is not None and self.mode == 'BENCHMARK':
            _answer_raw = self.suite._extract_any_boxed(final_text)

        result = AttemptResult(
            agent_name=self.name, agent_suite=ec.suite_id,
            attempt_index=self.attempt_index, pb_id=pb_id,
            answer=answer, answer_raw=_answer_raw, answer_complete=answer is not None,
            final_text=final_text, response_length=total_tokens,
            python_calls=python_calls, python_errors=python_errors,
            attempt_duration_sec=duration, messages=conv_msgs,
            correct=None, eval_method=None, is_incremental=False,
        )
        self.blackboard.post_result(result, self.suite)
        return result

    def _post_incremental(
        self, pb_id, answer, total_tokens, python_calls, python_errors, t_start, messages
    ) -> None:
        """Post partial AttemptResult immediately on mid-stream answer detection."""
        conv_msgs = _serialize(messages, self.suite) if self.mode == 'BENCHMARK' else []
        partial = AttemptResult(
            agent_name=self.name, agent_suite=self.suite.ec.suite_id,
            attempt_index=self.attempt_index, pb_id=pb_id,
            answer=answer, answer_raw=None, answer_complete=True, final_text='',
            response_length=total_tokens, python_calls=python_calls,
            python_errors=python_errors,
            attempt_duration_sec=time.time() - t_start,
            messages=conv_msgs, correct=None, eval_method=None,
            is_incremental=True,
        )
        self.blackboard.post_result(partial, self.suite)

### MetaAgent — Independent Reconciliation Solver

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# META AGENT
# ══════════════════════════════════════════════════════════════════════════════

_META_SYSTEM_SUFFIX = (
    '\n\n'
    'You are acting as an independent verification solver. '
    'Other solver agents have proposed competing answers shown above. '
    'Solve the problem completely from scratch using your own reasoning and Python tools. '
    'Do NOT simply echo or vote on the proposed answers — arrive at your own conclusion.'
)


class MetaAgent:
    """Independent TIR solver spawned on answer divergence. Shorter loop, narrower temp."""

    def __init__(
        self, name, suite, sandbox_pool, blackboard, cfg,
        meta_answers, meta_lock, seed, temperature, mode, pb_id,
    ) -> None:
        self.name         = name
        self.suite        = suite
        self.sandbox_pool = sandbox_pool
        self.blackboard   = blackboard
        self.cfg          = cfg
        self.meta_answers = meta_answers
        self.meta_lock    = meta_lock
        self.seed         = seed
        self.temperature  = temperature
        self.mode         = mode
        self.pb_id        = pb_id

    def run(
        self,
        answer_snapshot: list,
        pb_text:         str,
        stop_event:      threading.Event,
        deadline:        float,
    ) -> Optional[int]:
        context_block  = self._build_context_block(answer_snapshot)
        augmented_text = f'{pb_text}\n\n{context_block}'
        system_prompt  = self.suite.ec.system_prompt + _META_SYSTEM_SUFFIX

        messages = self.suite.build_initial_messages(
            augmented_text, system_prompt, spirit=None
        )

        answer  = None
        sandbox = None
        max_turns = max(1, self.cfg.max_turns // 2)

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)

            for _turn in range(max_turns):
                if stop_event.is_set() or time.time() > deadline:
                    break

                prompt_tokens = self.suite.count_prompt_tokens(messages)
                max_new = (
                    self.cfg.meta_context_tokens - prompt_tokens - self.cfg.buffer_tokens
                )
                if max_new <= self.cfg.buffer_tokens:
                    break

                stream = self.suite.create_stream(
                    messages, seed=self.seed,
                    max_tokens=max_new, temperature=self.temperature,
                )
                full_text, code, new_msgs, mid_answer = self.suite.parse_stream(
                    stream, stop_event, deadline, self.mode
                )
                _append_messages(messages, new_msgs)

                if answer is None:
                    answer = mid_answer or self.suite.scan_for_answer(full_text)

                if answer is not None:
                    with self.meta_lock:
                        self.meta_answers.append(answer)
                    break

                if self.suite.is_final_turn(full_text, new_msgs):
                    if answer is None:
                        answer = self.suite.handle_final_non_parsable(full_text, messages)
                    if answer is not None:
                        with self.meta_lock:
                            self.meta_answers.append(answer)
                    break

                if code is not None:
                    try:
                        print(f" [{self.name} - {self.pb_id} - {self.suite.ec.suite_id}]  | Turn {_turn} | {_elapsed(self.blackboard)} | Tokens Per Second Est {round((prompt_tokens+len(full_text)/4)/(_elapsed_duration_nb(self.blackboard)+0.1),1) } | running code",len(str(code)),"chars")
                    except Exception as ee: print(ee)
                    raw_out  = sandbox.execute(code, timeout=self.cfg.jupyter_timeout)
                    _append_messages(messages, [self.suite.build_tool_response_message(raw_out)])
                else:
                    _append_messages(messages, [self.suite.build_tool_response_message(
                        'No Python code detected. Use Python to verify each step.'
                    )])

        except queue.Empty:
            print(f'{_elapsed(self.blackboard)}   [{self.name}] No sandbox — meta skipped')
        except Exception as exc:
            print(f'{_elapsed(self.blackboard)}   [{self.name}] MetaAgent error: {exc}')
        finally:
            if sandbox is not None:
                try:
                    sandbox.reset()
                except Exception:
                    pass
                self.sandbox_pool.put(sandbox)

        # ── Save reasoning trace (BENCHMARK mode) ─────────────────────────
        if self.mode == 'BENCHMARK':
            trace_msgs = _serialize(messages, self.suite) if messages else []
            self.blackboard.post_meta_trace({
                'pb_id':      self.pb_id,
                'agent_name': self.name,
                'answer':     answer,
                'messages':   trace_msgs,
            })

        return answer

    def _build_context_block(self, snapshot: list) -> str:
        if not snapshot:
            return ''
        by_answer: dict = defaultdict(list)
        for rec in snapshot:
            by_answer[rec.answer].append(rec)
        sorted_ans = sorted(by_answer.items(), key=lambda kv: len(kv[1]), reverse=True)
        lines = ['--- Competing Proposals from Other Solvers ---']
        for rank, (ans, recs) in enumerate(sorted_ans, 1):
            best    = max(recs, key=lambda r: r.response_length)
            excerpt = self._format_excerpt(best.last_3_turns)
            lines.append(
                f'\nProposal {rank}: answer = {ans}  '
                f'({len(recs)} agent{"s" if len(recs)>1 else ""} proposed this)'
            )
            if excerpt:
                lines.append(f'Reasoning excerpt (last 3 turns):\n{excerpt}')
        lines.append(
            '\n--- End of Competing Proposals ---\n'
            'Solve the problem above independently using your own reasoning.'
        )
        return '\n'.join(lines)

    @staticmethod
    def _format_excerpt(turns: list) -> str:
        parts = []
        for t in turns:
            content = t.get('content', '')
            if len(content) > 400:
                content = content[:400] + '…'
            parts.append(f'[{t.get("role","?")}]: {content}')
        return '\n'.join(parts)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTROLLER
# Dispatches agents, monitors Blackboard, fires stop conditions, spawns MetaAgents.
# Runs in the main thread from AIMO3Solver.solve_problem().
# ══════════════════════════════════════════════════════════════════════════════
DEBUG_CONTROLLER = False  # Set True only for local testing, never in submission

class Controller:
    """
    Orchestrates one problem solve across all concurrent agents.

    Responsibilities:
      - Assign agent identities (name, spirit, suite, temperature, seed)
      - Submit SolverAgents to a ThreadPoolExecutor
      - Evaluate stop conditions via aggregate_answers() after each completed future
      - Spawn MetaAgents on answer divergence when ENABLE_META_AGENT is True
      - Cancel pending futures when a stop condition fires
      - Implement the final answer fallback chain
    """

    def __init__(
        self,
        blackboard:    Blackboard,
        cfg,
        engines:       list,    # list[InferenceEngineManager]
        suites:        list,    # list[ModelSuiteProtocol], parallel to engines
        sandbox_pool:  queue.Queue,
        mode:          str,
        gamer_alphabet:    list,
        mathematical_spirits: list,
        enable_meta_agent:   bool,
        n_meta_agents_max:   int,
        meta_engine_idx:     int,
    ) -> None:
        self.blackboard           = blackboard
        self.cfg                  = cfg
        self.engines              = engines
        self.suites               = suites
        self.sandbox_pool         = sandbox_pool
        self.mode                 = mode
        self.gamer_alphabet       = gamer_alphabet
        self.mathematical_spirits = mathematical_spirits
        self.enable_meta_agent    = enable_meta_agent
        self.n_meta_agents_max    = n_meta_agents_max
        self.meta_engine_idx      = meta_engine_idx

    # ── Public ────────────────────────────────────────────────────────────────

    def run(
        self,
        pb_id:           str,
        pb_text:         str,
        deadline:        float,
        expected_answer: Optional[str] = None,
        checker:         Optional[AnswerChecker] = None,
        stop_event:      Optional[threading.Event] = None,
        real_solution:   str = '',               # RAG context for JOURNEY mode
    ) -> tuple:
        """
        Solve one problem. Returns (final_answer: int|str, stop_reason: str).
        stop_event: shared Event — set by Controller (stop_A/B/C) or Solver (timeout/error).
        """
        cfg          = self.cfg
        if stop_event is None:
            stop_event = threading.Event()
        meta_answers: list = []
        meta_lock         = threading.Lock()
        meta_spawned      = 0
        meta_futures:list = []
        stop_reason       = 'stop_E'  # default: natural exhaustion
        agg               = {}

        # Determine total agents across all active engines
        total_agents = sum(ec.n_agents for ec in [s.ec for s in self.suites])
        n_workers    = max(cfg.workers, total_agents + self.n_meta_agents_max) + 2

        # Pre-build agent configs (name / spirit / suite / temp / seed)
        rng          = random.Random(cfg.seed)
        agent_cfgs   = []
        spirit_pool  = list(self.mathematical_spirits) if self.mathematical_spirits else [None]
        name_pool    = list(self.gamer_alphabet)
        random.shuffle(spirit_pool)
        random.shuffle(name_pool)
        
        if DEBUG_CONTROLLER:
            # Timeout Debug — interruptible: checks stop_event, not a busy spin
            print(f'[DEBUG_CONTROLLER] Simulating stuck controller for {pb_id}')
            stop_event.wait(timeout=9999)  # Releases when stop_event is set or deadline fires
            return ERROR_SENTINEL, 'debug_timeout'  # Return immediately with default
        idx = 0
        for suite_i, suite in enumerate(self.suites):
            for _ in range(suite.ec.n_agents):
                name    = name_pool[idx % len(name_pool)]
                spirit  = spirit_pool[idx % len(spirit_pool)]
                temp    = round(
                    rng.uniform(cfg.temp_agent_min, cfg.temp_agent_max), 3
                )
                seed_i  = cfg.seed + idx
                agent_cfgs.append((name, spirit, suite, temp, seed_i, idx))
                idx += 1

        # Do NOT use `with` — shutdown(wait=True) would hang if any agent
        # is stuck (hung HTTP stream). Manage lifecycle manually instead.
        executor = ThreadPoolExecutor(max_workers=n_workers)
        try:
            # Submit all solver agents
            # Even-index = RAG-like (with retrieved context) | Odd-index = non-RAG baseline
            futures = {}
            for name, spirit, suite, temp, seed_i, attempt_i in agent_cfgs:
                _agent_pb_text = pb_text
                if (JOURNEY_TO_THE_WEST
                    and self.mode == 'BENCHMARK'
                    and real_solution):
                    _agent_pb_text = pb_text + JOURNEY_SOLUTION_WRAPPER.format(
                        solution=real_solution
                    )

                agent = SolverAgent(
                    name          = name,
                    spirit        = spirit,
                    suite         = suite,
                    sandbox_pool  = self.sandbox_pool,
                    blackboard    = self.blackboard,
                    cfg           = cfg,
                    attempt_index = attempt_i,
                    seed          = seed_i,
                    temperature   = temp,
                    mode          = self.mode,
                )
                fut = executor.submit(agent.run, pb_id, _agent_pb_text, stop_event, deadline)
                futures[fut] = agent.name

            all_futures = list(futures.keys())

            # ── Main polling loop ─────────────────────────────────────────
            # Exit conditions (checked on every iteration):
            #   A. remaining empty    → all agents completed naturally
            #   B. stop_event is set  → stop_A/B/C fired inside the loop
            #   C. time > deadline    → timeout
            # Every 1s the loop wakes up to re-check even if no agent finished.
            remaining        = set(all_futures)
            prev_total_count = 0  # total vote count at last MetaAgent spawn

            while remaining and not stop_event.is_set() and time.time() <= deadline:

                done, remaining = concurrent.futures.wait(
                    remaining,
                    timeout=1.0,
                    return_when=concurrent.futures.FIRST_COMPLETED,
                )

                for fut in done:
                    agent_name = futures.get(fut)

                    # ── MetaAgent future ───────────────────────────────────
                    # MetaAgent.run() returns Optional[int], NOT AttemptResult.
                    # Its answer is already pushed to meta_answers via meta_lock
                    # inside MetaAgent.run(). Consume the result here to avoid
                    # "unhandled future" warnings; aggregation on the next
                    # SolverAgent completion (or post-loop re-aggregate at L233)
                    # will pick up the updated meta_answers for stop_C.
                    if agent_name is None:
                        try:
                            fut.result()
                        except Exception:
                            pass
                        continue

                    try:
                        result = fut.result()
                    except Exception as exc:
                        print(f'{_elapsed(self.blackboard)}   [{agent_name}] Future error: {exc}')
                        continue

                    # Correctness labeling (BENCHMARK only)
                    if checker and expected_answer is not None and result is not None:
                        correct, eval_method = checker.check(result, expected_answer, pb_text)
                        result.correct       = correct
                        result.eval_method   = eval_method
                        marker = '✓' if correct else ('✗' if correct is False else '?')
                        print(f'{_elapsed(self.blackboard)}   [{agent_name} - {pb_id}] answer={result.answer} {marker} '
                              f'({eval_method}) [{result.response_length}tok]')
                    elif result is not None:
                        print(f'{_elapsed(self.blackboard)}   [{agent_name} - {pb_id}] answer={result.answer} '
                              f'[{result.response_length}tok]')

                    # Evaluate stop conditions after each completed future
                    snap = self.blackboard.get_answers_snapshot()
                    agg  = aggregate_answers(
                        snap,
                        k1_min_valid     = cfg.k1_min_valid,
                        effort_threshold = cfg.effort_stop_threshold,
                        k2_raw_vote      = cfg.k2_raw_vote_stop,
                        meta_answers     = meta_answers,
                        meta_vote_stop   = cfg.meta_vote_stop,
                    )

                    # Total votes across all answers — counts duplicates
                    # {1:1, 2:1}→2  |  {1:1, 2:2}→3  ← still triggers
                    n_total_now = sum(agg.get('raw_votes', {}).values())

                    if agg.get('stop_any'):
                        stop_reason = self._stop_reason_from_agg(agg, timed_out=False, exhausted=False)
                        remaining.clear()  # exit while on next iteration check
                        break

                    # ── MetaAgent spawn trigger ────────────────────────────
                    # Conditions (all must hold):
                    #   1. at least 2 distinct answers (divergence exists)
                    #   2. total vote count grew since last spawn
                    #      {1:1,2:1}→{1:1,2:2} still triggers (sum 2→3)
                    #   3. spawn budget not exhausted
                    if (self.enable_meta_agent
                            and len(agg.get('raw_votes', {})) >= 2
                            and n_total_now > prev_total_count
                            and meta_spawned < self.n_meta_agents_max):

                        prev_total_count = n_total_now   # update baseline
                        meta_spawned    += 1
                        meta_suite       = self.suites[self.meta_engine_idx]
                        meta_temp        = round(
                            rng.uniform(cfg.temp_meta_min, cfg.temp_meta_max), 3
                        )
                        print(f'{_elapsed(self.blackboard)} Spawn MetaAgent META-{meta_spawned} '
                              f'(total_votes={n_total_now}, distinct={len(agg.get("raw_votes", {}))})')
                        meta_agent = MetaAgent(
                            name         = f'META-{meta_spawned}',
                            suite        = meta_suite,
                            sandbox_pool = self.sandbox_pool,
                            blackboard   = self.blackboard,
                            cfg          = cfg,
                            meta_answers = meta_answers,
                            meta_lock    = meta_lock,
                            seed         = cfg.seed + 1000 + meta_spawned,
                            temperature  = meta_temp,
                            mode         = self.mode,
                            pb_id        = pb_id,
                        )
                        try:
                            mf = executor.submit(
                                meta_agent.run, snap, pb_text, stop_event, deadline
                            )
                            meta_futures.append(mf)
                            remaining.add(mf)  # poll meta futures in same loop
                        except RuntimeError:
                            pass  # executor already shut down

            # ── Loop exited — determine why and resolve final answer ───────
            # Three exit conditions:
            #   A. remaining is empty  → all agents completed naturally (stop_E)
            #   B. stop_event is set   → stop_A/B/C fired inside the loop
            #   C. time.time() > deadline → timeout, no agent triggered a stop
            if time.time() > deadline and not agg.get('stop_any'):
                stop_reason = 'stop_D'

            # Re-aggregate with full snapshot in case loop exited before
            # processing all completed futures (timeout / stop_event exit)
            snap = self.blackboard.get_answers_snapshot()
            agg  = aggregate_answers(
                snap,
                k1_min_valid     = cfg.k1_min_valid,
                effort_threshold = cfg.effort_stop_threshold,
                k2_raw_vote      = cfg.k2_raw_vote_stop,
                meta_answers     = meta_answers,
                meta_vote_stop   = cfg.meta_vote_stop,
            )

            # Signal stop to all remaining agents and cancel pending futures
            stop_event.set()
            for fut in all_futures:
                fut.cancel()
            for mf in meta_futures:
                try:
                    mf.result(timeout=5)
                except Exception:
                    pass

        finally:
            # shutdown(wait=False) — do not block on stuck agent threads.
            # Agents observe stop_event/deadline and exit on their own schedule.
            executor.shutdown(wait=False, cancel_futures=True)

        # If we exited the loop without a stop_any (natural exhaustion)
        if not agg.get('stop_any') and stop_reason == 'stop_E':
            snap = self.blackboard.get_answers_snapshot()
            agg  = aggregate_answers(
                snap,
                k1_min_valid     = cfg.k1_min_valid,
                effort_threshold = cfg.effort_stop_threshold,
                k2_raw_vote      = cfg.k2_raw_vote_stop,
                meta_answers     = meta_answers,
                meta_vote_stop   = cfg.meta_vote_stop,
            )

        final_answer, stop_reason = self._resolve_final_answer(agg, stop_reason, meta_answers)
        return final_answer, stop_reason

    # ── Internal ──────────────────────────────────────────────────────────────

    def _resolve_final_answer(
        self,
        agg:          dict,
        stop_reason:  str,
        meta_answers: list = None,
    ) -> tuple:
        """
        Fallback chain for final answer selection.

        Priority:
          1. Effort-weighted winner from aggregate_answers (SolverAgents)
          2. Combined raw vote: SolverAgent answers + MetaAgent answers
          3. Highest-effort valid answer from all_results (SolverAgents only)
          4. handle_final_non_parsable on the last result — BENCHMARK only
          5. ERROR_SENTINEL (-1) — no valid answer found; clamped to 0 at
             the submission boundary in predict()

        Note: LLM-based extraction (priority 4) is skipped in SUBMISSION mode.
        The combined vote (priority 2) is the primary safety net when no
        stop condition fired — it respects all available evidence including
        MetaAgent outputs that did not reach the meta_vote_stop threshold.
        """
        # Priority 1: effort-weighted winner
        winner = agg.get('winner')
        if winner is not None:
            return winner, stop_reason

        # Priority 2: combined raw vote across SolverAgent + MetaAgent answers
        all_res  = self.blackboard.get_all_results()
        valid    = [r for r in all_res if r.answer is not None]
        combined = [r.answer for r in valid] + (meta_answers or [])
        if combined:
            combined_winner = Counter(combined).most_common(1)[0][0]
            return combined_winner, 'fallback_combined_vote'

        # Priority 3: highest-effort valid answer from SolverAgents only
        if valid:
            best = max(valid, key=lambda r: r.response_length)
            return best.answer, 'fallback_aggregate'

        # Priority 4: LLM extraction — BENCHMARK only (too slow for SUBMISSION)
        if all_res and self.mode == 'BENCHMARK':
            last   = max(all_res, key=lambda r: r.attempt_duration_sec)
            suite  = self.suites[0]
            answer = suite.handle_final_non_parsable(last.final_text, [])
            if answer is not None:
                return answer, 'fallback_nonparsable'

        # Priority 5: sentinel — no valid answer from any source
        return ERROR_SENTINEL, 'fallback_zero'

    @staticmethod
    def _stop_reason_from_agg(agg: dict, timed_out: bool, exhausted: bool) -> str:
        if agg.get('stop_meta'):  return 'stop_C'
        if agg.get('stop_A'):     return 'stop_A'
        if agg.get('stop_B'):     return 'stop_B'
        if timed_out:             return 'stop_D'
        if exhausted:             return 'stop_E'
        return 'stop_E'


## Block H - Solver Orchestrator

`AIMO3Solver` is the single public facade for the entire system. Its public
interface consists of two methods only: `build_and_start()`, which is the
factory called once at notebook startup and wires all components together, and
`solve_problem()`, which is called once per problem at inference time. All
internal complexity - engine lifecycle, sandbox management, agent dispatch,
stop condition evaluation, JSON output - is hidden behind these two entry
points.

#### Adaptive time budget

Each problem receives a dynamically computed deadline rather than a fixed
timeout. The formula, implemented in `_compute_deadline()`, is:

```
reserved  = (problems_remaining - 1) x rolling_avg_solve_time
time_left = (run_start + notebook_limit) - now
budget    = clamp(time_left - reserved, base_problem_timeout, high_problem_timeout)
```

`rolling_avg_solve_time` is `RunTracker.last_n_avg(5)` - the mean solve
duration across the most recent five problems. Reserving this amount for each
remaining problem ensures that the current problem cannot consume time that
would prevent later problems from being attempted.

The clamping bounds (`base_problem_timeout = 276s`,
`high_problem_timeout = 895s`) prevent pathological behaviour in both
directions. A sequence of very fast early problems cannot inflate the budget
above 895s; a sequence of very slow problems cannot compress the budget below
276s. The 276s floor is not arbitrary: $276 \times 50 = 13{,}800$s, leaving
approximately 3,600s of slack against the 17,400s total wall clock.

The self-correcting property of the formula is worth noting. After a slow
problem, the rolling average rises, the reserved block expands, and the budget
for the next problem shrinks - a natural brake on depth when time is scarce.
After a fast problem, the rolling average falls, the reserved block contracts,
and the next problem receives more budget - releasing capacity for a hard
problem that follows an easy one.

#### BENCHMARK output

In BENCHMARK mode, `AIMO3Solver` creates three output subdirectories at init
time (`attempts/`, `meta/`, `problems/`) and writes individual JSON files
after each problem completes. Each file is written atomically, so interrupted
runs produce usable partial output without risk of corrupted append buffers.

Three pure conversion functions - `_attempt_record()`, `_meta_trace_record()`,
and `_problem_record()` - translate the internal data structures into flat
JSON-serialisable dicts. Integer keys in `Counter` and `defaultdict` objects
are cast to strings for JSON compatibility. These functions have no side
effects and are the only points where the internal data model crosses the JSON
boundary.

| Directory / Pattern | Unit | Key fields |
|---|---|---|
| `attempts/{pb_id}__{agent}__{idx}.json` | One file per `AttemptResult` (SolverAgents) | `agent_name`, `agent_suite`, `answer`, `response_length`, `python_calls`, `python_errors`, `correct`, `eval_method`, `messages` (full trace in BENCHMARK, empty list in SUBMISSION) |
| `meta/{pb_id}__{META-N}.json` | One file per MetaAgent run | `pb_id`, `agent_name`, `answer`, `messages` (full trace) |
| `problems/{pb_id}.json` | One file per solved problem | `pb_id`, `final_answer`, `stop_reason`, `answer_distribution`, `effort_distribution`, `weighted_pct_winner`, `solve_duration_sec`, `n_correct` |

The `messages` field in attempt JSON files is populated only in BENCHMARK mode.
In SUBMISSION mode it is an empty list. Callers can filter on this field to
distinguish modes without a separate flag.

These files are the primary data source for empirical analysis of the
architecture's behaviour: per-agent accuracy, per-suite performance, stop
condition firing distributions, effort-accuracy correlation, and MetaAgent
contribution to correctness. Every claim about the architecture that goes
beyond structural description should be validated against this output.


### Initialisation: build_and_start()

`build_and_start()` is the single entry point for system initialisation. It
accepts `engine_configs` (the list of active `EngineConfig` instances from
Block I) and a `CFG` instance, and returns `(engines, solver)`. All
initialisation that requires GPU or file-system access happens here, in a
fixed order that respects component dependencies.

```
1. Engine startup    - for each EngineConfig: preload weights, launch vLLM,
                       poll until ready. Engines start sequentially; the
                       first engine must be ready before the second starts,
                       ensuring GPU memory allocation does not overlap.

2. Suite construction - make_model_suite(engine, ec, cfg) for each engine.
                       Qwen35ModelSuite and its subclasses load the AutoTokenizer
                       here. This is the only point where tokenizer files are
                       read from disk.

3. Sandbox pool      - _init_sandbox_pool() spawns all kernel processes in
                       parallel using cfg.workers threads. Pool size:
                       total_agents + N_META_AGENTS_MAX (when ENABLE_META_AGENT).
                       All kernel startup latency is paid here, not on the
                       problem hot path.

4. atexit cleanup    - engine.stop() is registered for each engine via
                       atexit.register(). This ensures server processes are
                       terminated cleanly on notebook kernel exit, preventing
                       zombie vLLM processes from holding GPU memory across
                       sessions.

5. AIMO3Solver       - all components are passed to the constructor and wired
                       together. The solver instance is the only object the
                       caller needs to retain.
```

#### Sandbox pool sizing

The pool size formula is:

```
total_agents = sum(ec.n_agents for ec in engine_configs)
n_sandboxes  = total_agents + N_META_AGENTS_MAX   (if ENABLE_META_AGENT)
             = total_agents                        (otherwise)
```

This guarantees that peak concurrency - all SolverAgents plus all possible
concurrent MetaAgents, each holding one sandbox - never causes contention on
the pool. `queue.Queue.get(timeout=cfg.sandbox_timeout)` in each agent is a
safety net only; under normal operation the pool always has a free sandbox
available when an agent needs one.

Parallel initialisation with `ThreadPoolExecutor` amortises kernel startup
time. On a typical Kaggle session, starting 12 kernels sequentially takes
approximately 30-40 seconds; parallel startup reduces this to roughly
5-8 seconds, time that is subtracted from the competition wall clock.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RECORD BUILDERS
# Pure conversion functions. No side effects. JSON-safe types only.
# Used by AIMO3Solver._write_json() to convert typed objects to dicts.
# ══════════════════════════════════════════════════════════════════════════════

def _attempt_record(result: AttemptResult) -> dict:
    """
    Convert an AttemptResult to a JSON-serialisable dict.

    messages contains the full serialised conversation in BENCHMARK mode,
    or an empty list in SUBMISSION mode — callers can filter on this field.
    Only final (non-incremental) results should be written; the caller
    (AIMO3Solver._write_json) uses get_all_results() which returns
    deduplicated final posts only.
    """
    return {
        'agent_name':           result.agent_name,
        'agent_suite':          result.agent_suite,
        'attempt_index':        result.attempt_index,
        'pb_id':                result.pb_id,
        'answer':               result.answer,
        'answer_complete':      result.answer_complete,
        'response_length':      result.response_length,
        'python_calls':         result.python_calls,
        'python_errors':        result.python_errors,
        'attempt_duration_sec': result.attempt_duration_sec,
        'correct':              result.correct,
        'eval_method':          result.eval_method,
        'messages':             result.messages,   # [] in SUBMISSION
    }


def _meta_trace_record(trace: dict) -> dict:
    """
    Convert a MetaAgent reasoning trace to a JSON-serialisable dict.

    trace = {pb_id: str, agent_name: str, answer: int|None, messages: list[dict]}
    Only written in BENCHMARK mode.
    """
    return {
        'pb_id':      trace.get('pb_id', ''),
        'agent_name': trace.get('agent_name', ''),
        'answer':     trace.get('answer'),
        'messages':   trace.get('messages', []),
    }


def _problem_record(summary: ProblemSummary) -> dict:
    """
    Convert a ProblemSummary to a JSON-serialisable dict.

    Integer keys in Counter/dict are cast to str for JSON compatibility.
    """
    return {
        'pb_id':               summary.pb_id,
        'pb_text_len':         summary.pb_text_len,
        'n_agents_fired':      summary.n_agents_fired,
        'n_valid_answers':     summary.n_valid_answers,
        'n_correct':           summary.n_correct,
        'final_answer':        summary.final_answer,
        'expected_answer':     summary.expected_answer,
        'solve_duration_sec':  summary.solve_duration_sec,
        'stop_reason':         summary.stop_reason,
        'answer_distribution': {
            str(k): v for k, v in summary.answer_distribution.items()
        },
        'effort_distribution': {
            str(k): v for k, v in summary.effort_distribution.items()
        },
        'weighted_pct_winner': summary.weighted_pct_winner,
        'timestamp':           summary.timestamp,
    }

In [ ]:
class AIMO3Solver:
    """
    Public facade for the multi-agent TIR system.

    Callers interact with two methods only:
      - build_and_start()     — factory, called once at notebook startup
      - solve_problem()       — called once per problem (by predict() or predict_and_benchmark())

    Everything else — engine lifecycle, sandbox management, agent dispatch,
    stop condition evaluation — is internal.
    """

    def __init__(
        self,
        engines:      list,         # list[InferenceEngineManager]
        suites:       list,         # list[ModelSuiteProtocol]
        blackboard:   Blackboard,
        sandbox_pool: queue.Queue,
        cfg,
        mode:         str,
        checker:      Optional[AnswerChecker],
    ) -> None:
        self.engines      = engines
        self.suites       = suites
        self.blackboard   = blackboard
        self.sandbox_pool = sandbox_pool
        self.cfg          = cfg
        self.mode         = mode
        self.checker      = checker
        self._run_start   = time.time()

        # BENCHMARK output dirs — created once at solver init
        if mode == 'BENCHMARK':
            for sub in ('attempts', 'meta', 'problems'):
                os.makedirs(os.path.join(BENCHMARK_OUTPUT_DIR, sub), exist_ok=True)

    # ── Public ────────────────────────────────────────────────────────────────

    def solve_problem(
        self,
        pb_id:           str,
        pb_text:         str,
        expected_answer: Optional[str] = None,
        real_solution:   str = '',              # RAG context passthrough
    ):
        """
        Solve one problem. Returns the final answer (int or str in BENCHMARK mode).

        expected_answer is used in BENCHMARK mode only for correctness labeling.

        Safety layers (all internal — callers see only the int return value)
        --------------------------------------------------------------------
        Layer 1 — Global session guard:
            If less than GLOBAL_SESSION_SAFETY_MARGIN seconds remain in the
            session, skips agent dispatch and returns 0 immediately.
            Prevents starting work that cannot complete.

        Layer 2 — Per-problem hard timeout:
            controller.run() executes in a dedicated thread.
            hard_timeout = adaptive budget + PREDICT_HARD_TIMEOUT_GRACE.
            If the thread has not returned within this window, stop.set()
            is called, the thread is given 10s to unwind, and ERROR_SENTINEL is returned.

        Layer 3 — Exception containment:
            Any uncaught exception from controller.run() is caught, printed
            with pb_id and exception type, stop.set() kills agents,
            and ERROR_SENTINEL is returned. The run continues to the next problem.

        Layer 4 — Finalize guard:
            blackboard.finalize_problem() and _write_json() failures are
            caught separately and printed as non-fatal. The answer is already
            decided before this step.
        """
        # ── Layer 1: global session guard ─────────────────────────────────
        time_left_global = (self._run_start + self.cfg.notebook_limit) - time.time()
        if time_left_global < GLOBAL_SESSION_SAFETY_MARGIN:
            print(
                f'[{pb_id}] SKIP — only {time_left_global:.0f}s remaining in session '
                f'(margin={GLOBAL_SESSION_SAFETY_MARGIN}s)'
            )
            return ERROR_SENTINEL

        # ── Per-problem adaptive deadline + hard timeout ───────────────────
        deadline     = self._compute_deadline()
        hard_timeout = max(
            (deadline - time.time()) + PREDICT_HARD_TIMEOUT_GRACE,
            self.cfg.base_problem_timeout + PREDICT_HARD_TIMEOUT_GRACE,
        )

        # ── Build controller ───────────────────────────────────────────────
        self.blackboard.reset_for_problem(pb_id, pb_text)

        # One stop event per problem. Set = stop. Never cleared.
        # Next problem creates a new one; orphans keep the old (set) reference.
        stop = threading.Event()

        controller = Controller(
            blackboard           = self.blackboard,
            cfg                  = self.cfg,
            engines              = self.engines,
            suites               = self.suites,
            sandbox_pool         = self.sandbox_pool,
            mode                 = self.mode,
            gamer_alphabet       = GAMER_ALPHABET,
            mathematical_spirits = MATHEMATICAL_SPIRITS,
            enable_meta_agent    = ENABLE_META_AGENT,
            n_meta_agents_max    = N_META_AGENTS_MAX,
            meta_engine_idx      = META_AGENT_ENGINE_IDX,
        )

        # ── Layers 2 + 3: run controller in thread with hard timeout ───────
        final_answer = ERROR_SENTINEL
        stop_reason  = 'stop_E'

        _exec = concurrent.futures.ThreadPoolExecutor(max_workers=1)
        _fut  = _exec.submit(
            controller.run,
            pb_id           = pb_id,
            pb_text         = pb_text,
            deadline        = deadline,
            expected_answer = expected_answer,
            checker         = self.checker,
            stop_event      = stop,
            real_solution   = real_solution,
        )

        try:
            final_answer, stop_reason = _fut.result(timeout=hard_timeout)

        except concurrent.futures.TimeoutError:
            print(
                f'{_elapsed(self.blackboard)} [{pb_id}] HARD TIMEOUT after {hard_timeout:.0f}s '
                f'— aborting agents'
            )
            stop.set()
            try:
                _fut.result(timeout=10)
            except Exception:
                pass
            final_answer = ERROR_SENTINEL
            stop_reason  = 'stop_hard_timeout'

        except Exception as exc:
            print(
                f'{_elapsed(self.blackboard)} [{pb_id}] ERROR — {type(exc).__name__}: {exc} '
                f'— aborting agents'
            )
            stop.set()
            try:
                _fut.result(timeout=10)
            except Exception:
                pass
            final_answer = ERROR_SENTINEL
            stop_reason  = 'stop_error'

        finally:
            stop.set()   # always — no-op if already set, kills stragglers
            _exec.shutdown(wait=False, cancel_futures=True)

        # ── Layer 4: finalize + write — errors are non-fatal ──────────────
        try:
            summary = self.blackboard.finalize_problem(
                final_answer, stop_reason, expected_answer
            )
            self._print_problem_result(summary)
            if self.mode == 'BENCHMARK':
                self._write_json(summary)
        except Exception as exc:
            print(f'{_elapsed(self.blackboard)} [{pb_id}] finalize error (non-fatal): {type(exc).__name__}: {exc}')

        return final_answer

    def close(self) -> None:
        """No-op: per-file JSON writes close automatically. Kept for API compatibility."""
        pass

    # ── Properties ────────────────────────────────────────────────────────────

    @property
    def problems_solved(self) -> int:
        return self.blackboard.run_tracker.run_stats.total_problems_seen

    @property
    def problems_remaining(self) -> int:
        return max(0, self.cfg.total_problems - self.problems_solved)

    # ── Internal ──────────────────────────────────────────────────────────────

    def _compute_deadline(self) -> float:
        """
        Adaptive per-problem deadline.

        Uses a rolling average of recent solve times to reserve time for remaining
        problems, preventing slow early problems from starving later ones.
        """
        avg_t     = self.blackboard.run_tracker.last_n_avg(5)
        remaining = self.problems_remaining
        reserved  = max(0, (remaining - 1) * avg_t) if avg_t > 0 else 0.0

        time_left = (self._run_start + self.cfg.notebook_limit) - time.time()
        budget    = max(
            self.cfg.base_problem_timeout,
            min(self.cfg.high_problem_timeout, time_left - reserved),
        )
        return time.time() + budget

    def _print_problem_result(self, summary: ProblemSummary) -> None:
        rs      = self.blackboard.run_tracker.run_stats
        n_seen  = rs.total_problems_seen
        elapsed = summary.solve_duration_sec

        top_ans  = summary.final_answer
        top_pct  = f'{summary.weighted_pct_winner*100:.0f}%'
        raw_dist = dict(sorted(summary.answer_distribution.items(),
                               key=lambda kv: kv[1], reverse=True)[:4])

        print(f'\n{_elapsed(self.blackboard)} [{summary.pb_id}] → {top_ans}  '
              f'(effort {top_pct}, stop={summary.stop_reason}, {elapsed:.1f}s)')
        print(f'{_elapsed(self.blackboard)}   votes={dict(raw_dist)}  valid={summary.n_valid_answers}')

        if self.mode == 'BENCHMARK' and summary.n_correct is not None:
            acc = rs.total_problems_correct / n_seen * 100
            marker = '✓' if summary.n_correct > 0 else '✗'
            print(f'{_elapsed(self.blackboard)}   {marker}  running accuracy: {rs.total_problems_correct}/{n_seen}'
                  f' = {acc:.1f}%')

    def _write_json(self, summary: ProblemSummary) -> None:
        """Write one JSON file per attempt, per meta trace, and per problem."""
        pb_id = summary.pb_id

        # ── attempts/pb_id__AGENT__idx.json — one file per agent×problem ─────
        attempts_dir = os.path.join(BENCHMARK_OUTPUT_DIR, 'attempts')
        for result in self.blackboard.get_all_results():
            fname = f'{pb_id}__{result.agent_name}__{result.attempt_index}.json'
            with open(os.path.join(attempts_dir, fname), 'w') as fh:
                json.dump(_attempt_record(result), fh, indent=2)

        # ── meta/pb_id__META-N.json — one file per MetaAgent×problem ─────────
        meta_dir = os.path.join(BENCHMARK_OUTPUT_DIR, 'meta')
        for trace in self.blackboard.get_meta_traces():
            fname = f'{pb_id}__{trace["agent_name"]}.json'
            with open(os.path.join(meta_dir, fname), 'w') as fh:
                json.dump(_meta_trace_record(trace), fh, indent=2)

        # ── problems/pb_id.json — one file per problem ────────────────────────
        problems_dir = os.path.join(BENCHMARK_OUTPUT_DIR, 'problems')
        with open(os.path.join(problems_dir, f'{pb_id}.json'), 'w') as fh:
            json.dump(_problem_record(summary), fh, indent=2)

### Initialisation: `build_and_start()`

In [ ]:
import atexit

# ══════════════════════════════════════════════════════════════════════════════
# SANDBOX POOL INITIALISATION
# ══════════════════════════════════════════════════════════════════════════════

def _init_sandbox_pool(engine_configs: list, cfg) -> queue.Queue:
    """
    Pre-initialise all Jupyter sandbox kernels in parallel.

    Pool size:
        total_agents + N_META_AGENTS_MAX + SANDBOX_POOL_MARGIN  (if ENABLE_META_AGENT)
        total_agents + SANDBOX_POOL_MARGIN                      (otherwise)

    The margin absorbs timing jitter during sandbox release (kernel reset
    takes 1–2s, during which the sandbox is still held by the releasing agent).

    All kernel startup latency is paid here, at solver init time, so that
    SolverAgent and MetaAgent never wait for kernel startup in the hot path.
    """
    total_agents = sum(ec.n_agents for ec in engine_configs)
    n_meta       = N_META_AGENTS_MAX if ENABLE_META_AGENT else 0
    n_sandboxes  = total_agents + n_meta + SANDBOX_POOL_MARGIN

    pool = queue.Queue()
    print(
        f'Initialising {n_sandboxes} Jupyter sandboxes '
        f'({total_agents} agent + {n_meta} meta + {SANDBOX_POOL_MARGIN} margin) …'
    )
    t0 = time.time()

    def _make_sandbox(_):
        return JupyterSandbox(timeout=cfg.jupyter_timeout)

    with ThreadPoolExecutor(max_workers=cfg.workers) as ex:
        futs = [ex.submit(_make_sandbox, i) for i in range(n_sandboxes)]
        for fut in as_completed(futs):
            try:
                pool.put(fut.result())
            except Exception as exc:
                print(f'  [WARN] Sandbox init failed: {exc}')

    print(f'  Sandboxes ready in {time.time()-t0:.1f}s  (pool size: {pool.qsize()})\n')
    return pool


# ══════════════════════════════════════════════════════════════════════════════
# BUILD AND START — System factory
# ══════════════════════════════════════════════════════════════════════════════

def build_and_start(engine_configs: list, cfg) -> tuple:
    """
    Initialise the full multi-agent system and return (engines, solver).

    Initialisation order:
      1. Start all InferenceEngineManagers  (preload → launch vLLM → wait ready)
      2. Build ModelSuiteProtocol instances  (loads tokenizer for Qwen35)
      3. Initialise JupyterSandbox pool      (parallel kernel startup)
      4. Register atexit cleanup             (engine.stop() on kernel exit)
      5. Instantiate AIMO3Solver

    Args:
        engine_configs: list[EngineConfig] — e.g. ACTIVE_ENGINE_CONFIGS
        cfg:            CFG instance

    Returns:
        (engines: list[InferenceEngineManager], solver: AIMO3Solver)
    """
    # ── 1. Engine startup ──────────────────────────────────────────────────
    engines: list = []
    for ec in engine_configs:
        mgr = InferenceEngineManager(ec, cfg)
        mgr.start()
        engines.append(mgr)

    # ── 2. Model suites ────────────────────────────────────────────────────
    suites = [make_model_suite(mgr, mgr.ec, cfg) for mgr in engines]

    # ── 3. Sandbox pool ────────────────────────────────────────────────────
    sandbox_pool = _init_sandbox_pool(engine_configs, cfg)

    # ── 4. AnswerChecker (BENCHMARK only) ──────────────────────────────────
    checker = AnswerChecker(engines) if MODE == 'BENCHMARK' else None

    # ── 5. Solver ──────────────────────────────────────────────────────────
    blackboard = Blackboard()
    solver     = AIMO3Solver(
        engines      = engines,
        suites       = suites,
        blackboard   = blackboard,
        sandbox_pool = sandbox_pool,
        cfg          = cfg,
        mode         = MODE,
        checker      = checker,
    )

    # ── 6. Cleanup registration ────────────────────────────────────────────
    def _cleanup():
        solver.close()
        for mgr in engines:
            try:
                mgr.stop()
            except Exception:
                pass

    atexit.register(_cleanup)
    print('\n' + '═'*55)
    print(f'  AIMO3Solver ready  |  mode={MODE}  |  engines={len(engines)}')
    print(f'  suites: {[s.ec.suite_id for s in suites]}')
    total_agents = sum(ec.n_agents for ec in engine_configs)
    print(f'  agents={total_agents}  sandboxes={sandbox_pool.qsize()}')
    print('═'*55 + '\n')

    return engines, solver


# ══════════════════════════════════════════════════════════════════════════════
# NUKE GPU MEMORY — Utility for inter-experiment VRAM reclamation
# ══════════════════════════════════════════════════════════════════════════════

VLLM_PORTS = [8000,8001,8002]

def nuke_gpu_memory() -> None:
    import subprocess, signal, os, time, gc

    own_pid = os.getpid()

    # ── 1. Find EVERY pid using the GPU via nvidia-smi ───────────────────────
    # This is the only reliable source — catches detached workers, zombie
    # parents that still hold CUDA contexts, triton daemons, everything.
    def get_gpu_pids() -> list:
        try:
            r = subprocess.run(
                ['nvidia-smi', '--query-compute-apps=pid',
                 '--format=csv,noheader,nounits'],
                capture_output=True, text=True
            )
            return [
                int(p.strip()) for p in r.stdout.strip().splitlines()
                if p.strip().isdigit() and int(p.strip()) != own_pid
            ]
        except Exception as e:
            print(f'[nuke] nvidia-smi failed: {e}')
            return []

    gpu_pids = get_gpu_pids()
    print(f'[nuke] GPU pids before kill: {gpu_pids}')

    # ── 2. SIGTERM first — let CUDA teardown gracefully ──────────────────────
    for pid in gpu_pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except (ProcessLookupError, PermissionError):
            pass

    time.sleep(2)   # give CUDA a moment to release contexts cleanly

    # ── 3. SIGKILL anything still alive ──────────────────────────────────────
    for pid in gpu_pids:
        try:
            os.kill(pid, signal.SIGKILL)
            print(f'[nuke] SIGKILL → {pid}')
        except (ProcessLookupError, PermissionError):
            pass   # already dead from SIGTERM — good

    # ── 4. Kill by port (catches server process if missed by nvidia-smi) ─────
    for port in VLLM_PORTS:
        try:
            subprocess.run(
                ['fuser', '-k', '-9', f'{port}/tcp'],
                capture_output=True
            )
        except FileNotFoundError:
            try:
                r = subprocess.run(
                    ['lsof', '-ti', f'tcp:{port}'],
                    capture_output=True, text=True
                )
                for pid in [int(p) for p in r.stdout.split() if p.strip().isdigit()]:
                    if pid != own_pid:
                        os.kill(pid, signal.SIGKILL)
            except Exception:
                pass

    # ── 5. Stop engine managers (close file handles + subprocess refs) ────────
    for var in ['ENGINES', 'SOLVER']:
        obj = globals().get(var)
        if obj is None:
            continue
        for t in (obj if isinstance(obj, list) else [obj]):
            try:
                t.stop()
            except Exception:
                pass

    # ── 6. Reap zombies ───────────────────────────────────────────────────────
    try:
        while True:
            pid, _ = os.waitpid(-1, os.WNOHANG)
            if pid == 0:
                break
    except ChildProcessError:
        pass

    # ── 7. Shared memory cleanup ──────────────────────────────────────────────
    try:
        subprocess.run(
            'ipcs -m | awk \'NR>3 {print $2}\' | xargs -r ipcrm -m',
            shell=True, capture_output=True
        )
    except Exception:
        pass

    # ── 8. Wait, then verify GPU is actually free ─────────────────────────────
    time.sleep(3)

    survivors = get_gpu_pids()
    if survivors:
        print(f'[nuke] ⚠️  GPU still held by pids: {survivors}')
        print('[nuke] Attempting nvidia-smi reset...')
        # Last resort: reset the GPU context entirely.
        # WARNING: this will kill ALL processes using the GPU on this machine,
        # including the notebook kernel's own CUDA context if it has one.
        # Only safe if the notebook kernel itself has no active CUDA state.
        r = subprocess.run(
            ['nvidia-smi', '--gpu-reset'],
            capture_output=True, text=True
        )
        print(f'[nuke] gpu-reset: {r.stdout.strip() or r.stderr.strip()}')
    else:
        print('[nuke] ✅ GPU fully released')

    # ── 9. Python GC + CUDA flush ─────────────────────────────────────────────
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            before = torch.cuda.memory_allocated() / 1e9
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            after = torch.cuda.memory_allocated() / 1e9
            total = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f'[nuke] GPU: {before:.2f} → {after:.2f} GB '
                  f'| free {total - after:.2f} / {total:.2f} GB')
    except ImportError:
        pass

    # ── 10. Final port check ──────────────────────────────────────────────────
    for port in VLLM_PORTS:
        r = subprocess.run(
            ['lsof', '-ti', f'tcp:{port}'], capture_output=True, text=True
        )
        status = f'HELD by {r.stdout.strip()}' if r.stdout.strip() else 'free ✅'
        print(f'[nuke] port {port}: {status}')


## Block I - Run Configuration

This is the primary edit target between experiments. All class and function
definitions from Blocks C through H are complete at this point, so
`EngineConfig` subclasses can be safely instantiated here without
forward-reference errors.

The configuration is organised into six sections in the code cell below.

#### Separation of configuration

| Concern | Location |
|---|---|
| Model paths, ports, GPU memory utilization | `EngineConfig` instances in this block |
| Agent count per engine | `n_agents` field on each `EngineConfig` |
| Speculative decoding | `spec_config=` field on each `EngineConfig` |
| Which engines are active | `ACTIVE_ENGINE_CONFIGS` list at the end of this block |
| Timeouts, temperatures, vote thresholds | `CFG` dataclass in Block J |
| Prompt text | Block K strings, injected into `EngineConfig` fields at the end of Block K |

This separation means that switching models requires changes only in this
block. Tuning solver behaviour (stop conditions, temperatures, context budget)
requires changes only in Block J. Neither affects the other.

#### Mode toggle

```python
MODE = 'SUBMISSION'   # default
MODE = 'BENCHMARK'    # enable for empirical validation
```

One reassignment switches the entire system. In BENCHMARK mode: correctness
labeling is active, per-problem console output is verbose, and individual JSON
files are written to `attempts/`, `meta/`, and `problems/` subdirectories of
`BENCHMARK_OUTPUT_DIR` after each problem. In SUBMISSION mode: none of the
above occurs, conversation traces are not serialised, and `AnswerChecker` is
never instantiated.

#### Engine configuration guide

Each `EngineConfig` instance describes one vLLM server process. The fields
that require calibration per deployment are:

**`model_path`**: absolute path to the model directory on the Kaggle dataset
mount. Must exist at session start; the preload step in
`InferenceEngineManager._preload_weights()` will warn and skip if the path
is missing, but the server launch will then fail.

**`port`**: localhost port for the vLLM HTTP server. In single-suite
configurations, any free port works. In multi-suite configurations, each
engine must use a distinct port. Consecutive ports (8000, 8001, ...) are
the convention used throughout this notebook.

**`n_agents`**: number of concurrent `SolverAgent` instances for this engine.
Higher values increase answer diversity and improve Stop-A/B convergence on
easy problems, but reduce per-agent token throughput because vLLM must batch
more concurrent requests. The sweet spot depends on model size and GPU memory.
For `gpt-oss-120b` solo on H100: `n_agents=8`.

**`gpu_memory_utilization`**: fraction of GPU memory reserved for vLLM KV
cache and model weights. In single-suite mode, `0.96` is appropriate for
most models on H100. In multi-suite mode, the sum across all active engines
must leave headroom for CUDA kernels and system overhead - a safe starting
point for two engines is `0.50 / 0.45` split.

**`batch_size`** (`--max-num-seqs`): maximum sequences per vLLM scheduling
step. Set equal to `n_agents` for standard operation. Reduce to 4-16 when
speculative decoding is active to avoid GPU OOM under concurrent draft
verification.

**`spec_config`**: set to a `SpecConfig` instance to enable speculative
decoding, or `None` to disable. Available presets defined in this block:

| Variable | Method | Target model | Status |
|---|---|---|---|
| `spec_decode_amazon` | Eagle3 (amazon variant) | `gpt-oss-120b` | Experimental |
| `spec_decode_wenliang` | Eagle3 (wenliang1990 variant) | `gpt-oss-120b` | Experimental |
| `spec_decode_nvidia_tp` | Eagle3 (NVIDIA throughput) | `gpt-oss-120b` | Experimental |
| `spec_decode_nvidia_lc` | Eagle3 (NVIDIA long-context) | `gpt-oss-120b` | Experimental |
| `spec_decode_zhuyksir` | Eagle3 (zhuyksir variant) | `gpt-oss-120b` | Experimental |
| `spec_decode_qwen35` | MTP built-in head | `Qwen3.5-27B/35B` | Stable |

Eagle3 variants require the corresponding draft model to be mounted as a
Kaggle dataset input. All five `gpt-oss-120b` Eagle3 variants are bundled
in the `eagle3-go` dataset at `EAGLE3_BASE`. See the Eagle3 community thread
at [github.com/juemifuji/eagle3-aimo3](https://github.com/juemifuji/eagle3-aimo3)
for per-variant throughput benchmarks.

#### Multi-suite configuration

The baseline uses `gpt-oss-120b` alone:
```python
ACTIVE_ENGINE_CONFIGS = [_ec_gpt_120b]
```

For a two-suite experiment, add a second engine:
```python
ACTIVE_ENGINE_CONFIGS = [_ec_gpt_20b, _ec_qwen_35_27b]
```

`build_and_start()` reads `ACTIVE_ENGINE_CONFIGS` and computes all derived
sizes (sandbox pool, executor worker count) automatically. No other cells
require modification. The only manual adjustment needed is
`gpu_memory_utilization` on each engine - the two values must together leave
adequate headroom on the GPU.

#### Agent identity pools

`GAMER_ALPHABET` assigns process labels to agent instances for logging. These
names appear in console output and JSON records to identify which agent
produced which result. They are never shown to the model and have no effect on
reasoning.

`MATHEMATICAL_SPIRITS` is the pool of persona prefix sentences (one per
historical mathematician) from which each agent draws its spirit at the start
of each problem. See Block G for the role of spirits in agent diversity.

#### MetaAgent configuration

Three constants control MetaAgent behaviour:

| Constant | Default | Effect |
|---|---|---|
| `ENABLE_META_AGENT` | `True` | Enable or disable MetaAgent spawning entirely. |
| `N_META_AGENTS_MAX` | `3` | Maximum MetaAgents spawned per problem. Also determines the number of extra sandbox slots pre-allocated by `_init_sandbox_pool()`. |
| `META_AGENT_ENGINE_IDX` | `0` | Index into `ACTIVE_ENGINE_CONFIGS` selecting which engine MetaAgents use. In a multi-suite run, setting this to `1` directs MetaAgents to the secondary model family. |

`JUDGE_ENGINE_IDX` (default: `0`) selects the engine used by `AnswerChecker`
for LLM judge calls in BENCHMARK mode. Independent of `META_AGENT_ENGINE_IDX`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK I — RUN CONFIGURATION
# Primary edit target between experiments.
# ══════════════════════════════════════════════════════════════════════════════

# ── RAG-like vs non-RAG agent comparison (BENCHMARK only) ────────────────
JOURNEY_TO_THE_WEST       = True   # True: enable RAG-like context injection for half the agents
JOURNEY_INJECTION_RATIO   = 1.0     # Fraction of agents operating in RAG-like mode
JOURNEY_DATASET_PATH      = '/kaggle/input/datasets/khoinguyennguyen/journey-to-the-west/hard_math_benchmark_dataset.parquet'
JOURNEY_DATASET_PATH      = '/kaggle/input/datasets/khoinguyennguyen/journey-to-the-west-experimental/stress_test_reference.parquet'



# ── Execution mode ────────────────────────────────────────────────────────────
MODE = 'SUBMISSION'   # 'SUBMISSION' | 'BENCHMARK'
MODE = 'BENCHMARK'   # 'SUBMISSION' | 'BENCHMARK'

# ── Model paths ───────────────────────────────────────────────────────────────
GPT_OSS_120B_PATH  = '/kaggle/input/gpt-oss-120b/transformers/default/1'
GPT_OSS_120B_PATH  = '/kaggle/input/models/huikang/gpt-oss-120b-aimo3/transformers/160a/20'
#GPT_OSS_120B_PATH  = '/kaggle/input/models/huikang/gpt-oss-120b-aimo3/transformers/160a/14'
#GPT_OSS_120B_PATH  = '/kaggle/input/models/huikang/gpt-oss-120b-aimo3/transformers/160a/8'
GPT_OSS_20B_PATH  = '/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1'
QWEN35_27B_PATH   = '/kaggle/input/models/shelterw/qwen3.5/transformers/qwen3.5-27b-fp8/1'
QWEN35_35A3B_PATH   = '/kaggle/input/models/shelterw/qwen3.5/transformers/qwen3.5-35b-a3b-fp8/1'


GPT_OSS_20B_SFT_1_PATH  = '/kaggle/input/models/shelterw/gpt-sft/transformers/gpt-oss-20b-tir-7k-5e/1'
GPT_OSS_20B_SFT_1_PATH  = '/kaggle/input/models/nguyennguyen599/gpt-oss-20b-ft/transformers/gpt-oss-20b-lora-400/2'


# ── New model paths TO ADD WITH CORRECT DATASET ───────────────────────────────────────────────────────────
# DeepSeek-R1-Distill (Qwen backbone)
DS_R1_QWEN_1B5_PATH  = '/kaggle/input/deepseek-r1-distill-qwen-1.5b/transformers/default/1'
DS_R1_QWEN_7B_PATH   = '/kaggle/input/models/deepseek-ai/deepseek-r1/transformers/deepseek-r1-distill-qwen-7b/2'
DS_R1_QWEN_14B_PATH  = '/kaggle/input/deepseek-r1-distill-qwen-14b/transformers/default/1'
DS_R1_QWEN_32B_PATH  = '/kaggle/input/deepseek-r1-distill-qwen-32b/transformers/default/1'
DS_R1_QWEN3_8B_PATH  = '/kaggle/input/models/deepseek-ai/deepseek-r1-0528/transformers/deepseek-r1-0528-qwen3-8b/1'
# DeepSeek-R1-Distill (Llama backbone)
DS_R1_LLAMA_8B_PATH  = '/kaggle/input/deepseek-r1-distill-llama-8b/transformers/default/1'
DS_R1_LLAMA_70B_PATH = '/kaggle/input/deepseek-r1-distill-llama-70b/transformers/default/1'
# OpenReasoning-Nemotron (Qwen2.5 backbone — AIMO-2 winners)
OR_1B5_PATH  = '/kaggle/input/openreasoning-nemotron-1.5b/transformers/default/1'
OR_7B_PATH   = '/kaggle/input/openreasoning-nemotron-7b/transformers/default/1'
OR_14B_PATH  = '/kaggle/input/openreasoning-nemotron-14b/transformers/default/1'
OR_32B_PATH  = '/kaggle/input/openreasoning-nemotron-32b/transformers/default/1'
# Gemma3 Instruct
GEMMA3_4B_PATH  = '/kaggle/input/gemma-3-4b-it/transformers/default/1'
GEMMA3_12B_PATH = '/kaggle/input/gemma-3-12b-it/transformers/default/1'
GEMMA3_27B_PATH = '/kaggle/input/gemma-3-27b-it/transformers/default/1'


# ── Eagle3 speculative decoding draft model paths ─────────────────────────────
EAGLE3_BASE  = '/kaggle/input/datasets/khoinguyennguyen/eagle3-go'
#/kaggle/input/datasets/khoinguyennguyen/eagle3-go/nvidia/gpt-oss-120b-Eagle3-long-context
#/kaggle/input/datasets/khoinguyennguyen/eagle3-go/nvidia/gpt-oss-120b-Eagle3-throughput
#/kaggle/input/datasets/khoinguyennguyen/eagle3-go/amazon/gpt-oss-120b-p-eagle
#/kaggle/input/datasets/khoinguyennguyen/eagle3-go/zhuyksir/EAGLE3-gpt-oss-120b-bf16
#/kaggle/input/datasets/khoinguyennguyen/eagle3-go/lmsys/EAGLE3-gpt-oss-120b-bf16
#/kaggle/input/datasets/khoinguyennguyen/eagle3-go/wenliang1990/gpt-oss-120b-eagle3-aimo3

GPT_OSS_120B_EAGLE3_PATHS = {
    'amazon':       f'{EAGLE3_BASE}/amazon/gpt-oss-120b-p-eagle',
    'zhuyksir':   f'{EAGLE3_BASE}/zhuyksir/EAGLE3-gpt-oss-120b-bf16',
    'lmsys': f'{EAGLE3_BASE}/lmsys/EAGLE3-gpt-oss-120b-bf16',
    'wenliang1990': f'{EAGLE3_BASE}/wenliang1990/gpt-oss-120b-eagle3-aimo3',
    'nvidia_tp':     f'{EAGLE3_BASE}/nvidia/gpt-oss-120b-Eagle3-throughput',
    'nvidia_lc':   f'{EAGLE3_BASE}/nvidia/gpt-oss-120b-Eagle3-long-context',
}




# EXPERIMENTAL: IMPROVE GPT-OSS-120B THROUGHPUT WITH EAGLE3 - SPECULATIVE DECODING - UNSTABLE - NEED TO FIND SWEETSPOT
spec_decode_wenliang = eagle3_spec(GPT_OSS_120B_EAGLE3_PATHS['wenliang1990'], n = 3) #3-7
spec_decode_amazon = eagle3_spec(GPT_OSS_120B_EAGLE3_PATHS['amazon'], n = 3)
spec_decode_nvidia_tp = eagle3_spec(GPT_OSS_120B_EAGLE3_PATHS['nvidia_tp'], n = 3)
spec_decode_nvidia_lc = eagle3_spec(GPT_OSS_120B_EAGLE3_PATHS['nvidia_lc'], n = 3)
spec_decode_zhuyksir = eagle3_spec(GPT_OSS_120B_EAGLE3_PATHS['zhuyksir'], n = 3)

# EXPERIMENTAL: IMPROVE QWEN3.5 THROUGHPUT WITH QWEN MULTI-TOKEN PREDICTION - SPECULATIVE DECODING : STABLE ~ x 150-200% SPEED

spec_decode_qwen35 = mtp_spec(n=5)

# ── Eagle3 CUDA graph sizing helpers ──────────────────────────────────────────
# Eagle3 tree speculation creates larger effective batch sizes than MTP.
# For batch_size=B and num_speculative_tokens=N:
#   linear verification batch = B × (1 + N)
#   tree topology can exceed this → add headroom above linear max.
#
# Formula: capture up to 2× linear max, clamped to 64.
# Override cudagraph_capture_sizes per-instance if you hit OOM or eager fallback.

def _eagle3_cudagraph_sizes(batch_size: int, num_spec: int) -> list:
    """Compute safe CUDA graph capture sizes for Eagle3 tree speculation."""
    linear_max = batch_size * (1 + num_spec)
    # Add headroom for tree expansion (2× linear, clamped)
    safe_max = min(linear_max * 2, 64)
    sizes = set()
    s = 1
    while s <= safe_max:
        sizes.add(s)
        s *= 2
    # Fill gaps in steps of batch_size
    v = batch_size
    while v <= safe_max:
        sizes.add(v)
        v += batch_size
    sizes.add(safe_max)
    return sorted(sizes)


# ── Agent identity pools ──────────────────────────────────────────────────────
# GAMER_ALPHABET: process labels for logging — never shown to the model.

GAMER_ALPHABET_1 = [
    'ALPHA',    'BRAVO',    'CHARLIE',  'DELTA',    'ECHO',     'FOXTROT',  'GOLF',     'HOTEL',      'INDIA',    'JULIET',   'KILO',     'LIMA',
    'MIKE',     'NOVEMBER', 'OSCAR',    'PAPA',    'QUEBEC',   'ROMEO',    'SIERRA',   'TANGO',    'UNIFORM',  'VICTOR',   'WHISKEY',  'XRAY',
    'YANKEE',   'ZULU',
]

GAMER_ALPHABET_2 = [
    'MIGHTY_MOUSE', 'NOBLE_BUFFALO','FIERCE_TIGER', 'RICH_CAT',     'GOLDEN_DRAGON','WISE_SERPENT', 'WILD_HORSE',   'GENTLE_GOAT',  'CLEVER_MONKEY','PROUD_ROOSTER','LOYAL_HOUND',  'LUCKY_BOAR',
    'ARIES',        'TAURUS',       'GEMINI',       'CANCER',       'LEO',          'VIRGO',        'LIBRA',        'SCORPIO',      'SAGITTARIUS',  'CAPRICORN',    'AQUARIUS',     'PISCES',
    'AZURE_DRAGON', 'WHITE_TIGER',  'VERMILION_BIRD','BLACK_TORTOISE','QILIN',       'FENGHUANG',    'PIXIU',        'BAXIA',        'LONGMA',       'TAOTIE',       'HUNDUN',       'QIONGQI',
    'GARUDA',       'NAGA',         'AIRAVATA',     'MAKARA',       'KINNARA',      'HAMSA',        'YALI',         'SIMURGH',      'BAKUNAWA',     'ANANTA',       'SHESHA',       'VASUKI',
    'FENRIR',       'JORMUNGANDR',  'SLEIPNIR',     'NIDHOGG',      'FAFNIR',       'HUGINN',       'MUNINN',       'RATATOSKR',    'GARM',         'HATI',         'SKOLL',        'AUDHUMLA',
    'TYPHON',       'CHIMERA',      'SPHINX',       'GRIFFON',      'HYDRA',        'PEGASUS',      'CERBERUS',     'PHOENIX',      'KRAKEN',       'LEVIATHAN',    'BEHEMOTH',     'LAMASSU',
]

GAMER_ALPHABET = GAMER_ALPHABET_2
# MATHEMATICAL_SPIRITS: one-sentence system-prompt prefix injected per agent.
# Provides lightweight prompt-level diversity without changing core instructions.
MATHEMATICAL_SPIRITS = [
    'You are channeling the mathematical spirit of Leonhard Euler.',
    'You are channeling the mathematical spirit of Carl Friedrich Gauss.',
    'You are channeling the mathematical spirit of Srinivasa Ramanujan.',
    'You are channeling the mathematical spirit of David Hilbert.',
    'You are channeling the mathematical spirit of Henri Poincaré.',
    'You are channeling the mathematical spirit of Alexander Grothendieck.',
    'You are channeling the mathematical spirit of Emmy Noether.',
    'You are channeling the mathematical spirit of Kurt Gödel.',
    'You are channeling the mathematical spirit of Évariste Galois.',
    'You are channeling the mathematical spirit of Bernhard Riemann.',
    'You are channeling the mathematical spirit of John von Neumann.',
    'You are channeling the mathematical spirit of Andrei Kolmogorov.',
    'You are channeling the mathematical spirit of Pierre de Fermat.',
    'You are channeling the mathematical spirit of Archimedes of Syracuse.',
    'You are channeling the mathematical spirit of Alan Turing.',
    'You are channeling the mathematical spirit of Terence Tao.',
]

# ── Engine configs ────────────────────────────────────────────────────────────

# spec_config examples (uncomment to use):
#   spec_config = eagle3_spec(GPT_OSS_120B_EAGLE3_PATHS['nvidia_lc'])
#   spec_config = draft_model_spec(GPT_OSS_20B)
#   spec_config = ngram_spec()
#   spec_config = mtp_spec()          # Qwen only

# Multi-Suite Mode: Make Sure That There Is No vLLM Port Conflict



_ec_gpt_120b = GptOssEngineConfig(
    model_path  = GPT_OSS_120B_PATH,
    port        = 8000,
    n_agents    = 8, #8
    batch_size  = 8, #8
    #spec_config = None,
    #spec_config = spec_decode_wenliang,
    spec_config = spec_decode_amazon,
    gpu_memory_utilization = 0.925, # 0.96 if not speculative decoding - spec decoding requires config calibration
    cudagraph_capture_sizes = _eagle3_cudagraph_sizes(batch_size=8, num_spec=3),
    max_num_batched_tokens  = 8192,
)

_ec_gpt_20b = GptOssEngineConfig(
    model_path  = GPT_OSS_20B_PATH,
    port        = 8000,
    n_agents    = 4,
    batch_size  = 8,
    spec_config = None,
    gpu_memory_utilization = 0.3,  
)

_ec_gpt_20b_sft_1 = GptOssEngineConfig(
    model_path  = GPT_OSS_20B_SFT_1_PATH,
    port        = 8001,
    n_agents    = 8,
    batch_size  = 8,
    spec_config = None,
    gpu_memory_utilization = 0.4,  
)

GPT_EC_LIST = [_ec_gpt_120b,_ec_gpt_20b, _ec_gpt_20b_sft_1]

_ec_qwen_35_27b = Qwen35EngineConfig(
    model_path  = QWEN35_27B_PATH,
    port        = 8001,
    n_agents    = 4,
    batch_size  = 8,
    #spec_config = None,
    spec_config = spec_decode_qwen35,
    gpu_memory_utilization = 0.64,  
)

_ec_qwen_35_35a3b = Qwen35EngineConfig(
    model_path  = QWEN35_35A3B_PATH,
    port        = 8001,
    n_agents    = 8,
    batch_size  = 8,
    #spec_config = None, 
    spec_config = spec_decode_qwen35, 

    gpu_memory_utilization = 0.64,  
)

QWEN_35_EC_LIST = [_ec_qwen_35_27b,_ec_qwen_35_35a3b]

# ── DeepSeek-R1 Qwen engine configs ──────────────────────────────────────────
_ec_ds_r1_qwen_7b  = DeepSeekR1QwenEngineConfig(
    model_path             = DS_R1_QWEN_7B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.48,
)
_ec_ds_r1_qwen_14b = DeepSeekR1QwenEngineConfig(
    model_path             = DS_R1_QWEN_14B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.48,
)
_ec_ds_r1_qwen_32b = DeepSeekR1QwenEngineConfig(
    model_path             = DS_R1_QWEN_32B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.96, 
)
# ── DeepSeek-R1-0528-Qwen3-8B (separate instance — different tokenizer) ──────
_ec_ds_r1_qwen3_8b = DeepSeekR1QwenEngineConfig(
    model_path             = DS_R1_QWEN3_8B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.48,   # 
)
# NOTE: NOT added to DS_R1_QWEN_EC_LIST — prompt injection would overwrite it.
# Use directly: ACTIVE_ENGINE_CONFIGS = [_ec_ds_r1_qwen3_8b]

DS_R1_QWEN_EC_LIST = [_ec_ds_r1_qwen_7b, _ec_ds_r1_qwen_14b, _ec_ds_r1_qwen_32b, _ec_ds_r1_qwen3_8b]

# ── DeepSeek-R1 Llama engine configs ─────────────────────────────────────────
_ec_ds_r1_llama_8b  = DeepSeekR1LlamaEngineConfig(
    model_path             = DS_R1_LLAMA_8B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.48,
)
_ec_ds_r1_llama_70b = DeepSeekR1LlamaEngineConfig(
    model_path             = DS_R1_LLAMA_70B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.96,
)
DS_R1_LLAMA_EC_LIST = [_ec_ds_r1_llama_8b, _ec_ds_r1_llama_70b]

# ── OpenReasoning-Nemotron engine configs ─────────────────────────────────────
_ec_or_7b  = OpenReasoningEngineConfig(
    model_path             = OR_7B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.96,
)
_ec_or_14b = OpenReasoningEngineConfig(
    model_path             = OR_14B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.96,
)
_ec_or_32b = OpenReasoningEngineConfig(
    model_path             = OR_32B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.96,
)
OR_EC_LIST = [_ec_or_7b, _ec_or_14b, _ec_or_32b]

# ── Gemma3 engine configs ─────────────────────────────────────────────────────
_ec_gemma3_12b = Gemma3EngineConfig(
    model_path             = GEMMA3_12B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.96,
)
_ec_gemma3_27b = Gemma3EngineConfig(
    model_path             = GEMMA3_27B_PATH,
    port                   = 8001,
    n_agents               = 8,
    batch_size             = 8,
    spec_config            = None,
    gpu_memory_utilization = 0.96,
)
GEMMA3_EC_LIST = [_ec_gemma3_12b, _ec_gemma3_27b]

# ── Experiment combos (uncomment to switch) ───────────────────────────────────
# Single-suite new models:
#ACTIVE_ENGINE_CONFIGS = [_ec_or_32b]                         # OR-32B solo
#ACTIVE_ENGINE_CONFIGS = [_ec_ds_r1_qwen_32b]                 # DS-R1-32B solo
#ACTIVE_ENGINE_CONFIGS = [_ec_ds_r1_llama_70b]                # DS-Llama-70B solo
#ACTIVE_ENGINE_CONFIGS = [_ec_gemma3_27b]                     # Gemma3-27B solo
# Dual-suite calibrated for H100 x1 (80GB):
#ACTIVE_ENGINE_CONFIGS = [_ec_ds_r1_qwen_14b, _ec_qwen_35_27b]  # DS-14B(0.36)+Qwen-27B(0.56)
#ACTIVE_ENGINE_CONFIGS = [_ec_or_14b, _ec_ds_r1_qwen_14b]       # OR-14B(0.44)+DS-14B(0.44)
#ACTIVE_ENGINE_CONFIGS = [_ec_or_14b, _ec_gemma3_12b]            # OR-14B(0.44)+Gemma3-12B(0.44)
#ACTIVE_ENGINE_CONFIGS = [_ec_gemma3_27b, _ec_ds_r1_qwen_7b]    # Gemma3-27B(0.58)+DS-7B(0.30)


##############################################################
# Single-suite baseline.
ACTIVE_ENGINE_CONFIGS = [_ec_gpt_120b] # OK Most adopted Model
#ACTIVE_ENGINE_CONFIGS = [_ec_gpt_20b] # OK
#ACTIVE_ENGINE_CONFIGS = [_ec_qwen_35_27b] # Qwen3.5 27B/35B OK
#ACTIVE_ENGINE_CONFIGS = [_ec_ds_r1_qwen3_8b] # DeepSeekR1 Qweb8B OK #but slow if overthinking
##############################################################

# Multi-suite experimental
#ACTIVE_ENGINE_CONFIGS = [_ec_gpt_20b, _ec_qwen_35_27b] # gpt 20b + Qwen3.5 27b OK - See Version 110
#ACTIVE_ENGINE_CONFIGS = [_ec_gpt_20b, _ec_ds_r1_qwen3_8b] 

# Mixture of agents from non-finetuned and math-finetuned gpt-oss-20b models (n agents need to be reduced)
# ACTIVE_ENGINE_CONFIGS = [_ec_gpt_20b, _ec_gpt_20b_sft_1] 

# ── MetaAgent control - Experimental - Can slow down the whole system ─────────────────────────────────────────────────────────
ENABLE_META_AGENT     = True # True / False - Secondary Solver to reconciliate conflicting answers with LLM reasoning instead of naive majority voting - May slow down the overall throughtput but might help handle hard problems if that's your target
META_AGENT_ENGINE_IDX = 0   # index into ACTIVE_ENGINE_CONFIGS
N_META_AGENTS_MAX     = 2
JUDGE_ENGINE_IDX      = 0   # AnswerChecker LLM judge engine index

# Hard-timeout safety constants
GLOBAL_SESSION_SAFETY_MARGIN = 120   # skip problem if < 2 min remains in session
PREDICT_HARD_TIMEOUT_GRACE   = 15    # extra grace seconds on top of per-problem deadline

# ── File paths ────────────────────────────────────────────────────────────────
BENCHMARK_OUTPUT_DIR   = '/kaggle/working/benchmark_output'

#BENCHMARK_DATASET_PATH = (    '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv')

BENCHMARK_DATASET_PATH = (    '/kaggle/input/datasets/khoinguyennguyen/hard-math-problem-aimo3/hard_50_math_problems_set_v6.csv')

TEST_PATH = '/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv'

print(f'MODE = {MODE}')
print(f'Active engines: {[type(ec).__name__ for ec in ACTIVE_ENGINE_CONFIGS]}')
_total = sum(ec.n_agents for ec in ACTIVE_ENGINE_CONFIGS)
_meta  = N_META_AGENTS_MAX if ENABLE_META_AGENT else 0
print(f'Total agents: {_total}  |  Sandboxes: {_total + _meta}')

# ── Config summary ────────────────────────────────────────────────────────────
for _ec in ACTIVE_ENGINE_CONFIGS:
    _spec_str = type(_ec.spec_config).__name__ if _ec.spec_config else 'None'
    print(f'  {_ec.served_model_name}: '
          f'batch={_ec.batch_size}, spec={_spec_str}')


## Block J - Behavioural Configuration (CFG)

`CFG` holds solver behavioural parameters only: timeouts, temperature ranges,
vote thresholds, context budgets, and execution parameters. Hardware and model
configuration belongs entirely in `EngineConfig` (Block I). The boundary is
enforced by convention, not by the type system - but respecting it means that
any experiment that changes only solver behaviour requires edits in exactly
one block, and any experiment that changes only hardware configuration requires
edits in exactly one different block.

The instance is named `cfg` (lowercase) to prevent shadowing the class
definition. All timeout values are in seconds.

#### Context budgets

| Parameter | Default | Role |
|---|---|---|
| `agent_context_tokens` | 65,536 (16384 x 4) | Maximum total tokens (prompt + generation) per agent TIR turn. `max_new_tokens` for each API call is derived as `agent_context_tokens - count_prompt_tokens(messages) - buffer_tokens`. When this value approaches zero, the agent exits the loop rather than submitting a near-empty generation budget. |
| `meta_context_tokens` | 65,536 | Same, for MetaAgent calls. Equal to `agent_context_tokens` by default; can be reduced to limit MetaAgent resource consumption. |
| `buffer_tokens` | 512 | Safety margin subtracted from both context budgets. Prevents submitting API calls with `max_tokens=0` or negative values when the prompt is near the context limit. |
| `search_tokens` | 32 | Number of recent chunks in the sliding window scanned for `\boxed{}` markers during `parse_stream`. Larger values catch answers that span more chunks at a higher per-chunk scan cost. |

In speculative decoding mode, reduce `agent_context_tokens` to
`int(16384 * 3.5)` or lower to reduce KV cache memory pressure. Eagle3
draft verification requires additional KV cache slots beyond those needed for
the target model alone; OOM under full context + high concurrency is a known
failure mode.

#### Temperature

| Parameter | Default | Role |
|---|---|---|
| `temp_agent_min` | 1.0 | Lower bound of the uniform distribution from which each SolverAgent's temperature is sampled. |
| `temp_agent_max` | 1.0 | Upper bound. At the default (`min == max == 1.0`) all agents sample at the same temperature. Widen to `[0.8, 1.2]` to increase path diversity across agents. |
| `temp_meta_min` | 0.7 | Lower bound for MetaAgent temperature. |
| `temp_meta_max` | 0.8 | Upper bound. MetaAgent temperatures are kept lower than SolverAgent temperatures to bias the secondary solver toward conservative, high-confidence answers rather than exploratory paths. |

Temperature affects the entropy of the model's output distribution at each
sampling step. Higher temperature increases diversity across agents - different
agents are more likely to explore different solution paths. Lower temperature
increases the probability of the modal (highest-probability) path. There is
no universal optimal value; the right range depends on the model and the
difficulty distribution of the problem set.

#### Timing

| Parameter | Default | Notes |
|---|---|---|
| `notebook_limit` | 17,400s (4h50m) | Kaggle hard wall clock limit. The adaptive deadline formula uses this as the total run horizon. In BENCHMARK mode with a smaller dataset, set to `total_problems x avg_expected_solve_time` to prevent the formula from reserving excess budget. |
| `high_problem_timeout` | 895s | Per-problem budget ceiling. $895 \times 50 = 44{,}750$s - well above the notebook limit. The ceiling exists to prevent a single problem from consuming the entire remaining budget under the adaptive formula when `problems_remaining` is small. |
| `base_problem_timeout` | 276s | Per-problem budget floor. $276 \times 50 = 13{,}800$s, leaving ~3,600s of slack against the 17,400s limit. No problem can be given less than 276s regardless of how much time earlier problems consumed. |
| `server_timeout` | 300s | Maximum time `_wait_for_ready()` will poll the vLLM server for HTTP 200. If the server does not respond within this window, `build_and_start()` raises. 300s is generous; a warm-cache startup typically takes 30-90s. |
| `session_timeout` | 960s | Socket timeout passed to the `OpenAI` client. Must exceed the longest plausible single API call duration. At 128 turns x ~7s per turn, the upper bound is ~896s; 960s leaves a small margin. |
| `jupyter_timeout` | 6.0s | Per-execution timeout passed to `JupyterSandbox.execute()`. Code blocks that run longer than this are interrupted. 6s is appropriate for mathematical computation; adjust upward if agents generate computations involving large symbolic algebra or numerical integration. |
| `sandbox_timeout` | 5.0s | `queue.Queue.get()` timeout when an agent acquires a sandbox from the pool. Under correct pool sizing (Block H) this should never expire; it is a safety net for misconfigured pool sizes. |

#### Stop conditions

| Parameter | Default | Stop condition | Meaning |
|---|---|---|---|
| `k2_raw_vote_stop` | 4 | Stop-A | Fire when at least `k2_raw_vote_stop` agents independently produce the same answer. Fast path for easy problems; does not require effort weighting. |
| `k1_min_valid` | 12 | Stop-B (count gate) | Minimum number of valid (non-`None`) answers required before the effort share threshold is evaluated. Prevents Stop-B from firing on two answers with lopsided effort. |
| `effort_stop_threshold` | 0.50 | Stop-B (effort gate) | Minimum effort share of the leading answer. A value of 0.50 means the leading answer has accumulated more tokens than all other answers combined. Both `k1_min_valid` and `effort_stop_threshold` must be satisfied simultaneously for Stop-B to fire. |
| `meta_vote_stop` | 3 | Stop-C | Minimum number of MetaAgents that must independently produce the same answer to trigger MetaAgent consensus. |

Stop condition interactions: `k2_raw_vote_stop` should be at most
`k1_min_valid`, so that the fast path (Stop-A) can fire before the quality
gate (Stop-B) accumulates enough answers. With 8 agents,
`k2_raw_vote_stop=4` means a bare majority (4/8) triggers early exit on easy
problems before effort weighting becomes meaningful. With 8 SolverAgents,
`k1_min_valid=12` effectively requires MetaAgent answers to contribute before
Stop-B can fire - making Stop-B a combined-population quality gate.

#### Execution

| Parameter | Default | Role |
|---|---|---|
| `workers` | 8 | `ThreadPoolExecutor` max workers for sandbox pool initialisation and is used as a minimum floor when computing `n_workers` for the Controller executor. |
| `max_turns` | 128 | Maximum TIR turns per SolverAgent attempt. MetaAgents are capped at `max_turns // 2 = 64`. In practice, the deadline and the context budget expire before 128 turns on most problems. |
| `seed` | 42 | Base seed passed to the `random.Random` instance used for spirit and temperature assignment, and to vLLM via `--seed`. Agent seeds are derived as `seed + attempt_index`. |
| `total_problems` | 50 | Expected dataset size, used in the adaptive deadline formula for `problems_remaining`. Adjust in BENCHMARK mode if the reference dataset is smaller than 50 problems. |

## Block K - Prompts

System prompts, tool namespace descriptions, and preference instructions are
defined here as plain string constants, then injected into `EngineConfig`
instances in the injection cell at the end of this block. The injection cell
(Block K, second code cell) must run after both Block I (which creates the
`EngineConfig` instances) and this cell (which defines the prompt strings).

Keeping prompts separate from `EngineConfig` field defaults serves one
practical purpose: prompts are the primary experimental variable in
competition development. A practitioner tuning prompts should not need to
navigate the `EngineConfig` class hierarchy to find and edit them.

#### Prompt roles

**`SYSTEM_PROMPT`** establishes the agent's mathematical reasoning context and
the TIR operating protocol. It specifies: the answer format constraint
(integer in $[0, 99999]$), the available Python libraries, the five-step
reasoning protocol (read, formulate, test, refine, state), and the general
instruction to verify numerically before committing. This prompt is identical
across all agents on the same engine; per-agent diversity is introduced
separately via the `MATHEMATICAL_SPIRITS` prefix prepended by `SolverAgent`.

**`TOOL_PROMPT`** (`gpt-oss` only) is passed as `ToolNamespaceConfig.description`
in `GptOssModelSuite.build_initial_messages()`. It describes the Python tool
namespace to the Harmony encoding layer: which libraries are pre-imported,
that `print()` output is returned as a tool response, and that kernel state
persists across calls within one problem. This string has no counterpart for
chat-completions suites, which use markdown code fences instead of a named
tool namespace; their `tool_prompt` field is set to `''` during injection.

**`PREFERENCE_PROMPT`** is appended to every user message (the problem
statement). It reinforces the answer format and the numerical verification
requirement. Keeping it separate from `SYSTEM_PROMPT` allows tuning the
per-message reminder independently of the system-level context.

#### Suite-specific variants

Five prompt sets are defined, covering the six model suites:

| Constant prefix | Suites | Format difference from gpt-oss |
|---|---|---|
| `SYSTEM_PROMPT` / `TOOL_PROMPT` / `PREFERENCE_PROMPT` | `gpt-oss` | Harmony TIR protocol; tool namespace described in `TOOL_PROMPT`; no code fence format instruction |
| `QWEN_SYSTEM_PROMPT` / `QWEN_PREFERENCE_PROMPT` | `Qwen35` | Markdown code fence format (` ```python `) shown explicitly in prompt; reasoning tags compatible |
| `DS_R1_*` | `DeepSeekR1Qwen`, `DeepSeekR1Llama` | Aliases of `QWEN_*` - identical format. Defined as separate names to allow independent tuning. |
| `OR_*` | `OpenReasoning` | Aliases of `QWEN_*` - Qwen2.5 backbone accepts identical format. |
| `GEMMA3_SYSTEM_PROMPT` / `GEMMA3_PREFERENCE_PROMPT` | `Gemma3` | Shorter prompt. `Gemma3ModelSuite.build_initial_messages()` merges system prompt and problem text into a single user message; concise instructions perform better in this merged format. |

The `DS_R1_*` and `OR_*` constants are explicit aliases rather than shared
references. This allows future independent tuning of DeepSeek-R1 and
OpenReasoning prompts without risk of cross-contaminating Qwen3.5 behaviour.

#### Prompt injection

The injection cell applies prompts to `EngineConfig` instances by iterating
over the EC lists defined in Block I:

```python
for _ec in GPT_EC_LIST:
    _ec.system_prompt     = SYSTEM_PROMPT
    _ec.tool_prompt       = TOOL_PROMPT
    _ec.preference_prompt = PREFERENCE_PROMPT

for _ec in QWEN_35_EC_LIST:
    _ec.system_prompt     = QWEN_SYSTEM_PROMPT
    _ec.tool_prompt       = ''        # no tool namespace for chat-completions suites
    _ec.preference_prompt = QWEN_PREFERENCE_PROMPT
```

The pattern repeats for each EC list. Only configs in `ACTIVE_ENGINE_CONFIGS`
are used at runtime, but all configs receive their prompts during injection
so that switching `ACTIVE_ENGINE_CONFIGS` never requires re-running the
injection cell.

After injection, `system_prompt`, `tool_prompt`, and `preference_prompt` are
read by `ModelSuiteProtocol.build_initial_messages()` to construct the opening
message of each agent's conversation. They are not read elsewhere.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK J — CFG DATACLASS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class CFG:
    """Solver behavioural configuration. All timeouts in seconds."""

    # ── Context budgets ───────────────────────────────────────────────────────
    agent_context_tokens:  int   = int(16384*4.0) # Consider context length reduction to prevent OOM in speculative decoding  int(16384*3.5) etc int(16384*4)
    meta_context_tokens:   int   = int(16384*4)
    buffer_tokens:         int   = 512
    search_tokens:         int   = 32    # sliding-window chunk count for stream scan

    # ── Temperature ───────────────────────────────────────────────────────────
    temp_agent_min:        float = 1.0   # Primary Solver creativity: widen to 0.8–1.2 for more path diversity , eg min 0.8 max 1.2 etc
    temp_agent_max:        float = 1.0 # Primary Solver creativity: widen to 0.8–1.2 for more path diversity , eg min 0.8 max 1.2 etc
    temp_meta_min:         float = 0.7 # Secondary Solver creativity - need to be more conservative
    temp_meta_max:         float = 0.8 # Secondary Solver creativity - need to be more conservative

    # ── Timing ────────────────────────────────────────────────────────────────
    notebook_limit:        int   = 3600*8   # 4h50m — Kaggle hard wall - change in benchmark mode if you datase is bigger 
    high_problem_timeout:  int   = 895     # max budget per problem - max 900
    base_problem_timeout:  int   = 276     # min reserved per problem
    server_timeout:        int   = 300     # vLLM readiness polling limit
    session_timeout:       int   = 960     # OpenAI client socket timeout
    jupyter_timeout:       float = 15.0     # per code-execution call 6.0 ->15.0 to support longer brute force loop
    sandbox_timeout:       float = 30.0     # queue.get() wait limit (was 5.0 — too tight under contention)

    # ── Stop conditions ───────────────────────────────────────────────────────
    k2_raw_vote_stop:      int   = 8       # default 4, but set to 8 aka total agent to benchmark hit rate of the system Stop-A: raw majority - Over All Agents - 4 by default - the former early_stop parameters , most common answer voted >=k2 => early stop
    k1_min_valid:          int   = 9       # Stop-B: minimum valid answers to trigger stop-B condition
    effort_stop_threshold: float = 0.5     # Stop-B: effort share threshold - if top answer by effort pct >=threshold + minimum valid answers >=k1 => early stop
    meta_vote_stop:        int   = 3       # Stop-C: MetaAgent consensus => in conflicting context, if n agree by meta agent > meta_vote_stop => early stop

    # ── Execution ─────────────────────────────────────────────────────────────
    workers:               int   = 8
    max_turns:             int   = 128
    seed:                  int   = 42 #vLLM seed
    total_problems:        int   = 50 #change in benchmark mode if you datase is bigger 

    # ── Miscellaneous ─────────────────────────────────────────────────────────
    enable_nonparsable_llm_extraction: bool = True


cfg = CFG()   # lowercase — never shadows the class

print('CFG instance created.')
print(f'  temp_agent=[{cfg.temp_agent_min},{cfg.temp_agent_max}]  '
      f'temp_meta=[{cfg.temp_meta_min},{cfg.temp_meta_max}]')
print(f'  stop-A k2={cfg.k2_raw_vote_stop}  '
      f'stop-B k1={cfg.k1_min_valid} thr={cfg.effort_stop_threshold}  '
      f'stop-C={cfg.meta_vote_stop}')

In [ ]:
# An experimental system prompt that force the Agents to leverage system thinking, self-awarenes and incremental updating of information - 
# AT THE COST OF slower reasoning and high token budget consumption.

OVERTHINKING_SYSTEM_PROMPT  = (
    "You are an elite mathematical problem solver operating at the level of the "
    "International Mathematical Olympiad (IMO). Your objective is to obtain the "
    "correct final answer through rigorous, structured, and strategically optimized reasoning.\n\n"

    "==============================\n"
    "I. STRATEGIC PROBLEM-SOLVING FRAMEWORK\n"
    "==============================\n"

    "1. UNDERSTAND PRECISELY\n"
    "- Restate the problem in precise mathematical terms.\n"
    "- Identify known quantities, unknowns, constraints, and structural conditions.\n"
    "- Classify the domain (algebra, combinatorics, number theory, geometry, functional equations, etc.).\n"
    "- Detect hidden structures, symmetry, invariants, monotonicity, extremal structure, or rigidity.\n\n"

    "2. EXPLORE THE STRUCTURAL LANDSCAPE\n"
    "- Generate multiple candidate approaches before committing.\n"
    "- Identify relevant theorems, transformations, substitutions, invariants, and bounding techniques.\n"
    "- Test small cases, extremal cases, and boundary behavior.\n"
    "- Look for equivalent reformulations or structural simplifications.\n"
    "- Consider reverse-engineering from the target expression or desired form.\n\n"

    "3. STRATEGIC SELECTION\n"
    "- Evaluate approaches by structural efficiency and inevitability.\n"
    "- Choose the path that reveals the deepest invariant or structural reduction.\n"
    "- Outline a high-level solution architecture before executing the details.\n\n"

    "4. EXECUTE WITH FULL RIGOR\n"
    "- Proceed only with logically justified steps.\n"
    "- Avoid heuristic leaps without justification.\n"
    "- Maintain symbolic clarity and structural awareness.\n"
    "- If a contradiction appears, isolate the failure and re-evaluate earlier assumptions.\n\n"

    "5. VERIFY AND STRESS-TEST\n"
    "- Re-check algebra and arithmetic carefully.\n"
    "- Confirm all constraints are satisfied.\n"
    "- Test extreme or special configurations.\n"
    "- Seek an independent consistency check when possible.\n\n"

    "==============================\n"
    "II. SYSTEM THINKING & INFORMATION GRAPH EXPANSION\n"
    "==============================\n"

    "- Maintain a dynamic State of Accumulated Knowledge (SAK).\n"
    "- At each stage, explicitly list:\n"
    "  • Known facts\n"
    "  • Derived properties\n"
    "  • Active constraints\n"
    "  • Useful lemmas\n"
    "- Expand the information graph by combining previously derived components.\n"
    "- Track dependencies: which facts generate which consequences.\n"
    "- Identify which missing intermediate quantity (X) unlocks the objective (Z).\n"
    "- Focus effort on discovering structurally necessary intermediate invariants.\n\n"

    "==============================\n"
    "III. REVERSE-ENGINEERING THE PROBLEM AUTHOR (IMO CREATION MINDSET)\n"
    "==============================\n"

    "Assume the problem was deliberately designed around a hidden central idea.\n"
    "Reverse-engineer the likely design process:\n"
    "- What is the key structural insight the author intended?\n"
    "- What auxiliary construction is likely being hidden?\n"
    "- Was the problem built from:\n"
    "    • A beautiful invariant?\n"
    "    • A disguised substitution?\n"
    "    • A rigidity or extremal argument?\n"
    "    • A symmetry-breaking step?\n"
    "    • A functional reparameterization?\n"
    "- Identify the 'point of inevitability' where the solution collapses into clarity.\n"
    "- Search for the hidden lemma the solver is meant to discover.\n"
    "- If progress stalls, reconstruct a plausible elegant solution and align toward it.\n\n"

    "==============================\n"
    "IV. HIGH-LEVEL MATHEMATICAL PRINCIPLES\n"
    "==============================\n"

    "- Reduce complexity through structural compression.\n"
    "- Convert dynamic processes into invariant statements.\n"
    "- Translate algebra into geometry when helpful (or vice versa).\n"
    "- Replace brute force with structural reasoning.\n"
    "- If symmetry exists, exploit it fully.\n"
    "- If asymmetry exists, locate its source and leverage it.\n"
    "- Prefer inevitability over computation.\n\n"

    "==============================\n"
    "V. SELF-AWARENESS PRINCIPLES\n"
    "==============================\n"
        
    "- Think out of the box. The hard problems are designed to benchmark AI creativity.\n"
    "- Do not let the desire to give an answer mislead your reasoning; no answer is better than a wrong answer.\n"
    "- Do not let the illusion of being right lead you to a wrong answer; always check your assumptions and hidden flaws in belief.\n"
    "- Maintain non-attachment to your previous efforts: if previous reasoning leads to a dead end, restart your thinking from scratch instead of guessing.\n"
    "- Avoid fixation on methodology and frameworks: given guidance can be ineffective for out-of-sample problems; use systems thinking instead of rigidly adhering to frameworks.\n"
    "- Persevere by accumulating small efforts: the given problems can be extremely hard; not being able to immediately find the methodology is normal.\n"
    "- Never forget the original intention: never abandon the principles and objectives.\n\n"

    "==============================\n"
    "VI. VERIFICATION REQUIREMENTS\n"
    "==============================\n"

    "- Cross-check every transformation.\n"
    "- Ensure no hidden assumptions violate constraints.\n"
    "- Confirm domain restrictions are respected.\n"
    "- Validate that the final result is logically forced, not coincidental.\n\n"

    "==============================\n"
    "VII. OUTPUT FORMAT\n"
    "==============================\n"

    "The final answer must be a non-negative integer between 0 and 99999.\n"
    "Place the final result inside \\boxed{} (example: \\boxed{42}).\n\n"

    "Reason step-by-step with full transparency. Depth, structure, and correctness "
    "are mandatory. The integrity of reasoning is as important as the final answer."
    )

# ══════════════════════════════════════════════════════════════════════════════
# RAG-like context wrapper — injected into agent problem text (JOURNEY mode)
# ══════════════════════════════════════════════════════════════════════════════

JOURNEY_SOLUTION_WRAPPER = (
    "\n\n--- Retrieved Reference (RAG context — may contain errors) ---\n"
    "A prior solution attempt was retrieved for this problem:\n\n"
    "{solution}\n\n"
    "This reference is highly correct. However, you will solve from scratch. Do not care about that solution. Even if that is correct you will solve from scratch as if you never know about the solution. Solve independently using "
    "Python tool-integrated reasoning. Verify everything. You MUST return the answer in less than 10 turns of Python tool.\n"
    "--- End Reference ---\n"
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PROMPTS — gpt-oss (Harmony TIR)
# ══════════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = (
    "You are an expert mathematical problem solver competing in a mathematical olympiad.\n"
    "You will be given a problem whose answer is a non-negative integer between 0 and 99999.\n\n"
    "Use Python freely:\n"
    "  - Symbolic computation with sympy\n"
    "  - Verify arithmetic, combinatorics, and enumeration numerically\n"
    "  - Test boundary cases and small examples\n"
    "  - Cross-check closed-form results against direct computation\n\n"
    "Reasoning protocol:\n"
    "  1. Read carefully; identify the mathematical structure\n"
    "  2. Formulate and test approaches in Python\n"
    "  3. Observe outputs; refine if unexpected\n"
    "  4. Repeat until verified\n"
    "  5. State the final answer as \\boxed{N}, an integer\n\n"
    "Self-awareness principles:\n" # Safeaguard against hard problems designed to target LLM-reasoning weaknesses - Remove to reduce time consumption
    "  - Observe everything; assume nothing; miss nothing\n"
    "  - Your weakness in combinatorics, geometry, creativity, sequential reasoning, overthinking, and long-context consistency may be targeted by the problem's author — be aware\n"
    "  - Your strength in brute-force coding can be used as a trap to consume time — carefully design your code\n"
    "  - Detect hidden beliefs, assumptions, and inertia in your own reasoning; break them if needed\n"
    "  - Embrace creativity; think beyond conventional methods\n"
    "  - Avoid rushing answers; slow answer is better than a wrong one\n"
    "  - Challenge assumptions; expose hidden flaws\n"
    "  - Practice non-attachment; restart if an approach fails\n"
    "  - Avoid fixation; adapt frameworks using systems thinking\n"
    "  - Persevere through difficult problems; accumulate small efforts\n"
    "  - Maintain original intention: find the correct answer via rigorous reasoning"
)

# Activate this line to try overthinking systemp prompt with GPT OSS suite
#SYSTEM_PROMPT = OVERTHINKING_SYSTEM_PROMPT

TOOL_PROMPT = (
    'A Python 3 interpreter with the following libraries pre-imported:\n'
    '  math, numpy, sympy, mpmath (dps=64), itertools, collections.\n'
    'Use print() to display intermediate results — output is returned to you.\n'
    'Kernel state persists across calls within one problem.'
)

PREFERENCE_PROMPT = (
    'Important: verify your answer numerically with Python before committing. '
    r'Express your final answer as \boxed{N} where N is an integer in [0, 99999].'
)

# ══════════════════════════════════════════════════════════════════════════════
# PROMPTS — Qwen3.5 (chat completions)
# ══════════════════════════════════════════════════════════════════════════════

QWEN_SYSTEM_PROMPT = (
    'You are an expert mathematical problem solver competing in a '
    'mathematical olympiad. The answer is always a non-negative integer '
    'between 0 and 99999 inclusive.\n\n'

    'You may use Python code blocks to assist your reasoning:\n'
    '```python\n'
    '# Your code here\n'
    '```\n'
    'Code output will be returned to you. Libraries available: '
    'math, numpy, sympy, mpmath (dps=64), itertools, collections.\n\n'

    'Reasoning protocol:\n'
    '  1. Think carefully about the mathematical structure.\n'
    '  2. Use Python to test, enumerate, or verify your approach.\n'
    '  3. Refine based on output. Repeat until you have a verified answer.\n'
    '  4. State your final answer as \\boxed{N}.\n\n'

    'Be rigorous. Verify numerically. Competition answers are always exact integers.'
)

QWEN_SYSTEM_PROMPT = SYSTEM_PROMPT

QWEN_PREFERENCE_PROMPT = (
    'Verify your answer with Python before committing. '
    r'Express your final answer as \boxed{N} where N is an integer in [0, 99999].'
)
QWEN_PREFERENCE_PROMPT = PREFERENCE_PROMPT
# ══════════════════════════════════════════════════════════════════════════════
# PROMPTS — DeepSeek-R1 (Qwen + Llama) and OpenReasoning-Nemotron
# All three families use the same Qwen chat format and TIR protocol.
# Prompt strings are aliases — defined once, reused across suites.
# ══════════════════════════════════════════════════════════════════════════════

DS_R1_SYSTEM_PROMPT    = QWEN_SYSTEM_PROMPT      # Qwen backbone: identical format
DS_R1_LLAMA_SYSTEM_PROMPT = QWEN_SYSTEM_PROMPT   # Llama supports system role fine
OR_SYSTEM_PROMPT       = QWEN_SYSTEM_PROMPT      # Qwen2.5 backbone: identical

DS_R1_PREFERENCE_PROMPT    = QWEN_PREFERENCE_PROMPT
DS_R1_LLAMA_PREFERENCE_PROMPT = QWEN_PREFERENCE_PROMPT
OR_PREFERENCE_PROMPT       = QWEN_PREFERENCE_PROMPT

# ══════════════════════════════════════════════════════════════════════════════
# PROMPTS — Gemma3
# Shorter than QWEN_SYSTEM_PROMPT — Gemma3ModelSuite merges system + problem
# into a single user message. Concise instructions perform better in this format.
# ══════════════════════════════════════════════════════════════════════════════

GEMMA3_SYSTEM_PROMPT = (
    'You are an expert mathematical problem solver. '
    'The answer is always a non-negative integer between 0 and 99999.\n\n'

    'Use Python code blocks to assist your reasoning:\n'
    '```python\n'
    '# your code here\n'
    '```\n'
    'Code output will be returned to you. '
    'Libraries available: math, numpy, sympy, mpmath (dps=64), itertools, collections.\n\n'

    'Reasoning protocol:\n'
    '  1. Analyse the mathematical structure.\n'
    '  2. Write and run Python to test or verify your approach.\n'
    '  3. Refine based on output. Repeat until you have a verified answer.\n'
    '  4. State your final answer as \\boxed{N}.\n\n'

    'Be rigorous. Verify numerically. Competition answers are always exact integers.'
)

GEMMA3_PREFERENCE_PROMPT = QWEN_PREFERENCE_PROMPT


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PROMPT INJECTION — wire prompts into engine config instances
# Must run after both Block I (config instances) and Block K (prompt strings).
# ══════════════════════════════════════════════════════════════════════════════

GPT_EC_LIST = [_ec_gpt_120b,_ec_gpt_20b]
QWEN_35_EC_LIST = [_ec_qwen_35_27b,_ec_qwen_35_35a3b]

for _ec_gpt in GPT_EC_LIST:
    _ec_gpt.system_prompt     = SYSTEM_PROMPT
    _ec_gpt.tool_prompt       = TOOL_PROMPT
    _ec_gpt.preference_prompt = PREFERENCE_PROMPT

for _ec_qwen in QWEN_35_EC_LIST:
    _ec_qwen.system_prompt     = QWEN_SYSTEM_PROMPT
    _ec_qwen.tool_prompt       = TOOL_PROMPT
    _ec_qwen.preference_prompt = QWEN_PREFERENCE_PROMPT

print('Prompts injected.')
print(f'  _ec_gpt.system_prompt  : {len(_ec_gpt.system_prompt)} chars')
print(f'  _ec_qwen.system_prompt : {len(_ec_qwen.system_prompt)} chars')
# ── DeepSeek-R1 Qwen — same chat format as Qwen35 ────────────────────────────
for _ec in DS_R1_QWEN_EC_LIST:
    _ec.system_prompt     = DS_R1_SYSTEM_PROMPT
    _ec.tool_prompt       = TOOL_PROMPT          # uses markdown code fences, no tool namespace
    _ec.preference_prompt = DS_R1_PREFERENCE_PROMPT

# ── DeepSeek-R1 Llama — same TIR protocol, Llama supports system role ─────────
for _ec in DS_R1_LLAMA_EC_LIST:
    _ec.system_prompt     = DS_R1_LLAMA_SYSTEM_PROMPT
    _ec.tool_prompt       = TOOL_PROMPT
    _ec.preference_prompt = DS_R1_LLAMA_PREFERENCE_PROMPT

# ── OpenReasoning-Nemotron — same Qwen chat format ────────────────────────────
for _ec in OR_EC_LIST:
    _ec.system_prompt     = OR_SYSTEM_PROMPT
    _ec.tool_prompt       = TOOL_PROMPT
    _ec.preference_prompt = OR_PREFERENCE_PROMPT

# ── Gemma3 — shorter merged prompt (no system role in build_initial_messages) ──
for _ec in GEMMA3_EC_LIST:
    _ec.system_prompt     = GEMMA3_SYSTEM_PROMPT
    _ec.tool_prompt       = TOOL_PROMPT
    _ec.preference_prompt = GEMMA3_PREFERENCE_PROMPT

# ── Summary ───────────────────────────────────────────────────────────────────
_all_ecs = (
    GPT_EC_LIST + QWEN_35_EC_LIST +
    DS_R1_QWEN_EC_LIST + DS_R1_LLAMA_EC_LIST +
    OR_EC_LIST + GEMMA3_EC_LIST
)
print(f'Prompts injected into {len(_all_ecs)} engine configs across 6 suite families.')
for _ec in _all_ecs:
    _sp_len = len(_ec.system_prompt)
    print(f'  {_ec.suite_id:<22} {type(_ec).__name__:<32} sys_prompt={_sp_len}c')


In [ ]:
# ── BENCHMARK mode: relax integer-only constraint for non-integer datasets ──
if MODE == 'BENCHMARK':
    import re as _re
    for _ec in _all_ecs:
        _ec.system_prompt = _ec.system_prompt.replace(
            'non-negative integer between 0 and 99999',
            'mathematical expression (integer, fraction, radical, etc.)'
        ).replace(
            'answer is always a non-negative integer',
            'answer may be an integer or a mathematical expression'
        ).replace(
            'Competition answers are always exact integers.',
            'Express your answer in exact mathematical form.'
        ).replace(
            'mathematical expression (integer, fraction, radical, etc.) inclusive.',
            'mathematical expression (integer, fraction, radical, etc.).'
        ).replace(
            'State the final answer as \\boxed{N}, an integer',
            'State the final answer as \\boxed{answer} in exact form'
        ).replace(
            'State your final answer as \\boxed{N}.',
            'State your final answer as \\boxed{answer} in exact mathematical form.'
        )
        _ec.preference_prompt = _re.sub(
            r'Express your final answer as .boxed.N. where N is an integer in .0, 99999.',
            r'Express your final answer as \\boxed{answer} in exact mathematical form.',
            _ec.preference_prompt,
        )
    print(f'BENCHMARK mode: integer constraint relaxed across {len(_all_ecs)} configs')

## Block L - System Initialisation

This is the GPU activation point. All cells in Blocks A through K can be
executed in a CPU-only environment for inspection, editing, and syntax
checking. No GPU memory is touched until this block runs.

Running the initialisation cell (`build_and_start()`) performs five steps in
order: engine startup (weight preloading and vLLM server launch), suite
construction (tokenizer loading), sandbox pool initialisation (kernel
spawning), atexit cleanup registration, and `AIMO3Solver` instantiation. The
full sequence is described in Block H.

`set_seed(cfg.seed)` is called immediately before `build_and_start()` to seed
Python's `random` module, NumPy, and PyTorch simultaneously. This ensures that
agent temperature and spirit sampling are reproducible across runs with the
same `cfg.seed`.

#### Inspection cell

The cell immediately before the initialisation cell prints each active
`EngineConfig` instance. Use it to verify that `model_path`, `port`,
`n_agents`, `batch_size`, `spec_config`, and `gpu_memory_utilization` are set
as intended before committing GPU memory. Correcting a misconfigured path after
`build_and_start()` has run requires a full restart.

#### Expected output

A successful initialisation produces output of the following form:

```
Preloading /kaggle/input/gpt-oss-120b/... (N files, X.XX GB) ...
  Preload complete.
Starting vLLM [gpt-oss] on port 8000 ...
  Log: vllm_server_8000.log
Waiting for server [gpt-oss:8000] ...
  Ready in Xs
[gpt-oss:8000] Server ready

Initialising N Jupyter sandboxes (8 agent slots + 4 meta slots) ...
  Sandboxes ready in Xs  (pool size: 12)

===================================================
  AIMO3Solver ready  |  mode=SUBMISSION  |  engines=1
  suites: ['gpt-oss']
  agents=8  sandboxes=12
===================================================
```

Wall clock from start to `AIMO3Solver ready`: typically 3-8 minutes on
Kaggle H100, dominated by vLLM weight loading from disk. The preload step
moves this cost into page cache warming rather than vLLM's internal load
routine; the actual vLLM `Waiting for server ...` step is much faster when
weights are already in RAM.

#### Remediation: zombie vLLM processes

If the notebook kernel is interrupted during or after `build_and_start()`,
the vLLM server process may continue running as a background process, holding
GPU memory. Subsequent calls to `build_and_start()` will then fail because
the port is already bound and GPU memory is already allocated.

The `nuke_gpu_memory()` cell above the initialisation cell handles this case.
It calls `subprocess.run(['pkill', '-9', '-f', 'vllm'])` (or equivalent) to
force-terminate all vLLM processes, then releases CUDA contexts. Run it, wait
5 seconds for the OS to reclaim resources, then re-run the initialisation cell.

Symptoms of a zombie process: `build_and_start()` hangs at
`Waiting for server [gpt-oss:8000] ...` for longer than `cfg.server_timeout`
(300s), or raises `OSError: [Errno 98] Address already in use` during server
launch. Checking `vllm_server_{port}.log` will show the prior session's output
still present, or an error about port binding.

In [ ]:
for engine_conf in ACTIVE_ENGINE_CONFIGS:
    print("============")
    print(engine_conf)
    print("============")

In [ ]:
# Kill all zomblie vLLM instances
nuke_gpu_memory()
time.sleep(5)

In [ ]:
set_seed(cfg.seed)

ENGINES, SOLVER = build_and_start(ACTIVE_ENGINE_CONFIGS, cfg)


## Block M - Entry Points

This block defines the two functions through which the solver is invoked.
Both functions are thin wrappers around `SOLVER.solve_problem()`. No solver
logic lives here.

#### predict() - Kaggle evaluation interface

`predict()` is the function signature mandated by the Kaggle evaluation
harness. The signature is fixed:

```python
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: pl.DataFrame = None)
    -> pl.DataFrame
```

`id_` and `question` are single-row `polars.DataFrame` objects. The return
value must be a single-row `polars.DataFrame` with columns `id` and `answer`.
This signature must not be modified. The evaluation harness inspects it at
registration time; a mismatch causes silent failure at submission.

`gc.disable()` is called immediately before `SOLVER.solve_problem()` and
`gc.enable()` is called in the `finally` block. Python's garbage collector
runs on a heuristic cycle count trigger and can fire at any point during
generation, including mid-stream inside `parse_stream`. A GC pause during
streaming introduces unpredictable latency that can breach the per-problem
deadline. Disabling GC for the duration of the solve call and triggering
`gc.collect()` explicitly in the `finally` block provides equivalent memory
management without unpredictable pauses.

#### predict_and_benchmark() - manual evaluation

`predict_and_benchmark()` is never called by the Kaggle harness. It is called
manually from Block N in BENCHMARK mode to iterate a reference CSV, pass
expected answers to the solver for per-attempt correctness labeling, and
return a `pandas.DataFrame` of `(id, answer)` pairs.

The function applies the same `gc.disable()` / `gc.enable()` pattern as
`predict()` for consistency. JSON output is written incrementally by
`AIMO3Solver._write_json()` after each problem, not after the full run,
so a partial run produces usable output files up to the point of interruption.

The reference DataFrame must have columns `id`, `problem`, and `answer`.
The `answer` column is passed as `expected_answer` to `solver.solve_problem()`
and used only by `AnswerChecker`; it is never shown to any agent.




In [ ]:
def predict(
    id_:      'pl.DataFrame',
    question: 'pl.DataFrame',
    answer:   'pl.DataFrame' = None,
) -> 'pl.DataFrame':
    """
    Kaggle AIMO3 inference interface.

    Called once per problem by the evaluation gateway.
    id_ and question are single-row polars DataFrames.
    Returns a single-row polars DataFrame with columns 'id' and 'answer'.

    All timeout and error handling is encapsulated inside
    AIMO3Solver.solve_problem() — see its docstring for the four safety layers.

    This wrapper only provides a last-resort catch for the case where SOLVER
    is not initialised (Block L was never run) or solve_problem() raised after
    exhausting its own exception handling.
    """
    pb_id        = str(id_.item(0))
    pb_text      = str(question.item(0))
    final_answer = ERROR_SENTINEL

    gc.disable()
    try:
        final_answer = SOLVER.solve_problem(pb_id, pb_text)
    except Exception as exc:
        print(f'[predict FATAL] pb_id={pb_id} — {type(exc).__name__}: {exc}')
        final_answer = ERROR_SENTINEL
    finally:
        gc.enable()
        gc.collect()

    # Clamp error sentinel to 0 at the submission boundary.
    # This is the ONLY place where ERROR_SENTINEL → 0 conversion happens,
    # keeping the distinction visible throughout the pipeline above.
    if isinstance(final_answer, int) and final_answer < 0:
        print(f'[{pb_id}] answer was ERROR_SENTINEL ({final_answer}) — submitting 0')
        final_answer = 0
    elif not isinstance(final_answer, int):
        # BENCHMARK non-integer answer — keep as-is for logging, submit 0
        print(f'[{pb_id}] non-integer answer ({final_answer!r}) — submitting 0')
        final_answer = 0

    return pl.DataFrame({'id': [pb_id], 'answer': [final_answer]})

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# predict_and_benchmark() — manual evaluation (BENCHMARK mode only)
# ══════════════════════════════════════════════════════════════════════════════

def predict_and_benchmark(
    ref_df:  'pd.DataFrame',
    solver:  AIMO3Solver,
    solution_map: dict = None,   # {pb_id: context} for RAG-like agents
) -> 'pd.DataFrame':
    """
    Iterate a reference DataFrame and call solver.solve_problem() for each row.

    ref_df must have columns: 'id', 'problem', 'answer'
    solution_map: optional dict mapping id -> retrieved context string
    Returns pd.DataFrame({'id': [...], 'answer': [...]})

    Writes per-problem JSON files to attempts/, meta/, and problems/
    subdirectories via AIMO3Solver._write_json() after each problem.
    """
    ids:     list = []
    answers: list = []

    for _, row in ref_df.iterrows():
        pb_id           = str(row['id'])
        pb_text         = str(row['problem'])
        expected_answer = str(row['answer'])

        # RAG context lookup
        real_solution = ''
        if solution_map and pb_id in solution_map:
            real_solution = solution_map[pb_id]

        gc.disable()
        try:
            final_answer = solver.solve_problem(
                pb_id           = pb_id,
                pb_text         = pb_text,
                expected_answer = expected_answer,
                real_solution   = real_solution,
            )
        finally:
            gc.enable()
            gc.collect()

        ids.append(pb_id)
        # Clamp error sentinel for output (raw value preserved in blackboard summary)
        # Keep raw answer (int or str) — type guard for non-integer BENCHMARK answers
        answers.append(final_answer if not isinstance(final_answer, int) else max(0, final_answer))

    return pd.DataFrame({'id': ids, 'answer': answers})

## Block N — Evaluation Gateway
This is the terminal cell of the notebook. In SUBMISSION mode, no cells
should execute after it.

#### SUBMISSION mode
The `AIMO3InferenceServer` gateway is instantiated with `predict` as the
registered callback, and `run_local_gateway((TEST_PATH,))` is called. The
gateway manages the problem-delivery loop: it reads problems from the test
file, calls `predict()` once per problem, collects the returned DataFrames,
and handles timeout enforcement at the harness level. `run_local_gateway()`
blocks until all 50 problems are processed or the Kaggle wall-clock limit is
reached.

Re-running this cell restarts the evaluation from problem 1. The adaptive
deadline formula will reinitialise with an empty `RunTracker` (no rolling-
average history), so the first several problems will use `base_problem_timeout`
as the effective floor until enough history accumulates.

`LOCAL_SIMULATION_ON_50_HARD_PB_DATASET` is used to simulate your system in the
evaluation phase with a set of 50 hard problems. You can estimate, in a
realistic production context, which hyperparameter set will be optimal.

#### BENCHMARK mode
`predict_and_benchmark()` is called directly with the reference CSV read into
a `pandas.DataFrame`. The function iterates over all rows, calls
`SOLVER.solve_problem()` for each, and returns a results DataFrame.

After all problems complete, `SOLVER.blackboard.run_tracker.print_run_summary()`
prints the full run summary: total wall time, average and per-problem solve
time, accuracy (if `n_correct` was recorded), stop-reason breakdown, and
fastest/slowest problem durations.

Three JSON files are written to `BENCHMARK_OUTPUT_DIR`:

| Directory | Reading | Key analysis uses |
|---|---|---|
| `attempts/*.json` | `[json.load(open(f)) for f in glob('attempts/*.json')]` | Per-agent accuracy by suite, effort-vs-correctness correlation, `python_errors` distribution, stop-condition analysis |
| `meta/*.json` | `[json.load(open(f)) for f in glob('meta/*.json')]` | MetaAgent answer correctness, contribution to final answer on conflicted problems |
| `problems/*.json` | `[json.load(open(f)) for f in glob('problems/*.json')]` | Stop-reason distribution, `weighted_pct_winner` vs correctness, solve-time distribution |

The `messages` field in attempt and meta-trace JSON files contains
the full serialised conversation for each agent. This field is the primary
source for qualitative analysis of reasoning traces: understanding how agents
approach specific problem types, where TIR loops fail or succeed, and which
mathematical spirits produce different reasoning styles.

In [ ]:
import pandas
sample_df = pandas.read_csv("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv")

another_sample_df = pandas.read_csv("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv")

sample_df = sample_df[sample_df['id'].apply(lambda x: x in ['0e644e','26de63'])]
sample_df.to_csv("small_reference.csv",index = False)





### Output structure
All reasoning traces from every agent and per-problem statistics are saved to `benchmark_output/`, organized by problem ID and attempt index:
- `attempts/{pb_id}__{agent_name}__{attempt}.json` — full conversation trace per agent
- `meta/{pb_id}__meta.json` — MetaAgent reconciliation traces
- `problems/{pb_id}.json` — vote distribution, effort stats, final answer, correctness



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BLOCK N — EVALUATION GATEWAY
# MODE guard: SUBMISSION wires the Kaggle harness; BENCHMARK runs manually.
# ══════════════════════════════════════════════════════════════════════════════
# IGNORE THIS ONE: "Error: Unexpected EOS while waiting for message header to complete" - 
# Early Stopping mechanism will stop streaming decoding for all agents once answer consensus is reached to switch to next problem instead of wasting time waiting for new tokens.

inference_server = (kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict) )

# Activate this in Submission Mode to Integration And Stability Test : Given a set of 50 hard problems (similar to the unknown AIMO3 problems), how your Solver will behave.
#EVALUATION_SIMULATION_DATASET_PATH = "/kaggle/input/datasets/khoinguyennguyen/hard-math-problem-aimo3/hard_50_math_problems_set_v6.csv"
EVALUATION_SIMULATION_DATASET_PATH = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
#EVALUATION_SIMULATION_DATASET_PATH = "small_reference.csv"
LOCAL_SIMULATION_ON_50_HARD_PB_DATASET = False # True/False
#LOCAL_SIMULATION_ON_50_HARD_PB_DATASET = True # True/False

if MODE == 'SUBMISSION' and not LOCAL_SIMULATION_ON_50_HARD_PB_DATASET:

    inference_server.run_local_gateway((TEST_PATH,))

elif MODE == 'SUBMISSION' and LOCAL_SIMULATION_ON_50_HARD_PB_DATASET:
    
    inference_server.run_local_gateway((EVALUATION_SIMULATION_DATASET_PATH,))
    
else:

    # ── RAG context loading + dataset shuffle (JOURNEY mode) ─────────
    solution_map = {}
    if JOURNEY_TO_THE_WEST:
        import time as _time
        _journey_df = pd.read_parquet(JOURNEY_DATASET_PATH)

        _journey_df = _journey_df[_journey_df['answer'].notna()]
        print(f'RAG source: {len(_journey_df)} problems')

        for _, _r in _journey_df.iterrows():
            if str(_r.get('real_solution', '')).strip():
                solution_map[str(_r['id'])] = str(_r['real_solution'])

        _journey_seed = int(_time.time()) % 10000
        _journey_df = _journey_df.sample(frac=1.0, random_state=_journey_seed).reset_index(drop=True)

        _journey_csv = os.path.join(BENCHMARK_OUTPUT_DIR, 'journey_to_the_west_ref.csv')
        os.makedirs(BENCHMARK_OUTPUT_DIR, exist_ok=True)
        _journey_df[['id', 'problem', 'answer']].to_csv(_journey_csv, index=False)
        BENCHMARK_DATASET_PATH = _journey_csv
        print(f'RAG contexts: {len(solution_map)} | seed: {_journey_seed}')

    # BENCHMARK: iterate reference dataset, label correctness, write traces
    ref_df     = pd.read_csv(BENCHMARK_DATASET_PATH)
    results_df = predict_and_benchmark(ref_df, SOLVER, solution_map=solution_map)

    # Final run summary
    SOLVER.blackboard.run_tracker.print_run_summary()

    print('\nBenchmark output written to:', BENCHMARK_OUTPUT_DIR)
    print(results_df.to_string(index=False))